# Mission 2 acoustic feature expansion

목표는 frozen E05 42D에 사전고정한 acoustic block을 **한 번에 하나씩** 추가해 동일 split·동일 M04-100 설정으로 비교하는 것이다. 이 Notebook은 결과를 가정하거나 candidate를 자동 승격하지 않는다.

허용 입력은 현재 발화의 waveform뿐이다. `text`, 다른 annotation, 발화 순서/index, 절대 `startAt`/`endAt`, call-relative 정보와 overlap metadata는 모델 feature가 아니다. `call_id`, 경계값과 `record_key`는 split·crop·추적에만 사용한다.

실행 제어는 artifact + fingerprint가 담당한다. expensive operation만 아래 두 Boolean으로 막으며, 조합·eGeMAPS·model-family search는 이 단계의 구현 범위가 아니다.


## Checkpoint A — setup과 유일한 두 실행 gate

저장 기본값은 항상 `False`다. 조회·로드·계약 검증에는 별도 switch를 만들지 않는다.


In [9]:
from pathlib import Path
import sys

# repo root 찾기
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

# src 경로를 Python import path에 추가
SRC_ROOT = REPO_ROOT / "src"
sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from dcc_data.audio import read_wav_frames
from dcc_data.mission2_model_experiments import (
    E05_FEATURE_NAMES,
    EXPECTED_INTERNAL_SPLIT,
    EXPECTED_OFFICIAL_CALLS,
    EXPECTED_OFFICIAL_ROWS,
    create_fixed_internal_split,
    load_standalone_e05_cache,
    standalone_group_split_indices,
    standalone_training_cache_paths,
)
from dcc_data.mission2_acoustic_feature_expansion import (
    BLOCK_BY_ID,
    FEATURE_BLOCKS,
    FROZEN_49D_FEATURE_NAMES,
    FROZEN_49D_REFERENCE_ID,
    M04_ITER100_ACCURACY_TARGET,
    M04_ITER100_FEATURE_SCREENING_CONFIG,
    PITCH_EXTRACTION_CONFIG,
    NEXT_CANDIDATE_BLOCK_IDS,
    ROUND1_BLOCK_IDS,
    ROUND2_BLOCK_IDS,
    ROUND2_M04_ITER100_ACCURACY_TARGET,
    ROUND2_REFERENCE_ID,
    audit_reference_reproduction,
    benchmark_pitch_extraction,
    build_candidate_feature_cache,
    build_frozen_49d_residual_analysis,
    build_reference_contract,
    build_result_diagnostics,
    candidate_cache_paths,
    candidate_hypothesis_table,
    candidate_manifest_table,
    compare_feature_candidate_to_reference,
    config_fingerprint,
    implemented_candidate_definition_table,
    load_candidate_feature_cache,
    prepare_single_block_screening,
    prepare_round2_single_block_screening,
    prepare_frozen_49d_single_block_screening,
    run_frozen_49d_screening_candidate,
    run_frozen_49d_screening_reference,
    run_round2_feature_screening_candidate,
    run_round2_feature_screening_reference,
    run_feature_screening_candidate,
    run_feature_screening_reference,
)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
RAW_DATA_ROOT = REPO_ROOT / 'data' / 'raw_original'
MANIFEST_PATH = REPO_ROOT / 'tmp' / 'mission2_eda' / 'utterance_manifest_v1.parquet'
CANDIDATE_CACHE_ROOT = REPO_ROOT / 'data' / 'processed' / 'mission2_acoustic_feature_expansion'

RUN_FEATURE_EXTRACTION = False
RUN_SINGLE_BLOCK_SCREENING = False

print('Repository:', REPO_ROOT)
print('Expensive gates:', RUN_FEATURE_EXTRACTION, RUN_SINGLE_BLOCK_SCREENING)


Repository: C:\Users\YW\Study\02_Projects\DCC_2026\prelim
Expensive gates: False False


## Checkpoint B — frozen E05와 canonical manifest 로드

기존 immutable cache만 읽는다. cache miss를 waveform 재추출로 우회하지 않는다. 출력은 E05가 MFCC mean 20 + MFCC std 20 + spectral rolloff + ZCR의 정확한 42D이고 canonical `record_key`와 1:1 정렬됨을 확인한다.


In [10]:
e05_cache = load_standalone_e05_cache(
    standalone_training_cache_paths(REPO_ROOT),
    split='Training',
    expected_rows=EXPECTED_OFFICIAL_ROWS['Training'],
    expected_calls=EXPECTED_OFFICIAL_CALLS['Training'],
)
utterance_manifest = pd.read_parquet(MANIFEST_PATH)
training_manifest = utterance_manifest.loc[
    utterance_manifest['split'].astype(str) == 'Training'
].reset_index(drop=True)
if len(training_manifest) != EXPECTED_OFFICIAL_ROWS['Training']:
    raise ValueError('Training manifest row contract mismatch')
if set(training_manifest['record_key'].astype(str)) != set(e05_cache.validated.record_keys):
    raise ValueError('E05/manifest record_key set mismatch')
if e05_cache.validated.feature_names != E05_FEATURE_NAMES:
    raise ValueError('E05 feature order changed')
print('E05:', e05_cache.validated.X.shape, e05_cache.validated.X.dtype)
print('E05 features:', E05_FEATURE_NAMES)
display(pd.DataFrame([e05_cache.cache_audit]))


E05: (889358, 42) float32
E05 features: ('mfcc_01_mean', 'mfcc_02_mean', 'mfcc_03_mean', 'mfcc_04_mean', 'mfcc_05_mean', 'mfcc_06_mean', 'mfcc_07_mean', 'mfcc_08_mean', 'mfcc_09_mean', 'mfcc_10_mean', 'mfcc_11_mean', 'mfcc_12_mean', 'mfcc_13_mean', 'mfcc_14_mean', 'mfcc_15_mean', 'mfcc_16_mean', 'mfcc_17_mean', 'mfcc_18_mean', 'mfcc_19_mean', 'mfcc_20_mean', 'mfcc_01_std', 'mfcc_02_std', 'mfcc_03_std', 'mfcc_04_std', 'mfcc_05_std', 'mfcc_06_std', 'mfcc_07_std', 'mfcc_08_std', 'mfcc_09_std', 'mfcc_10_std', 'mfcc_11_std', 'mfcc_12_std', 'mfcc_13_std', 'mfcc_14_std', 'mfcc_15_std', 'mfcc_16_std', 'mfcc_17_std', 'mfcc_18_std', 'mfcc_19_std', 'mfcc_20_std', 'spectral_rolloff', 'zcr')


,backend,split,rows,calls,canonical_order,record_key_signature,candidate_source_dtype,final_dtype,automatic_extraction
0,standalone_adapter,Training,889358,29200,frozen_mfcc_npz,7c1e27f4f45ce84f261467f5863fcf8df838a8f1b22dc2...,float64,float32,False


## Checkpoint C — 고정 GroupSplit 계약

모든 representation은 `call_id` 기준 동일 split을 공유한다. `call_id`는 모델 입력이 아니며 overlap은 0이어야 한다. scaler는 이후 각 representation별 internal Training 행에만 fit한다.


In [3]:
fixed_train_idx, fixed_validation_idx, fixed_split_audit = create_fixed_internal_split(
    e05_cache.tracking['call_id'].astype(str).to_numpy(),
    split_function=standalone_group_split_indices,
    expected_contract=EXPECTED_INTERNAL_SPLIT,
)
if fixed_split_audit['call_overlap'] != 0:
    raise AssertionError('call_id leakage detected')
display(pd.DataFrame([fixed_split_audit]))


,seed,test_size,total_rows,train_rows,validation_rows,train_calls,validation_calls,call_overlap
0,42,0.2,889358,711467,177891,23360,5840,0


## Checkpoint D — result-blind precommit과 M04-100 reference 정의

M04-100은 최종 모델이 아니라 Feature Screening Reference다. 기존 current-session Accuracy `0.859532`는 reproduction target일 뿐 단독 fail-fast 조건이 아니다. 실제 fit 전에 row/split/feature/scaler/config/seed fingerprint를 만들고, fit 뒤 그 구조가 먼저 일치해야 한다.

AF01의 log-RMS mean은 recording level/gain에 민감한 diagnostic risk다. AF02의 주파수·frame/hop·semitone reference·slope·최소 voiced frame·finite fallback은 아래 설정으로 고정된다. AF03은 명시적 normalized Euclidean spectral flux이며 `librosa.onset_strength`나 eGeMAPS 구현과 동일하다고 간주하지 않는다.


In [4]:
precommit_table = candidate_manifest_table()
display(precommit_table)
display(pd.DataFrame([dict(PITCH_EXTRACTION_CONFIG)]))
display(pd.DataFrame([{
    'reference': 'E05_42D + M04_ITER100',
    'accuracy_reproduction_target': M04_ITER100_ACCURACY_TARGET,
    'model_config_fingerprint': config_fingerprint(M04_ITER100_FEATURE_SCREENING_CONFIG),
    'accuracy_exact_equality_is_gate': 'NO',
}]))


,block_id,feature_names,acoustic_meaning,calculation,e05_overlap,mission2_hypothesis,research_basis,rule_compliance,diagnostic_risks,expected_cost,output_dimension
0,AF01_ENERGY_DYNAMICS,"(log_rms_mean_db, log_rms_std_db, log_rms_p90_...",utterance energy level and frame-to-frame ener...,"centered frame RMS (512/160), 20*log10(max(RMS...",MFCC moments contain related energy informatio...,caller and dispatcher speaking style may diffe...,GeMAPS treats energy/amplitude descriptors as ...,current-utterance waveform only; no annotation...,log-RMS mean is recording-level and gain sensi...,low,3
1,AF02_PITCH_VOICING,"(f0_semitone_median, f0_semitone_iqr, f0_semit...","F0 level, F0 variation, contour direction, and...",normalized autocorrelation at 8 kHz with fmin=...,E05 has no explicit F0 contour or voicing-prob...,prosodic and voicing patterns may add role-rel...,GeMAPS includes logarithmic F0 and voicing-rel...,current-utterance waveform only; no speaker me...,"pitch tracking can fail on short, noisy, clipp...",high relative to AF01/AF03; benchmark before f...,5
2,AF03_NORMALIZED_SPECTRAL_FLUX,"(normalized_spectral_flux_mean, normalized_spe...",frame-to-frame spectral-shape change,"L2-normalized magnitude STFT columns (Hann, 51...",rolloff and ZCR are static utterance summaries...,role-related speaking dynamics may remain afte...,spectral flux is used as a spectral-dynamics d...,current-utterance waveform only; no neighborin...,sensitive to transient noise and very short se...,medium; reuses one STFT per utterance,3


,method,fmin_hz,fmax_hz,frame_length,hop_length,semitone_reference_hz,minimum_voiced_frames,periodicity_threshold,minimum_frame_rms,slope_unit,fallback
0,normalized_autocorrelation,65.0,400.0,1024,160,27.5,3,0.6,0.0001,semitones_per_second,zero_f0_statistics_keep_voicing_statistics


,reference,accuracy_reproduction_target,model_config_fingerprint,accuracy_exact_equality_is_gate
0,E05_42D + M04_ITER100,0.859532,2c403b5c188d3d10c5655504b1743e1439289174d901cd...,NO


AF01 Energy Dynamics = 3D
- log-RMS mean / std / P90-P10
- E05와 일부 energy 정보 중복 가능성을 명시
- 특히 log-RMS mean의 녹음 gain 민감성을 diagnostic risk로 기록해 둠

AF02 Pitch/Voicing = 5D
- 65–400 Hz
- frame 1024, hop 160
- normalized autocorrelation
- periodicity threshold 0.6
- 최소 voiced frame 3
- slope 단위 semitones/sec
- pitch 추정 실패 시 F0 통계는 0으로 두고 voicing 통계는 유지
- E05에 명시적 F0/voicing 정보가 없으므로 새 정보 후보로 타당

AF03 Normalized Spectral Flux = 3D
- 정적인 rolloff/ZCR이 아니라 frame 간 spectral shape 변화량을 보겠다는 가설
- E05와 목적이 다름
- transient noise / 매우 짧은 발화 민감성도 risk로 기록됨

## Checkpoint E — full extraction 전 small runtime benchmark

Training의 canonical 선두 8개 유효 발화만 읽어 AF02 runtime과 finite contract를 실측한다. 이 결과는 전체 추출 결과가 아니며, 추정치는 운영 판단용 참고값이다. 이 셀을 검토한 뒤에만 Checkpoint F의 extraction gate를 바꾼다.


In [5]:
benchmark_rows = training_manifest.loc[training_manifest['crop_ok'].astype(bool)].head(8)
benchmark_segments = []
for benchmark_audio_path, benchmark_group in benchmark_rows.groupby('audio_path', sort=False):
    benchmark_audio = read_wav_frames(RAW_DATA_ROOT / str(benchmark_audio_path))
    if benchmark_audio.sample_rate != 8_000 or benchmark_audio.waveform.shape[0] != 1:
        raise ValueError('benchmark WAV contract mismatch')
    for benchmark_row in benchmark_group.itertuples(index=False):
        benchmark_segments.append(
            benchmark_audio.waveform[0, int(benchmark_row.crop_start_frame):int(benchmark_row.crop_end_frame)]
        )
pitch_runtime_benchmark = benchmark_pitch_extraction(benchmark_segments)
pitch_runtime_benchmark['naive_889358_utterance_estimate_hours'] = (
    pitch_runtime_benchmark['estimated_seconds_per_utterance']
    * EXPECTED_OFFICIAL_ROWS['Training']
    / 3_600.0
)
display(pd.DataFrame([pitch_runtime_benchmark]))


,utterance_count,audio_seconds,runtime_seconds,estimated_seconds_per_utterance,pitch_config_fingerprint,scope,naive_889358_utterance_estimate_hours
0,8,20.102,0.102515,0.012814,6a3dda4caa74914667125f04ad27eb4802fadb411ce515...,small_runtime_benchmark_only_not_full_extraction,3.165728


## Checkpoint F — AF01–AF03 cache 자동 재사용 또는 명시적 전체 추출

두 artifact가 모두 있으면 fingerprint 검증 후 자동 재사용한다. artifact가 없을 때만 이 셀을 실행하기 직전에 `RUN_FEATURE_EXTRACTION=True`로 바꾼다. partial/stale artifact는 덮어쓰지 않으며 `RUN_REBUILD_*` 우회로도 없다.


In [7]:
candidate_parquet_path, candidate_metadata_path = candidate_cache_paths(
    CANDIDATE_CACHE_ROOT, split='Training', block_ids=ROUND1_BLOCK_IDS
)
if candidate_parquet_path.is_file() and candidate_metadata_path.is_file():
    candidate_feature_frame = load_candidate_feature_cache(
        manifest=utterance_manifest, cache_root=CANDIDATE_CACHE_ROOT, split='Training',
        block_ids=ROUND1_BLOCK_IDS,
    )
elif candidate_parquet_path.exists() or candidate_metadata_path.exists():
    raise RuntimeError('STOP: partial candidate artifact exists; inspect it instead of rebuilding')
elif not RUN_FEATURE_EXTRACTION:
    raise RuntimeError(
        'STOP: candidate cache is absent. Review Checkpoint E, then set only '
        'RUN_FEATURE_EXTRACTION=True in Checkpoint A and rerun this cell.'
    )
else:
    build_candidate_feature_cache(
        manifest=utterance_manifest,
        data_root=RAW_DATA_ROOT,
        cache_root=CANDIDATE_CACHE_ROOT,
        split='Training',
        allow_full_extraction=RUN_FEATURE_EXTRACTION,
        on_call_done=lambda done, total, call_id: print(done, '/', total, call_id) if done % 500 == 0 else None,
        block_ids=ROUND1_BLOCK_IDS,
    )
    candidate_feature_frame = load_candidate_feature_cache(
        manifest=utterance_manifest, cache_root=CANDIDATE_CACHE_ROOT, split='Training',
        block_ids=ROUND1_BLOCK_IDS,
    )
print('Candidate cache:', candidate_feature_frame.shape, candidate_parquet_path)


500 / 29200 651e4932d392f1f6bd520f74_20220102
1000 / 29200 651e494aaf1b6ab3ae0e5878_20220105
1500 / 29200 651e498789e4964417c980ca_20220107
2000 / 29200 651e49aac10cdef4499a97a0_20220203
2500 / 29200 651e49d83f1c5e4f25eec794_20220205
3000 / 29200 651e49ec1d10874a0f4eccce_20220206
3500 / 29200 651e4a0693f80b92304ff484_20220301
4000 / 29200 651e4a3786dc055ca8bf435d_20220307
4500 / 29200 651e4a554767d37510dad841_20220401
5000 / 29200 651e4a6ede6495f4e9d36a02_20220406
5500 / 29200 651e4a8805085d58eeeb1792_20220407
6000 / 29200 651e4aaacb7c8f4a6a1ffaca_20220501
6500 / 29200 651e4b1829d82455557a22ff_20220502
7000 / 29200 651e4baacf5885969118c236_20220504
7500 / 29200 651e4bc5df6bf0d825aa53b7_20220505
8000 / 29200 651e4be244870884abc4760b_20220506
8500 / 29200 651e4bfaf12d88de38d49833_20220507
9000 / 29200 651e4c3e8c3bda5f9fa389a2_20220601
9500 / 29200 651e4c5b48bae90750379cab_20220602
10000 / 29200 651e4c8ad24efd3f59d0146b_20220607
10500 / 29200 651e4d9be84a30cdf982d12a_20220702
11000 / 2920

## Checkpoint G — single-block matrices와 pre-fit fingerprints

E05 prefix를 바꾸지 않고 AF01, AF02, AF03을 각각 따로 붙인다. 모든 representation은 같은 row/split을 쓰되 scaler는 각 representation의 internal Training에만 별도로 fit한다. 이 단계는 모델을 학습하지 않는다.


In [8]:
prepared_representations = prepare_single_block_screening(
    X_e05=e05_cache.validated.X,
    y=e05_cache.validated.y,
    record_keys=e05_cache.validated.record_keys,
    call_ids=e05_cache.tracking['call_id'].astype(str).to_numpy(),
    candidate_frame=candidate_feature_frame,
    expected_split_contract=EXPECTED_INTERNAL_SPLIT,
    block_ids=ROUND1_BLOCK_IDS,
)
reference_prefit_contract = build_reference_contract(prepared_representations['E05_REFERENCE'])
display(pd.DataFrame([reference_prefit_contract]))
display(pd.DataFrame([
    {
        'representation_id': key,
        'feature_dimension': len(value.feature_names),
        'train_rows': len(value.y_train),
        'validation_rows': len(value.y_evaluation),
        'scaler_fit_scope': 'internal_train_only',
    }
    for key, value in prepared_representations.items()
]))


,row_fingerprint,split_fingerprint,feature_fingerprint,scaler_fingerprint,config_fingerprint,seed,scaler_fit_scope
0,b875bdb484407d66cfe699cd8314311fede1bd798bdacc...,0b87b2459b5dc08a0930d007eafa310ed34b1fe5fc45c8...,4dc346cca55fb88175bcd568f62a42815d726f963796a3...,6b92cd924a222106ec55ac277cb488330e8f56fea10c2e...,2c403b5c188d3d10c5655504b1743e1439289174d901cd...,42,internal_train_only


,representation_id,feature_dimension,train_rows,validation_rows,scaler_fit_scope
0,E05_REFERENCE,42,711467,177891,internal_train_only
1,AF01_ENERGY_DYNAMICS,45,711467,177891,internal_train_only
2,AF02_PITCH_VOICING,47,711467,177891,internal_train_only
3,AF03_NORMALIZED_SPECTRAL_FLUX,45,711467,177891,internal_train_only


## Checkpoint H — same-run M04-100 single-block screening

competition-scale fitting을 실행할 때만 `RUN_SINGLE_BLOCK_SCREENING=True`로 바꾼다. E05 reference를 같은 실행에서 먼저 재현하고 fingerprint 계약을 통과시킨 뒤 각 single block을 학습한다. Accuracy target 불일치는 구조 계약 통과 후 검토 대상으로 남긴다.


In [10]:
if not RUN_SINGLE_BLOCK_SCREENING:
    raise RuntimeError(
        'STOP: set only RUN_SINGLE_BLOCK_SCREENING=True in Checkpoint A '
        'when competition-scale fitting is intentionally approved.'
    )
same_run_reference = run_feature_screening_reference(
    prepared_representations['E05_REFERENCE'],
    allow_fit=RUN_SINGLE_BLOCK_SCREENING,
)
reference_reproduction_audit = audit_reference_reproduction(
    same_run_reference, reference_prefit_contract
)
same_run_candidates = {
    block_id: run_feature_screening_candidate(
        prepared_representations[block_id],
        block_id=block_id,
        allow_fit=RUN_SINGLE_BLOCK_SCREENING,
    )
    for block_id in ROUND1_BLOCK_IDS
}
paired_comparisons = pd.DataFrame([
    compare_feature_candidate_to_reference(same_run_reference, same_run_candidates[block_id])
    for block_id in ROUND1_BLOCK_IDS
])
display(pd.DataFrame([reference_reproduction_audit]))
display(paired_comparisons)


,fingerprint_contract_passed,accuracy,accuracy_target,accuracy_delta_from_target,accuracy_target_match,accuracy_is_reproduction_target_not_gate,status
0,True,0.859532,0.859532,-3.941740e-08,True,True,REPRODUCTION_TARGET_MATCH


,reference_id,candidate_id,feature_block,total_dimension,train_rows,validation_rows,accuracy,accuracy_delta,confusion_matrix,recall_speaker_0,...,recall_speaker_1_delta,correct_count_delta,reference_wrong_to_candidate_correct,reference_correct_to_candidate_wrong,both_correct,both_wrong,prediction_disagreement_count,prediction_disagreement_rate,runtime_seconds,automatic_promotion
0,AF00_E05_M04_ITER100_REFERENCE,AF01_ENERGY_DYNAMICS,AF01_ENERGY_DYNAMICS,45,711467,177891,0.857148,-0.002383,"[[74156, 11195], [14217, 78323]]",0.868836,...,-0.053674,-424,9051,9475,143428,15937,18526,0.104142,149.861396,False
1,AF00_E05_M04_ITER100_REFERENCE,AF02_PITCH_VOICING,AF02_PITCH_VOICING,47,711467,177891,0.866266,0.006734,"[[70215, 15136], [8654, 83886]]",0.822662,...,0.006440,1198,5347,4149,148754,19641,9496,0.053381,150.621334,False
2,AF00_E05_M04_ITER100_REFERENCE,AF03_NORMALIZED_SPECTRAL_FLUX,AF03_NORMALIZED_SPECTRAL_FLUX,45,711467,177891,0.853821,-0.005711,"[[74429, 10922], [15082, 77458]]",0.872034,...,-0.063021,-1016,9026,10042,142861,15962,19068,0.107189,149.687161,False


## Checkpoint I — switch 없는 diagnostics와 사용자 판단용 STOP

이미 생성된 같은-run 결과에서 confusion matrix, speaker별 Recall, duration/overlap/confidence slice, 고신뢰 오류와 paired prediction 변화를 출력한다. 이 셀은 결과를 승격하지 않는다.


In [11]:
reference_diagnostics = build_result_diagnostics(
    same_run_reference, full_training_tracking=e05_cache.tracking
)
candidate_diagnostics = {
    block_id: build_result_diagnostics(
        same_run_candidates[block_id], full_training_tracking=e05_cache.tracking
    )
    for block_id in ROUND1_BLOCK_IDS
}
diagnostic_summary = pd.DataFrame([
    {
        'representation_id': result.experiment_id,
        'accuracy': result.metrics['accuracy'],
        'confusion_matrix': result.metrics['confusion_matrix'],
        'recall_speaker_0': result.metrics['recall_speaker_0'],
        'recall_speaker_1': result.metrics['recall_speaker_1'],
        'high_confidence_error_count': report['high_confidence_error_count'],
    }
    for result, report in [(same_run_reference, reference_diagnostics)] + [
        (same_run_candidates[block_id], candidate_diagnostics[block_id])
        for block_id in ROUND1_BLOCK_IDS
    ]
])
display(diagnostic_summary)
display(paired_comparisons)
print('STOP: 결과 해석, candidate 선택, 다음 실험 결정은 사용자가 수행한다.')


,representation_id,accuracy,confusion_matrix,recall_speaker_0,recall_speaker_1,high_confidence_error_count
0,AF00_E05_M04_ITER100_REFERENCE,0.859532,"[[69613, 15738], [9250, 83290]]",0.815608,0.900043,2900
1,AF01_ENERGY_DYNAMICS,0.857148,"[[74156, 11195], [14217, 78323]]",0.868836,0.846369,2814
2,AF02_PITCH_VOICING,0.866266,"[[70215, 15136], [8654, 83886]]",0.822662,0.906484,2580
3,AF03_NORMALIZED_SPECTRAL_FLUX,0.853821,"[[74429, 10922], [15082, 77458]]",0.872034,0.837022,2781


,reference_id,candidate_id,feature_block,total_dimension,train_rows,validation_rows,accuracy,accuracy_delta,confusion_matrix,recall_speaker_0,...,recall_speaker_1_delta,correct_count_delta,reference_wrong_to_candidate_correct,reference_correct_to_candidate_wrong,both_correct,both_wrong,prediction_disagreement_count,prediction_disagreement_rate,runtime_seconds,automatic_promotion
0,AF00_E05_M04_ITER100_REFERENCE,AF01_ENERGY_DYNAMICS,AF01_ENERGY_DYNAMICS,45,711467,177891,0.857148,-0.002383,"[[74156, 11195], [14217, 78323]]",0.868836,...,-0.053674,-424,9051,9475,143428,15937,18526,0.104142,149.861396,False
1,AF00_E05_M04_ITER100_REFERENCE,AF02_PITCH_VOICING,AF02_PITCH_VOICING,47,711467,177891,0.866266,0.006734,"[[70215, 15136], [8654, 83886]]",0.822662,...,0.006440,1198,5347,4149,148754,19641,9496,0.053381,150.621334,False
2,AF00_E05_M04_ITER100_REFERENCE,AF03_NORMALIZED_SPECTRAL_FLUX,AF03_NORMALIZED_SPECTRAL_FLUX,45,711467,177891,0.853821,-0.005711,"[[74429, 10922], [15082, 77458]]",0.872034,...,-0.063021,-1016,9026,10042,142861,15962,19068,0.107189,149.687161,False


STOP: 결과 해석, candidate 선택, 다음 실험 결정은 사용자가 수행한다.


E05 42D에 새 acoustic block을 단독 추가한 결과, AF02 Pitch/Voicing만 internal Accuracy를 0.859532 → 0.866266 (+0.006734)으로 개선했다. Speaker 0/1 Recall이 모두 개선되고 high-confidence error도 320건 감소했다. AF01/AF03은 Accuracy 하락 및 Speaker 1 Recall 악화로 승격하지 않는다. 따라서 E05 42D + AF02 5D = 47D를 다음 단계의 custom acoustic reference candidate로 고정한다. Final model 확정은 아니다.

## Round 2 — Error-analysis-driven feature hypotheses

현재 reference는 **E05 42D + AF02 Pitch/Voicing 5D = 47D**다. Round 2는 이 47D를 고정하고 후보를 한 번에 하나만 추가한다.

- **AF04 CPP (2D)**: remaining overlap/noise/high-confidence error에서 AF02가 직접 표현하지 않는 voice quality를 겨냥한다. Overlap을 해결한다고 가정하지 않으며, MFCC·AF02와 일부 중복될 수 있다.
- **AF05 Delta-MFCC Dynamics (2D)**: 반복적으로 약했던 0.5–1.0초 구간을 겨냥한다. 현재 MFCC40 mean/std는 frame order를 버리므로, MFCC 1..19의 명시적 delta 크기 mean/std를 측정한다. 이는 AF03의 normalized linear-spectrum flux와 다른 정의다.
- **AF06은 retired**: telephone distortion, content dependence, tracking failure, MFCC redundancy 위험이 커 이번 구현·screening 대상에서 제외한다.

비교는 `AF02_47D_REFERENCE` 대 `47D + AF04 = 49D`, `47D + AF05 = 49D` 두 개뿐이다. AF04+AF05 combination은 구현하거나 실행하지 않는다.


### Round 2 cache — legacy AF02 재사용 + AF04/AF05만 신규 추출

기존 Round 1 cache fingerprint를 그대로 검증해 AF02를 재사용한다. 새 artifact는 AF04/AF05 4D만 담으며, 없을 때만 `RUN_FEATURE_EXTRACTION=True`로 이 셀을 다시 실행한다.


In [ ]:
round1_feature_frame = load_candidate_feature_cache(
    manifest=utterance_manifest, cache_root=CANDIDATE_CACHE_ROOT, split='Training',
    block_ids=ROUND1_BLOCK_IDS,
)
round2_parquet_path, round2_metadata_path = candidate_cache_paths(
    CANDIDATE_CACHE_ROOT, split='Training', block_ids=ROUND2_BLOCK_IDS
)
if round2_parquet_path.is_file() and round2_metadata_path.is_file():
    round2_feature_frame = load_candidate_feature_cache(
        manifest=utterance_manifest, cache_root=CANDIDATE_CACHE_ROOT, split='Training',
        block_ids=ROUND2_BLOCK_IDS,
    )
elif round2_parquet_path.exists() or round2_metadata_path.exists():
    raise RuntimeError('STOP: partial Round 2 artifact exists; inspect it instead of rebuilding')
elif not RUN_FEATURE_EXTRACTION:
    raise RuntimeError(
        'STOP: Round 2 cache is absent. Set only RUN_FEATURE_EXTRACTION=True in setup, '
        'rerun this cell once, then restore it to False.'
    )
else:
    build_candidate_feature_cache(
        manifest=utterance_manifest, data_root=RAW_DATA_ROOT,
        cache_root=CANDIDATE_CACHE_ROOT, split='Training',
        allow_full_extraction=RUN_FEATURE_EXTRACTION, block_ids=ROUND2_BLOCK_IDS,
        on_call_done=lambda done, total, call_id: print(done, '/', total, call_id) if done % 500 == 0 else None,
    )
    round2_feature_frame = load_candidate_feature_cache(
        manifest=utterance_manifest, cache_root=CANDIDATE_CACHE_ROOT, split='Training',
        block_ids=ROUND2_BLOCK_IDS,
    )
print('Round 2 cache:', round2_feature_frame.shape, round2_parquet_path)


In [ ]:
if not np.array_equal(round1_feature_frame['record_key'], round2_feature_frame['record_key']):
    raise RuntimeError('STOP: Round 1/2 record_key ordering mismatch')
if not np.array_equal(round1_feature_frame['call_id'], round2_feature_frame['call_id']):
    raise RuntimeError('STOP: Round 1/2 call_id ordering mismatch')
round2_feature_names = tuple(
    name for block_id in ROUND2_BLOCK_IDS for name in BLOCK_BY_ID[block_id].feature_names
)
candidate_feature_frame_round2 = round1_feature_frame.copy()
for feature_name in round2_feature_names:
    candidate_feature_frame_round2[feature_name] = round2_feature_frame[feature_name].to_numpy(dtype=np.float32)
prepared_round2 = prepare_round2_single_block_screening(
    X_e05=e05_cache.validated.X, y=e05_cache.validated.y,
    record_keys=e05_cache.validated.record_keys,
    call_ids=e05_cache.tracking['call_id'].astype(str).to_numpy(),
    candidate_frame=candidate_feature_frame_round2,
    expected_split_contract=EXPECTED_INTERNAL_SPLIT,
)
round2_prefit_contract = build_reference_contract(prepared_round2[ROUND2_REFERENCE_ID])
display(pd.DataFrame([{
    'representation_id': key, 'feature_dimension': len(value.feature_names),
    'train_rows': len(value.y_train), 'validation_rows': len(value.y_evaluation),
} for key, value in prepared_round2.items()]))


In [ ]:
if not RUN_SINGLE_BLOCK_SCREENING:
    raise RuntimeError(
        'STOP: set only RUN_SINGLE_BLOCK_SCREENING=True when Round 2 fitting is approved.'
    )
round2_same_run_reference = run_round2_feature_screening_reference(
    prepared_round2[ROUND2_REFERENCE_ID], allow_fit=RUN_SINGLE_BLOCK_SCREENING
)
round2_reference_audit = audit_reference_reproduction(
    round2_same_run_reference, round2_prefit_contract,
    accuracy_target=ROUND2_M04_ITER100_ACCURACY_TARGET,
)
round2_same_run_candidates = {
    block_id: run_round2_feature_screening_candidate(
        prepared_round2[block_id], block_id=block_id,
        allow_fit=RUN_SINGLE_BLOCK_SCREENING,
    )
    for block_id in ROUND2_BLOCK_IDS
}
round2_paired_comparisons = pd.DataFrame([
    compare_feature_candidate_to_reference(round2_same_run_reference, round2_same_run_candidates[block_id])
    for block_id in ROUND2_BLOCK_IDS
])
display(pd.DataFrame([round2_reference_audit]))
display(round2_paired_comparisons)


In [ ]:
round2_reference_diagnostics = build_result_diagnostics(
    round2_same_run_reference, full_training_tracking=e05_cache.tracking
)
round2_candidate_diagnostics = {
    block_id: build_result_diagnostics(
        round2_same_run_candidates[block_id], full_training_tracking=e05_cache.tracking
    )
    for block_id in ROUND2_BLOCK_IDS
}
display(round2_paired_comparisons)
for diagnostic_id, report in {ROUND2_REFERENCE_ID: round2_reference_diagnostics, **round2_candidate_diagnostics}.items():
    print(diagnostic_id)
    display(report['duration_summary'])
    display(report['overlap_summary'])
    display(report['confidence_summary'])
    print('high-confidence errors:', report['high_confidence_error_count'])
print('STOP: Round 2 결과 해석과 다음 실험 결정은 사용자가 수행한다.')


### 사용자 판단 체크리스트

1. E05 reference의 fingerprint 계약이 먼저 통과했는가?
2. `0.859532`와 차이가 있다면 구조가 아니라 runtime/환경 차이인지 검토했는가?
3. candidate의 Accuracy뿐 아니라 speaker별 Recall, confusion matrix, paired 전환, duration/overlap/confidence slice를 확인했는가?
4. AF01 gain sensitivity, AF02 pitch failure, AF03 transient/channel sensitivity가 shortcut으로 보이지 않는가?
5. 결과를 자동 승격하지 말고 여기서 멈춰 다음 단계를 직접 결정한다.


In [16]:
# Optional historical 47D -> 49D residual analysis (no extraction or fitting)
has_historical_round2_results = (
    'round2_same_run_reference' in globals()
    and 'round2_same_run_candidates' in globals()
    and 'AF04_CPP' in round2_same_run_candidates
)
if has_historical_round2_results:
    frozen_49d_result = round2_same_run_candidates['AF04_CPP']
    frozen_49d_residual = build_frozen_49d_residual_analysis(
        round2_same_run_reference,
        frozen_49d_result,
        full_training_tracking=e05_cache.tracking,
    )
    display(pd.DataFrame([frozen_49d_residual['global_metrics']]))
    display(frozen_49d_residual['fixed_duration_summary'])
    display(frozen_49d_residual['overlap_summary'])
    display(frozen_49d_residual['confidence_summary'])
    display(pd.DataFrame([frozen_49d_residual['paired_transitions']]))
    display(frozen_49d_residual['error_group_summary'])
    display(frozen_49d_residual['error_group_overlap'])
    display(frozen_49d_residual['error_ledger'].head(100))
else:
    print(
        'SKIP: historical 47D/49D same-run prediction objects are unavailable. '
        'AF07/AF08/AF09 extraction and screening can continue independently.'
    )

SKIP: historical 47D/49D same-run prediction objects are unavailable. AF07/AF08/AF09 extraction and screening can continue independently.


In [4]:
# 1. Candidate hypotheses and exact implemented definitions
feature_hypothesis_pool = candidate_hypothesis_table()
next_candidate_definitions = implemented_candidate_definition_table()

display(feature_hypothesis_pool)
display(feature_hypothesis_pool.loc[feature_hypothesis_pool['status'] == 'IMPLEMENT'])
display(next_candidate_definitions)
print('Frozen prefix dimension:', len(FROZEN_49D_FEATURE_NAMES))
print('Independent candidates:', NEXT_CANDIDATE_BLOCK_IDS)

,candidate_id,status,residual_target,missing_information_axis,difference_from_49d,supporting_evidence,counter_evidence_uncertainty,short_8khz_stability,redundancy_risk,cost,testable_expectation
0,AF07_MFCC_CROSS_CORRELATION,IMPLEMENT,overlap persistent errors; high-confidence errors,MFCC cross-coefficient dependence,"49D keeps marginal MFCC mean/std, not covariance",overlap remains weak after AF02+AF04,noise can mimic changed covariance,fixed 8 kHz frontend; fallback unless active_f...,"moderate with MFCC moments, but cross terms ar...",medium,incremental overlap gain without no-overlap co...
1,AF08_COMPETING_PITCH,IMPLEMENT,overlap persistent and high-confidence errors,periodicity peak ambiguity,"AF02 stores the winning lag, not competitor st...",mixed periodic sources are plausible in overlap,noise and octave errors also narrow margins,65-400 Hz lag range; 3-frame fallback,moderate with AF02/CPP; ambiguity itself is new,high,paired corrections concentrate in voiced overl...
2,AF09_MULTIBAND_ENVELOPE_COHERENCE,IMPLEMENT,overlap and noisy high-confidence errors,coarse-band temporal coherence,49D has no cross-band envelope correlation,overlap remains weak while static spectrum is ...,stationary noise and channel filtering can dom...,four fixed Nyquist-safe bands; 5-frame fallback,low-to-moderate with AF07; coarse energy envel...,medium,overlap slice improves without broad class-rec...
3,AF05_DELTA_MFCC_DYNAMICS,DROP,0.5-1.0 second errors,MFCC derivative magnitude,frame-order derivative summary,original short-utterance hypothesis,fixed-duration result did not support the hypo...,implemented fallback but empirically unsupported,high with MFCC moments,medium,not reopened without new evidence
4,AF06_FORMANT_CENTERS,RETIRED,spectral-envelope role differences,formant locations,explicit resonance centers,possible vocal-tract information,telephone/noise tracking failure and content d...,insufficient,high with MFCC envelope,medium,retired
5,HNR_HARMONICITY,EXCLUDE,overlap/noise,harmonic-to-noise balance,explicit harmonic ratio,voice-quality relevance,AF02 periodicity and AF04 CPP already cover it,noise-sensitive,high,medium,excluded as redundant
6,MFCC_DIRECTIONAL_DELTA,EXCLUDE,short utterances,trajectory direction/curvature,direction rather than derivative magnitude,MFCC order is otherwise discarded,too close to failed AF05 and unstable when short,weak,high with AF05 family,medium,excluded pending contrary evidence
7,SPECTRAL_LOW_RANK,EXCLUDE,overlap,spectrogram component rank,joint time-frequency subspace,mixtures can increase component rank,substantially duplicates AF07 covariance struc...,rank is sample-size-sensitive,high with AF07,medium,excluded to avoid repackaging
8,WAVEFORM_CORRUPTION_RATES,HOLD,high-confidence errors,clipping/dropout artifacts,sample-level corruption structure,degradation can create confident mistakes,no current residual count links errors to corr...,stable,low but shortcut risk is high,low,hold until diagnostic evidence exists
9,MODULATION_SPECTRUM,EXCLUDE,speaking rhythm,slow envelope modulation,long-range rhythm,potential role-related cadence,0.5-1.0 second rows are too short for stable e...,insufficient for target slice,moderate with envelope dynamics,medium,excluded for short-window instability


,candidate_id,status,residual_target,missing_information_axis,difference_from_49d,supporting_evidence,counter_evidence_uncertainty,short_8khz_stability,redundancy_risk,cost,testable_expectation
0,AF07_MFCC_CROSS_CORRELATION,IMPLEMENT,overlap persistent errors; high-confidence errors,MFCC cross-coefficient dependence,"49D keeps marginal MFCC mean/std, not covariance",overlap remains weak after AF02+AF04,noise can mimic changed covariance,fixed 8 kHz frontend; fallback unless active_f...,"moderate with MFCC moments, but cross terms ar...",medium,incremental overlap gain without no-overlap co...
1,AF08_COMPETING_PITCH,IMPLEMENT,overlap persistent and high-confidence errors,periodicity peak ambiguity,"AF02 stores the winning lag, not competitor st...",mixed periodic sources are plausible in overlap,noise and octave errors also narrow margins,65-400 Hz lag range; 3-frame fallback,moderate with AF02/CPP; ambiguity itself is new,high,paired corrections concentrate in voiced overl...
2,AF09_MULTIBAND_ENVELOPE_COHERENCE,IMPLEMENT,overlap and noisy high-confidence errors,coarse-band temporal coherence,49D has no cross-band envelope correlation,overlap remains weak while static spectrum is ...,stationary noise and channel filtering can dom...,four fixed Nyquist-safe bands; 5-frame fallback,low-to-moderate with AF07; coarse energy envel...,medium,overlap slice improves without broad class-rec...


,candidate_id,feature_names,dimension,frame_length,hop_length,window,normalization,minimum_valid_frames,short_fallback,silence_behavior,non_finite_behavior,output_dtype
0,AF07_MFCC_CROSS_CORRELATION,"(mfcc_cross_corr_abs_mean, mfcc_cross_corr_pc1...",2,512,160,Hann,active frames; MFCC 1..19 coefficient-wise zer...,p_eff + 1,all-zero 2D block,all-zero 2D block,raise ValueError before extraction/cache,float32
1,AF08_COMPETING_PITCH,"(pitch_peak_margin_median, pitch_peak_margin_iqr)",2,1024,160,Hann,lag-wise autocorrelation divided by overlappin...,3,all-zero 2D block,all-zero 2D block,raise ValueError before extraction/cache,float32
2,AF09_MULTIBAND_ENVELOPE_COHERENCE,"(subband_envelope_corr_abs_mean, subband_envel...",3,512,160,Hann,log band power; each varying band zero mean an...,5,all-zero 3D block,all-zero 3D block,raise ValueError before extraction/cache,float32


Frozen prefix dimension: 49
Independent candidates: ('AF07_MFCC_CROSS_CORRELATION', 'AF08_COMPETING_PITCH', 'AF09_MULTIBAND_ENVELOPE_COHERENCE')


In [5]:
# 2. Execute full AF07/AF08/AF09 extraction and create the immutable cache
next_candidate_paths = build_candidate_feature_cache(
    manifest=utterance_manifest,
    data_root=RAW_DATA_ROOT,
    cache_root=CANDIDATE_CACHE_ROOT,
    split='Training',
    allow_full_extraction=True,
    block_ids=NEXT_CANDIDATE_BLOCK_IDS,
    on_call_done=(
        lambda done, total, call_id:
        print(done, '/', total, call_id) if done % 500 == 0 else None
    ),
)
print('Created immutable candidate cache:')
for cache_path in next_candidate_paths:
    print(cache_path)

FileExistsError: candidate cache already exists; validate and reuse it

In [19]:
# 3. Validate caches and prepare frozen 49D plus each independent candidate
legacy_af02 = load_candidate_feature_cache(
    manifest=utterance_manifest,
    cache_root=CANDIDATE_CACHE_ROOT,
    split='Training',
    block_ids=ROUND1_BLOCK_IDS,
)
legacy_af04 = load_candidate_feature_cache(
    manifest=utterance_manifest,
    cache_root=CANDIDATE_CACHE_ROOT,
    split='Training',
    block_ids=ROUND2_BLOCK_IDS,
)
next_features = load_candidate_feature_cache(
    manifest=utterance_manifest,
    cache_root=CANDIDATE_CACHE_ROOT,
    split='Training',
    block_ids=NEXT_CANDIDATE_BLOCK_IDS,
)

next_feature_names = tuple(
    feature_name
    for block_id in NEXT_CANDIDATE_BLOCK_IDS
    for feature_name in BLOCK_BY_ID[block_id].feature_names
)
next_cache_values = next_features.loc[:, next_feature_names].to_numpy(dtype=np.float32)
next_cache_audit = pd.DataFrame([
    {
        'cache_paths': tuple(str(path) for path in next_candidate_paths),
        'rows': len(next_features),
        'feature_dimension': len(next_feature_names),
        'record_key_unique': next_features['record_key'].astype(str).is_unique,
        'finite_float32': (
            next_cache_values.dtype == np.float32
            and np.isfinite(next_cache_values).all()
        ),
    }
])

combined_next_features = legacy_af02.loc[
    :,
    ['record_key', 'call_id', *BLOCK_BY_ID['AF02_PITCH_VOICING'].feature_names],
].copy()
for source, block_ids in (
    (legacy_af04, ('AF04_CPP',)),
    (next_features, NEXT_CANDIDATE_BLOCK_IDS),
):
    if not np.array_equal(
        combined_next_features['record_key'].astype(str),
        source['record_key'].astype(str),
    ):
        raise RuntimeError('STOP: candidate cache record_key ordering mismatch')
    if not np.array_equal(
        combined_next_features['call_id'].astype(str),
        source['call_id'].astype(str),
    ):
        raise RuntimeError('STOP: candidate cache call_id ordering mismatch')
    for block_id in block_ids:
        for feature_name in BLOCK_BY_ID[block_id].feature_names:
            combined_next_features[feature_name] = source[feature_name].to_numpy(
                dtype=np.float32
            )

prepared_next = prepare_frozen_49d_single_block_screening(
    X_e05=e05_cache.validated.X,
    y=e05_cache.validated.y,
    record_keys=e05_cache.validated.record_keys,
    call_ids=e05_cache.tracking['call_id'].astype(str).to_numpy(),
    candidate_frame=combined_next_features,
    expected_split_contract=EXPECTED_INTERNAL_SPLIT,
)
prepared_next_audit = pd.DataFrame([
    {
        'representation_id': representation_id,
        'dimension': len(payload.feature_names),
        'train_rows': payload.X_train_scaled.shape[0],
        'validation_rows': payload.X_evaluation_scaled.shape[0],
        'frozen_49d_prefix_preserved': (
            tuple(payload.feature_names[:49]) == FROZEN_49D_FEATURE_NAMES
        ),
    }
    for representation_id, payload in prepared_next.items()
])

display(next_cache_audit)
display(prepared_next_audit)

,cache_paths,rows,feature_dimension,record_key_unique,finite_float32
0,(C:\Users\YW\Study\02_Projects\DCC_2026\prelim...,889358,7,True,True


,representation_id,dimension,train_rows,validation_rows,frozen_49d_prefix_preserved
0,AF02_AF04_49D_REFERENCE,49,711467,177891,True
1,AF07_MFCC_CROSS_CORRELATION,51,711467,177891,True
2,AF08_COMPETING_PITCH,51,711467,177891,True
3,AF09_MULTIBAND_ENVELOPE_COHERENCE,52,711467,177891,True


In [20]:
# 4. Fit the same-run 49D reference and each 49D-plus-one candidate
next_reference = run_frozen_49d_screening_reference(
    prepared_next[FROZEN_49D_REFERENCE_ID],
    allow_fit=True,
)
next_results = {
    block_id: run_frozen_49d_screening_candidate(
        prepared_next[block_id],
        block_id=block_id,
        allow_fit=True,
    )
    for block_id in NEXT_CANDIDATE_BLOCK_IDS
}
print('Completed representations:', (FROZEN_49D_REFERENCE_ID, *NEXT_CANDIDATE_BLOCK_IDS))

Completed representations: ('AF02_AF04_49D_REFERENCE', 'AF07_MFCC_CROSS_CORRELATION', 'AF08_COMPETING_PITCH', 'AF09_MULTIBAND_ENVELOPE_COHERENCE')


In [21]:
# 5. Compare candidates and print residual diagnostics from this same run
next_comparisons = pd.DataFrame([
    compare_feature_candidate_to_reference(next_reference, next_results[block_id])
    for block_id in NEXT_CANDIDATE_BLOCK_IDS
])
next_diagnostics = {
    FROZEN_49D_REFERENCE_ID: build_result_diagnostics(
        next_reference,
        full_training_tracking=e05_cache.tracking,
    ),
    **{
        block_id: build_result_diagnostics(
            next_results[block_id],
            full_training_tracking=e05_cache.tracking,
        )
        for block_id in NEXT_CANDIDATE_BLOCK_IDS
    },
}

display(pd.DataFrame([{
    'representation_id': FROZEN_49D_REFERENCE_ID,
    **next_reference.metrics,
}]))
display(next_comparisons)
for representation_id, report in next_diagnostics.items():
    print(representation_id)
    display(report['duration_summary'])
    display(report['overlap_summary'])
    display(report['confidence_summary'])
    print('high-confidence errors:', report['high_confidence_error_count'])

,representation_id,sample_count,accuracy,confusion_matrix,recall_0,recall_1,recall_speaker_0,recall_speaker_1,correct_count
0,AF02_AF04_49D_REFERENCE,177891,0.868627,"[[70576, 14775], [8595, 83945]]",0.826891,0.907121,0.826891,0.907121,154521


,reference_id,candidate_id,feature_block,total_dimension,train_rows,validation_rows,accuracy,accuracy_delta,confusion_matrix,recall_speaker_0,...,recall_speaker_1_delta,correct_count_delta,reference_wrong_to_candidate_correct,reference_correct_to_candidate_wrong,both_correct,both_wrong,prediction_disagreement_count,prediction_disagreement_rate,runtime_seconds,automatic_promotion
0,AF02_AF04_49D_M04_ITER100_REFERENCE,AF07_MFCC_CROSS_CORRELATION,AF07_MFCC_CROSS_CORRELATION,51,711467,177891,0.869128,0.000500,"[[69705, 15646], [7635, 84905]]",0.816686,...,0.010374,89,3731,3642,150879,19639,7373,0.041447,186.945522,False
1,AF02_AF04_49D_M04_ITER100_REFERENCE,AF08_COMPETING_PITCH,AF08_COMPETING_PITCH,51,711467,177891,0.868403,-0.000225,"[[69598, 15753], [7657, 84883]]",0.815433,...,0.010136,-40,3601,3641,150880,19769,7242,0.040710,182.486998,False
2,AF02_AF04_49D_M04_ITER100_REFERENCE,AF09_MULTIBAND_ENVELOPE_COHERENCE,AF09_MULTIBAND_ENVELOPE_COHERENCE,52,711467,177891,0.868970,0.000343,"[[70926, 14425], [8884, 83656]]",0.830992,...,-0.003123,61,3397,3336,151185,19973,6733,0.037849,174.039988,False


AF02_AF04_49D_REFERENCE


,slice,count,accuracy
0,q25_q75,88589,0.850704
1,q75_q95,35967,0.928573
2,q05_q25,35746,0.831673
3,q00_q05,8687,0.864280
4,q95_q100,8902,0.957425


,slice,count,accuracy
0,no_overlap,99310,0.901480
1,overlap,78581,0.827108


...

high-confidence errors: 2835
AF09_MULTIBAND_ENVELOPE_COHERENCE


,slice,count,accuracy
0,q25_q75,88589,0.851404
1,q75_q95,35967,0.927294
2,q05_q25,35746,0.832009
3,q00_q05,8687,0.866697
4,q95_q100,8902,0.958773


,slice,count,accuracy
0,no_overlap,99310,0.901762
1,overlap,78581,0.827528


,slice,count,accuracy
0,>=0.90,109669,0.976338
1,0.60-0.70,15756,0.635187
2,0.70-0.80,13722,0.745081
3,0.80-0.90,20121,0.851797
4,<0.60,18623,0.544327


high-confidence errors: 2595


Frozen 49D reference:
- Accuracy = 0.868627
- Recall speaker 0(119대원) = 0.826891
- Recall speaker 1(신고자) = 0.907121

AF07_MFCC_CROSS_CORRELATION:
- Accuracy = 0.869128
- ΔAccuracy = +0.000500
- correct_count_delta = +89
- Recall 0 = 0.816686 → reference보다 하락
- Recall 1은 약 +0.010374 상승
- 전체 Accuracy는 소폭 올랐지만 speaker 0 성능을 희생하고 speaker 1 쪽으로 분류 경계가 이동했을 가능성이 있음
- 현재는 SELECT가 아니라 HOLD

AF08_COMPETING_PITCH:
- Accuracy = 0.868403
- ΔAccuracy = -0.000225
- correct_count_delta = -40
- 전체 성능이 악화됨
- 현재 가설은 지지되지 않았으며 DROP 유력

AF09_MULTIBAND_ENVELOPE_COHERENCE:
- Accuracy = 0.868970
- ΔAccuracy = +0.000343
- correct_count_delta = +61
- Recall 0 = 0.830992 → reference보다 상승
- Recall 1 delta = -0.003123
- AF07과 반대로 speaker 0을 개선하고 speaker 1을 일부 희생하는 방향
- 현재는 HOLD이며 AF07과 상보성이 있는지 확인 필요

중요한 관찰:
- AF07은 3731개를 고쳤지만 3642개를 새로 틀려 net +89
- AF08은 3601개를 고쳤지만 3641개를 새로 틀려 net -40
- AF09는 3397개를 고쳤지만 3336개를 새로 틀려 net +61

## AF07/AF08/AF09 same-run residual error analysis

아래 셀은 현재 메모리의 `next_reference`, `next_results`, `next_comparisons`, `next_diagnostics`만 사용한다. Feature extraction이나 model fitting은 수행하지 않는다.

In [25]:
# Align the same-run results and build one row-level diagnostic ledger
required_objects = (
    'next_reference',
    'next_results',
    'next_comparisons',
    'next_diagnostics',
)
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(
        'Run the existing AF07/AF08/AF09 screening cells first. '
        f'Missing objects: {missing_objects}'
    )

candidate_label_by_block = {
    'AF07_MFCC_CROSS_CORRELATION': 'AF07',
    'AF08_COMPETING_PITCH': 'AF08',
    'AF09_MULTIBAND_ENVELOPE_COHERENCE': 'AF09',
}
if set(next_results) != set(candidate_label_by_block):
    raise ValueError('next_results candidate set differs from AF07/AF08/AF09')
if set(next_comparisons['candidate_id']) != set(candidate_label_by_block):
    raise ValueError('next_comparisons candidate set differs from AF07/AF08/AF09')

analysis_results = {
    '49D': next_reference,
    **{
        candidate_label_by_block[block_id]: next_results[block_id]
        for block_id in NEXT_CANDIDATE_BLOCK_IDS
    },
}
reference_keys = np.asarray(
    next_reference.fit_audit['validation_record_keys']
).astype(str)
validation_idx = np.asarray(
    next_reference.fit_audit['validation_idx'],
    dtype=np.int64,
)
evaluation_tracking = e05_cache.tracking.iloc[validation_idx].reset_index(drop=True)
tracking_keys = evaluation_tracking['record_key'].astype(str).to_numpy()
if not np.array_equal(tracking_keys, reference_keys):
    raise ValueError('49D result and evaluation tracking record_key order differ')

analysis_truth = np.asarray(next_reference.y_true, dtype=np.int64)
analysis_predictions = {}
analysis_probabilities = {}
analysis_confidence = {}
for representation_id, result in analysis_results.items():
    result_keys = np.asarray(result.fit_audit['validation_record_keys']).astype(str)
    result_truth = np.asarray(result.y_true, dtype=np.int64)
    probabilities = np.asarray(result.probabilities, dtype=np.float64)
    if not np.array_equal(result_keys, reference_keys):
        raise ValueError(f'{representation_id} validation record_key order differs')
    if not np.array_equal(result_truth, analysis_truth):
        raise ValueError(f'{representation_id} validation labels differ')
    if probabilities.shape != (len(analysis_truth), 2):
        raise ValueError(f'{representation_id} probability shape differs')
    if not np.isfinite(probabilities).all() or not np.allclose(
        probabilities.sum(axis=1), 1.0, rtol=1e-6, atol=1e-8
    ):
        raise ValueError(f'{representation_id} probabilities are invalid')
    analysis_predictions[representation_id] = np.asarray(
        result.y_pred, dtype=np.int64
    )
    analysis_probabilities[representation_id] = probabilities
    analysis_confidence[representation_id] = probabilities.max(axis=1)

duration_sec = (
    pd.to_numeric(evaluation_tracking['endAt_ms'], errors='raise').to_numpy(
        dtype=np.float64
    )
    - pd.to_numeric(
        evaluation_tracking['startAt_ms'], errors='raise'
    ).to_numpy(dtype=np.float64)
) / 1_000.0
if not np.isfinite(duration_sec).all() or np.any(duration_sec < 0.0):
    raise ValueError('evaluation duration contains invalid values')

duration_order = ('<0.5s', '0.5-1.0s', '1-2s', '2-5s', '>=5s')
duration_slice = pd.cut(
    duration_sec,
    bins=[-np.inf, 0.5, 1.0, 2.0, 5.0, np.inf],
    labels=duration_order,
    right=False,
).astype(str)

overlap_work = evaluation_tracking.loc[
    :, ['call_id', 'startAt_ms', 'endAt_ms']
].copy()
overlap_work['_row'] = np.arange(len(overlap_work), dtype=np.int64)
overlap_work['startAt_ms'] = pd.to_numeric(
    overlap_work['startAt_ms'], errors='raise'
)
overlap_work['endAt_ms'] = pd.to_numeric(
    overlap_work['endAt_ms'], errors='raise'
)
ordered_overlap = overlap_work.sort_values(
    ['call_id', 'startAt_ms', 'endAt_ms', '_row'], kind='stable'
)
previous_max_end = ordered_overlap.groupby(
    'call_id', sort=False
)['endAt_ms'].transform(lambda values: values.cummax().shift(1))
next_start = ordered_overlap.groupby(
    'call_id', sort=False
)['startAt_ms'].shift(-1)
ordered_overlap['_overlap'] = (
    (ordered_overlap['startAt_ms'] < previous_max_end)
    | (next_start < ordered_overlap['endAt_ms'])
).fillna(False)
analysis_overlap = ordered_overlap.sort_values(
    '_row', kind='stable'
)['_overlap'].to_numpy(dtype=bool)

next_residual_ledger = evaluation_tracking.loc[
    :, ['record_key', 'call_id']
].copy()
next_residual_ledger['y_true'] = analysis_truth
next_residual_ledger['duration_sec'] = duration_sec
next_residual_ledger['duration_slice'] = duration_slice
next_residual_ledger['overlap'] = analysis_overlap

reference_correct = analysis_predictions['49D'] == analysis_truth
for representation_id in ('49D', 'AF07', 'AF08', 'AF09'):
    probabilities = analysis_probabilities[representation_id]
    prediction = analysis_predictions[representation_id]
    next_residual_ledger[f'{representation_id}_prediction'] = prediction
    next_residual_ledger[f'{representation_id}_probability_0'] = probabilities[:, 0]
    next_residual_ledger[f'{representation_id}_probability_1'] = probabilities[:, 1]
    next_residual_ledger[f'{representation_id}_confidence'] = (
        analysis_confidence[representation_id]
    )

for candidate_id in ('AF07', 'AF08', 'AF09'):
    candidate_correct = analysis_predictions[candidate_id] == analysis_truth
    status = np.full(len(analysis_truth), 'both_correct', dtype=object)
    status[(~reference_correct) & candidate_correct] = 'fixed'
    status[reference_correct & (~candidate_correct)] = 'regressed'
    status[(~reference_correct) & (~candidate_correct)] = 'persistent'
    next_residual_ledger[f'{candidate_id}_status'] = status
    next_residual_ledger[f'{candidate_id}_fixed'] = status == 'fixed'
    next_residual_ledger[f'{candidate_id}_regressed'] = status == 'regressed'
    next_residual_ledger[f'{candidate_id}_persistent'] = status == 'persistent'

display(next_comparisons)
display(pd.DataFrame([{
    'rows': len(next_residual_ledger),
    'record_key_unique': next_residual_ledger['record_key'].is_unique,
    'representations': tuple(analysis_results),
    'duration_slices': duration_order,
}]))

,reference_id,candidate_id,feature_block,total_dimension,train_rows,validation_rows,accuracy,accuracy_delta,confusion_matrix,recall_speaker_0,...,recall_speaker_1_delta,correct_count_delta,reference_wrong_to_candidate_correct,reference_correct_to_candidate_wrong,both_correct,both_wrong,prediction_disagreement_count,prediction_disagreement_rate,runtime_seconds,automatic_promotion
0,AF02_AF04_49D_M04_ITER100_REFERENCE,AF07_MFCC_CROSS_CORRELATION,AF07_MFCC_CROSS_CORRELATION,51,711467,177891,0.869128,0.000500,"[[69705, 15646], [7635, 84905]]",0.816686,...,0.010374,89,3731,3642,150879,19639,7373,0.041447,186.945522,False
1,AF02_AF04_49D_M04_ITER100_REFERENCE,AF08_COMPETING_PITCH,AF08_COMPETING_PITCH,51,711467,177891,0.868403,-0.000225,"[[69598, 15753], [7657, 84883]]",0.815433,...,0.010136,-40,3601,3641,150880,19769,7242,0.040710,182.486998,False
2,AF02_AF04_49D_M04_ITER100_REFERENCE,AF09_MULTIBAND_ENVELOPE_COHERENCE,AF09_MULTIBAND_ENVELOPE_COHERENCE,52,711467,177891,0.868970,0.000343,"[[70926, 14425], [8884, 83656]]",0.830992,...,-0.003123,61,3397,3336,151185,19973,6733,0.037849,174.039988,False


,rows,record_key_unique,representations,duration_slices
0,177891,True,"(49D, AF07, AF08, AF09)","(<0.5s, 0.5-1.0s, 1-2s, 2-5s, >=5s)"


In [6]:
# 1. Fixed-duration metrics for 49D and AF07/AF08/AF09
fixed_duration_metric_map = {}
for representation_id, prediction in analysis_predictions.items():
    fixed_duration_metric_map[representation_id] = {}
    for duration_name in duration_order:
        selected = duration_slice == duration_name
        selected_truth = analysis_truth[selected]
        selected_prediction = prediction[selected]
        count = int(np.sum(selected))
        if count == 0:
            metrics = {
                'count': 0,
                'accuracy': np.nan,
                'recall_speaker_0': np.nan,
                'recall_speaker_1': np.nan,
                'confusion_matrix': [[0, 0], [0, 0]],
            }
        else:
            true_0 = selected_truth == 0
            true_1 = selected_truth == 1
            tn = int(np.sum(true_0 & (selected_prediction == 0)))
            fp = int(np.sum(true_0 & (selected_prediction == 1)))
            fn = int(np.sum(true_1 & (selected_prediction == 0)))
            tp = int(np.sum(true_1 & (selected_prediction == 1)))
            metrics = {
                'count': count,
                'accuracy': float(np.mean(selected_truth == selected_prediction)),
                'recall_speaker_0': (
                    float(tn / (tn + fp)) if (tn + fp) else np.nan
                ),
                'recall_speaker_1': (
                    float(tp / (tp + fn)) if (tp + fn) else np.nan
                ),
                'confusion_matrix': [[tn, fp], [fn, tp]],
            }
        fixed_duration_metric_map[representation_id][duration_name] = metrics

fixed_duration_rows = []
for duration_name in duration_order:
    reference_metrics = fixed_duration_metric_map['49D'][duration_name]
    for representation_id in ('49D', 'AF07', 'AF08', 'AF09'):
        metrics = fixed_duration_metric_map[representation_id][duration_name]
        fixed_duration_rows.append({
            'duration_slice': duration_name,
            'representation_id': representation_id,
            'count': metrics['count'],
            'accuracy': metrics['accuracy'],
            'accuracy_delta_vs_49D': (
                metrics['accuracy'] - reference_metrics['accuracy']
            ),
            'recall_speaker_0': metrics['recall_speaker_0'],
            'recall_0_delta_vs_49D': (
                metrics['recall_speaker_0']
                - reference_metrics['recall_speaker_0']
            ),
            'recall_speaker_1': metrics['recall_speaker_1'],
            'recall_1_delta_vs_49D': (
                metrics['recall_speaker_1']
                - reference_metrics['recall_speaker_1']
            ),
            'confusion_matrix': metrics['confusion_matrix'],
        })

next_fixed_duration_summary = pd.DataFrame(fixed_duration_rows)
display(next_fixed_duration_summary)

NameError: name 'analysis_predictions' is not defined

In [7]:
# 2. Paired transitions by speaker, overlap, and fixed-duration slice
transition_order = (
    '49D_wrong_to_candidate_correct',
    '49D_correct_to_candidate_wrong',
    'both_wrong',
    'both_correct',
)
paired_transition_rows = []
transition_slice_specs = (
    (
        'speaker',
        np.where(analysis_truth == 0, 'speaker_0', 'speaker_1'),
        ('speaker_0', 'speaker_1'),
    ),
    (
        'overlap',
        np.where(analysis_overlap, 'overlap', 'no_overlap'),
        ('overlap', 'no_overlap'),
    ),
    ('duration', duration_slice, duration_order),
)

for candidate_id in ('AF07', 'AF08', 'AF09'):
    candidate_correct = analysis_predictions[candidate_id] == analysis_truth
    transition = np.full(len(analysis_truth), 'both_correct', dtype=object)
    transition[(~reference_correct) & candidate_correct] = (
        '49D_wrong_to_candidate_correct'
    )
    transition[reference_correct & (~candidate_correct)] = (
        '49D_correct_to_candidate_wrong'
    )
    transition[(~reference_correct) & (~candidate_correct)] = 'both_wrong'

    for slice_axis, slice_values, slice_order in transition_slice_specs:
        slice_values = np.asarray(slice_values).astype(str)
        for slice_name in slice_order:
            selected = slice_values == slice_name
            slice_count = int(np.sum(selected))
            for transition_name in transition_order:
                transition_count = int(
                    np.sum(selected & (transition == transition_name))
                )
                paired_transition_rows.append({
                    'candidate_id': candidate_id,
                    'slice_axis': slice_axis,
                    'slice': slice_name,
                    'slice_count': slice_count,
                    'transition': transition_name,
                    'count': transition_count,
                    'rate_within_slice': (
                        transition_count / slice_count
                        if slice_count
                        else np.nan
                    ),
                })

next_paired_transition_summary = pd.DataFrame(paired_transition_rows)
display(next_paired_transition_summary)

NameError: name 'analysis_truth' is not defined

In [28]:
# 3. AF07 versus AF09 complementarity on 49D errors and correct rows
af07_correct = analysis_predictions['AF07'] == analysis_truth
af09_correct = analysis_predictions['AF09'] == analysis_truth
complementarity_base = np.where(
    reference_correct, '49D_correct', '49D_wrong'
)
complementarity_group = np.full(len(analysis_truth), '', dtype=object)

reference_wrong = ~reference_correct
complementarity_group[
    reference_wrong & af07_correct & (~af09_correct)
] = 'AF07_only_fixed'
complementarity_group[
    reference_wrong & (~af07_correct) & af09_correct
] = 'AF09_only_fixed'
complementarity_group[
    reference_wrong & af07_correct & af09_correct
] = 'both_fixed'
complementarity_group[
    reference_wrong & (~af07_correct) & (~af09_correct)
] = 'neither_fixed'

complementarity_group[
    reference_correct & (~af07_correct) & af09_correct
] = 'AF07_only_regressed'
complementarity_group[
    reference_correct & af07_correct & (~af09_correct)
] = 'AF09_only_regressed'
complementarity_group[
    reference_correct & (~af07_correct) & (~af09_correct)
] = 'both_regressed'
complementarity_group[
    reference_correct & af07_correct & af09_correct
] = 'both_maintained'

next_residual_ledger['AF07_AF09_base'] = complementarity_base
next_residual_ledger['AF07_AF09_group'] = complementarity_group

complementarity_group_order = {
    '49D_wrong': (
        'AF07_only_fixed',
        'AF09_only_fixed',
        'both_fixed',
        'neither_fixed',
    ),
    '49D_correct': (
        'AF07_only_regressed',
        'AF09_only_regressed',
        'both_regressed',
        'both_maintained',
    ),
}
complementarity_summary_rows = []
complementarity_distribution_rows = []
distribution_specs = (
    (
        'speaker',
        np.where(analysis_truth == 0, 'speaker_0', 'speaker_1'),
        ('speaker_0', 'speaker_1'),
    ),
    ('duration', duration_slice, duration_order),
    (
        'overlap',
        np.where(analysis_overlap, 'overlap', 'no_overlap'),
        ('overlap', 'no_overlap'),
    ),
)

for base_name, group_order in complementarity_group_order.items():
    base_selected = complementarity_base == base_name
    base_count = int(np.sum(base_selected))
    for group_name in group_order:
        group_selected = complementarity_group == group_name
        group_count = int(np.sum(group_selected))
        complementarity_summary_rows.append({
            'base': base_name,
            'group': group_name,
            'base_count': base_count,
            'count': group_count,
            'rate_within_base': (
                group_count / base_count if base_count else np.nan
            ),
        })
        for distribution_axis, values, value_order in distribution_specs:
            values = np.asarray(values).astype(str)
            for distribution_value in value_order:
                distribution_count = int(
                    np.sum(group_selected & (values == distribution_value))
                )
                complementarity_distribution_rows.append({
                    'base': base_name,
                    'group': group_name,
                    'distribution_axis': distribution_axis,
                    'distribution_value': distribution_value,
                    'group_count': group_count,
                    'count': distribution_count,
                    'rate_within_group': (
                        distribution_count / group_count
                        if group_count
                        else np.nan
                    ),
                })

next_af07_af09_complementarity = pd.DataFrame(
    complementarity_summary_rows
)
next_af07_af09_distribution = pd.DataFrame(
    complementarity_distribution_rows
)
display(next_af07_af09_complementarity)
display(next_af07_af09_distribution)

,base,group,base_count,count,rate_within_base
0,49D_wrong,AF07_only_fixed,23370,1856,0.079418
1,49D_wrong,AF09_only_fixed,23370,1522,0.065126
2,49D_wrong,both_fixed,23370,1875,0.080231
3,49D_wrong,neither_fixed,23370,18117,0.775225
4,49D_correct,AF07_only_regressed,154521,2291,0.014826
5,49D_correct,AF09_only_regressed,154521,1985,0.012846
6,49D_correct,both_regressed,154521,1351,0.008743
7,49D_correct,both_maintained,154521,148894,0.963584


,base,group,distribution_axis,distribution_value,group_count,count,rate_within_group
0,49D_wrong,AF07_only_fixed,speaker,speaker_0,1856,505,0.272091
1,49D_wrong,AF07_only_fixed,speaker,speaker_1,1856,1351,0.727909
2,49D_wrong,AF07_only_fixed,duration,<0.5s,1856,488,0.262931
3,49D_wrong,AF07_only_fixed,duration,0.5-1.0s,1856,647,0.348599
4,49D_wrong,AF07_only_fixed,duration,1-2s,1856,417,0.224677
...,...,...,...,...,...,...,...
67,49D_correct,both_maintained,duration,1-2s,148894,47845,0.321336
68,49D_correct,both_maintained,duration,2-5s,148894,45743,0.307219
69,49D_correct,both_maintained,duration,>=5s,148894,10647,0.071507
70,49D_correct,both_maintained,overlap,overlap,148894,61634,0.413945


In [29]:
# 4-5. High-confidence errors and the complete row-level ledger
diagnostic_key_by_representation = {
    '49D': FROZEN_49D_REFERENCE_ID,
    **{
        candidate_label_by_block[block_id]: block_id
        for block_id in NEXT_CANDIDATE_BLOCK_IDS
    },
}
high_confidence_thresholds = {
    representation_id: float(
        next_diagnostics[diagnostic_key]['high_confidence_threshold']
    )
    for representation_id, diagnostic_key
    in diagnostic_key_by_representation.items()
}
if len(set(high_confidence_thresholds.values())) != 1:
    raise ValueError('high-confidence thresholds differ across representations')
high_confidence_threshold = high_confidence_thresholds['49D']

reference_prediction = analysis_predictions['49D']
reference_high_confidence_error = (
    (reference_prediction != analysis_truth)
    & (analysis_confidence['49D'] >= high_confidence_threshold)
)
high_confidence_rows = []
for representation_id in ('49D', 'AF07', 'AF08', 'AF09'):
    prediction = analysis_predictions[representation_id]
    confidence = analysis_confidence[representation_id]
    correct = prediction == analysis_truth
    high_confidence_error = (
        (~correct) & (confidence >= high_confidence_threshold)
    )
    diagnostic_key = diagnostic_key_by_representation[representation_id]
    expected_error_count = int(
        next_diagnostics[diagnostic_key]['high_confidence_error_count']
    )
    if int(np.sum(high_confidence_error)) != expected_error_count:
        raise ValueError(
            f'{representation_id} high-confidence count differs from '
            'next_diagnostics'
        )

    next_residual_ledger[
        f'{representation_id}_high_confidence_error'
    ] = high_confidence_error
    high_confidence_rows.append({
        'representation_id': representation_id,
        'threshold': high_confidence_threshold,
        'high_confidence_error_count': int(np.sum(high_confidence_error)),
        'high_confidence_FP_count': int(np.sum(
            high_confidence_error
            & (analysis_truth == 0)
            & (prediction == 1)
        )),
        'high_confidence_FN_count': int(np.sum(
            high_confidence_error
            & (analysis_truth == 1)
            & (prediction == 0)
        )),
        'reference_high_confidence_error_fixed_count': (
            np.nan
            if representation_id == '49D'
            else int(np.sum(reference_high_confidence_error & correct))
        ),
        'candidate_high_confidence_error_regressed_count': (
            np.nan
            if representation_id == '49D'
            else int(np.sum(reference_correct & high_confidence_error))
        ),
        'candidate_high_confidence_persistent_count': (
            np.nan
            if representation_id == '49D'
            else int(np.sum(
                (~reference_correct) & (~correct) & high_confidence_error
            ))
        ),
    })

next_high_confidence_summary = pd.DataFrame(high_confidence_rows)
display(next_high_confidence_summary)
print('Row-level ledger shape:', next_residual_ledger.shape)
display(next_residual_ledger)

,representation_id,threshold,high_confidence_error_count,high_confidence_FP_count,high_confidence_FN_count,reference_high_confidence_error_fixed_count,candidate_high_confidence_error_regressed_count,candidate_high_confidence_persistent_count
0,49D,0.9,2707,1413,1294,NaN,NaN,NaN
1,AF07,0.9,2774,1493,1281,10.0,19.0,2755.0
2,AF08,0.9,2835,1546,1289,7.0,5.0,2830.0
3,AF09,0.9,2595,1218,1377,14.0,8.0,2587.0


Row-level ledger shape: (177891, 40)


,record_key,call_id,y_true,duration_sec,duration_slice,overlap,49D_prediction,49D_probability_0,49D_probability_1,49D_confidence,...,AF09_status,AF09_fixed,AF09_regressed,AF09_persistent,AF07_AF09_base,AF07_AF09_group,49D_high_confidence_error,AF07_high_confidence_error,AF08_high_confidence_error,AF09_high_confidence_error
0,Training|651e464d69a4f266f0626820_20220101|0,651e464d69a4f266f0626820_20220101,0,1.570,1-2s,False,0,0.995414,0.004586,0.995414,...,both_correct,False,False,False,49D_correct,both_maintained,False,False,False,False
1,Training|651e464d69a4f266f0626820_20220101|1,651e464d69a4f266f0626820_20220101,1,2.798,2-5s,False,1,0.004737,0.995263,0.995263,...,both_correct,False,False,False,49D_correct,both_maintained,False,False,False,False
2,Training|651e464d69a4f266f0626820_20220101|2,651e464d69a4f266f0626820_20220101,0,0.661,0.5-1.0s,True,0,0.992799,0.007201,0.992799,...,both_correct,False,False,False,49D_correct,both_maintained,False,False,False,False
3,Training|651e464d69a4f266f0626820_20220101|3,651e464d69a4f266f0626820_20220101,1,0.537,0.5-1.0s,True,1,0.256988,0.743012,0.743012,...,both_correct,False,False,False,49D_correct,both_maintained,False,False,False,False
4,Training|651e464d69a4f266f0626820_20220101|4,651e464d69a4f266f0626820_20220101,0,0.418,<0.5s,False,0,0.949543,0.050457,0.949543,...,both_correct,False,False,False,49D_correct,both_maintained,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177886,Training|651e5e261da44bb23c5a3fcc_20221204|19,651e5e261da44bb23c5a3fcc_20221204,1,3.510,2-5s,False,0,0.509252,0.490748,0.509252,...,persistent,False,False,True,49D_wrong,neither_fixed,False,False,False,False
177887,Training|651e5e261da44bb23c5a3fcc_20221204|20,651e5e261da44bb23c5a3fcc_20221204,0,2.829,2-5s,False,0,0.990019,0.009981,0.990019,...,both_correct,False,False,False,49D_correct,both_maintained,False,False,False,False
177888,Training|651e5e261da44bb23c5a3fcc_20221204|21,651e5e261da44bb23c5a3fcc_20221204,1,2.162,2-5s,False,0,0.528878,0.471122,0.528878,...,persistent,False,False,True,49D_wrong,neither_fixed,False,False,False,False
177889,Training|651e5e261da44bb23c5a3fcc_20221204|22,651e5e261da44bb23c5a3fcc_20221204,0,1.768,1-2s,False,0,0.999016,0.000984,0.999016,...,both_correct,False,False,False,49D_correct,both_maintained,False,False,False,False


- AF07은 전체 Accuracy는 조금 올랐지만, 119대원 Recall이 크게 떨어지고 신고자 Recall이 올라갔다.
  → 신고자 쪽으로 예측이 치우친 것으로 보인다.
  → 단독 사용은 보류.

- AF08은 Accuracy도 떨어지고 high-confidence error도 늘었다.
  → DROP.

- AF09는 전체 Accuracy 상승폭은 작지만, 모든 duration 구간에서 119대원 Recall이 올라갔다.
  특히 1초 미만 짧은 발화에서 개선이 컸다.
  → 현재 49D의 약점을 보완하는 방향이라 가장 유망하다.

- AF07과 AF09는 서로 다른 오류를 고치는 경우가 많았다.
  → 둘을 같이 넣은 조합 실험은 해볼 가치가 있다.

- AF09는 high-confidence error를 2707 → 2595로 줄였다.
  특히 119대원을 신고자로 강하게 틀리는 FP가 1413 → 1218로 감소했다.

### 현재 판단
- AF07: HOLD
- AF08: DROP
- AF09: SELECT 후보


## Current reproducible 52D AF09 reference vs 54D AF09+AF07

저장된 cache만 사용해 현재 재현 가능한 52D와 54D를 같은 split, train-only StandardScaler, M04 architecture, seed, `max_iter=100`으로 비교한다. 과거 Accuracy `0.868970`은 historical non-reproduced result로만 보존하며 gate에 사용하지 않는다. 결과가 좋아도 54D를 자동 승격하지 않는다.

In [11]:
# 1. Strict-load existing caches and prepare aligned 52D/54D representations
from dcc_data.mission2_acoustic_feature_expansion import (
    CANONICAL_52D_ACCURACY,
    CANONICAL_52D_CONFUSION_MATRIX,
    _prepare_representation,
    _run_prepared_representation,
    resolve_canonical_52d_reference,
)
from dcc_data.mission2_model_experiments import calculate_prediction_metrics

cache_af02 = load_candidate_feature_cache(
    manifest=utterance_manifest,
    cache_root=CANDIDATE_CACHE_ROOT,
    split='Training',
    block_ids=ROUND1_BLOCK_IDS,
)
cache_af04 = load_candidate_feature_cache(
    manifest=utterance_manifest,
    cache_root=CANDIDATE_CACHE_ROOT,
    split='Training',
    block_ids=ROUND2_BLOCK_IDS,
)
cache_af07_af08_af09 = load_candidate_feature_cache(
    manifest=utterance_manifest,
    cache_root=CANDIDATE_CACHE_ROOT,
    split='Training',
    block_ids=NEXT_CANDIDATE_BLOCK_IDS,
)

canonical_record_keys = np.asarray(
    e05_cache.validated.record_keys
).astype(str)
canonical_call_ids = e05_cache.tracking['call_id'].astype(str).to_numpy()
for cache_name, cache_frame in (
    ('AF02 cache', cache_af02),
    ('AF04 cache', cache_af04),
    ('AF07/AF08/AF09 cache', cache_af07_af08_af09),
):
    cache_record_keys = cache_frame['record_key'].astype(str).to_numpy()
    cache_call_ids = cache_frame['call_id'].astype(str).to_numpy()
    if not np.array_equal(cache_record_keys, canonical_record_keys):
        raise ValueError(f'{cache_name} record_key order differs from E05')
    if not np.array_equal(cache_call_ids, canonical_call_ids):
        raise ValueError(f'{cache_name} call_id order differs from E05')

af02_feature_names = BLOCK_BY_ID['AF02_PITCH_VOICING'].feature_names
af04_feature_names = BLOCK_BY_ID['AF04_CPP'].feature_names
af07_feature_names = BLOCK_BY_ID['AF07_MFCC_CROSS_CORRELATION'].feature_names
af09_feature_names = BLOCK_BY_ID[
    'AF09_MULTIBAND_ENVELOPE_COHERENCE'
].feature_names

X_e05 = np.asarray(e05_cache.validated.X, dtype=np.float32)
X_af02 = cache_af02.loc[:, af02_feature_names].to_numpy(dtype=np.float32)
X_af04 = cache_af04.loc[:, af04_feature_names].to_numpy(dtype=np.float32)
X_af07 = cache_af07_af08_af09.loc[
    :, af07_feature_names
].to_numpy(dtype=np.float32)
X_af09 = cache_af07_af08_af09.loc[
    :, af09_feature_names
].to_numpy(dtype=np.float32)

X_frozen_49d = np.concatenate(
    [X_e05, X_af02, X_af04],
    axis=1,
).astype(np.float32, copy=False)
feature_names_52d = (
    *FROZEN_49D_FEATURE_NAMES,
    *af09_feature_names,
)
feature_names_54d = (
    *feature_names_52d,
    *af07_feature_names,
)
X_52d = np.concatenate(
    [X_frozen_49d, X_af09],
    axis=1,
).astype(np.float32, copy=False)
X_54d = np.concatenate(
    [X_52d, X_af07],
    axis=1,
).astype(np.float32, copy=False)

if X_frozen_49d.shape[1] != 49:
    raise AssertionError('frozen 49D matrix dimension changed')
if X_52d.shape[1] != 52 or len(feature_names_52d) != 52:
    raise AssertionError('52D feature dimension/order contract failed')
if X_54d.shape[1] != 54 or len(feature_names_54d) != 54:
    raise AssertionError('54D feature dimension/order contract failed')
if tuple(feature_names_52d[:49]) != FROZEN_49D_FEATURE_NAMES:
    raise AssertionError('52D no longer preserves the frozen 49D prefix')
if tuple(feature_names_54d[:52]) != tuple(feature_names_52d):
    raise AssertionError('54D no longer preserves the exact 52D prefix')
if not np.array_equal(X_54d[:, :52], X_52d):
    raise AssertionError('54D matrix changed the 52D values')
if not np.isfinite(X_52d).all() or not np.isfinite(X_54d).all():
    raise ValueError('52D/54D matrix contains NaN or inf')

labels_52d_54d = np.asarray(e05_cache.validated.y, dtype=np.int64)
train_idx_52d_54d, validation_idx_52d_54d, split_audit_52d_54d = (
    create_fixed_internal_split(
        canonical_call_ids,
        split_function=standalone_group_split_indices,
        expected_contract=EXPECTED_INTERNAL_SPLIT,
    )
)
prepared_52d = _prepare_representation(
    representation_id='AF09_52D_REFERENCE',
    X=X_52d,
    y=labels_52d_54d,
    record_keys=canonical_record_keys,
    feature_names=feature_names_52d,
    train_idx=train_idx_52d_54d,
    validation_idx=validation_idx_52d_54d,
    split_audit=split_audit_52d_54d,
)
prepared_54d = _prepare_representation(
    representation_id='AF09_AF07_54D_CANDIDATE',
    X=X_54d,
    y=labels_52d_54d,
    record_keys=canonical_record_keys,
    feature_names=feature_names_54d,
    train_idx=train_idx_52d_54d,
    validation_idx=validation_idx_52d_54d,
    split_audit=split_audit_52d_54d,
)

if not np.array_equal(prepared_52d.train_idx, prepared_54d.train_idx):
    raise AssertionError('52D/54D train rows differ')
if not np.array_equal(
    prepared_52d.validation_idx,
    prepared_54d.validation_idx,
):
    raise AssertionError('52D/54D validation rows differ')
if not np.array_equal(
    prepared_52d.training_record_keys,
    prepared_54d.training_record_keys,
):
    raise AssertionError('52D/54D training record_key order differs')
if not np.array_equal(
    prepared_52d.evaluation_record_keys,
    prepared_54d.evaluation_record_keys,
):
    raise AssertionError('52D/54D validation record_key order differs')

preparation_52d_54d_audit = pd.DataFrame([
    {
        'representation_id': '52D_AF09_REFERENCE',
        'dimension': len(prepared_52d.feature_names),
        'train_rows': len(prepared_52d.train_idx),
        'validation_rows': len(prepared_52d.validation_idx),
        'scaler_fit_rows': int(prepared_52d.scaler.n_samples_seen_),
        'scaler_policy': 'internal_train_only',
        'frozen_prefix_preserved': (
            tuple(prepared_52d.feature_names[:49])
            == FROZEN_49D_FEATURE_NAMES
        ),
    },
    {
        'representation_id': '54D_AF09_AF07_CANDIDATE',
        'dimension': len(prepared_54d.feature_names),
        'train_rows': len(prepared_54d.train_idx),
        'validation_rows': len(prepared_54d.validation_idx),
        'scaler_fit_rows': int(prepared_54d.scaler.n_samples_seen_),
        'scaler_policy': 'internal_train_only',
        'frozen_prefix_preserved': (
            tuple(prepared_54d.feature_names[:52])
            == tuple(feature_names_52d)
        ),
    },
])
display(preparation_52d_54d_audit)

,representation_id,dimension,train_rows,validation_rows,scaler_fit_rows,scaler_policy,frozen_prefix_preserved
0,52D_AF09_REFERENCE,52,711467,177891,711467,internal_train_only,True
1,54D_AF09_AF07_CANDIDATE,54,711467,177891,711467,internal_train_only,True


In [12]:
# 2. Reuse only a canonical 52D object; stale or missing objects are refit.
existing_result_52d = globals().get('result_52d')
canonical_52d_resolution = resolve_canonical_52d_reference(
    prepared_52d,
    existing_result=existing_result_52d,
    canonical_matrix=X_52d,
    canonical_labels=labels_52d_54d,
    canonical_record_keys=canonical_record_keys,
    canonical_train_idx=train_idx_52d_54d,
    canonical_validation_idx=validation_idx_52d_54d,
    expected_split_contract=EXPECTED_INTERNAL_SPLIT,
    allow_refit=True,
)
result_52d = canonical_52d_resolution.result

if tuple(result_52d.feature_names) != tuple(prepared_52d.feature_names):
    raise ValueError('current result_52d feature names/order differ')
if result_52d.train_shape != prepared_52d.X_train_scaled.shape:
    raise ValueError('current result_52d training shape differs')
if result_52d.evaluation_shape != prepared_52d.X_evaluation_scaled.shape:
    raise ValueError('current result_52d validation shape differs')
for audit_name, prepared_values in (
    ('train_idx', prepared_52d.train_idx),
    ('validation_idx', prepared_52d.validation_idx),
    ('training_record_keys', prepared_52d.training_record_keys),
    ('validation_record_keys', prepared_52d.evaluation_record_keys),
):
    if not np.array_equal(
        result_52d.fit_audit[audit_name],
        prepared_values,
    ):
        raise ValueError(
            f'current result_52d {audit_name} differs from prepared_52d'
        )
if not np.array_equal(result_52d.y_true, prepared_52d.y_evaluation):
    raise ValueError('current result_52d validation labels differ')
if dict(result_52d.model_config) != dict(
    M04_ITER100_FEATURE_SCREENING_CONFIG
):
    raise ValueError('current result_52d model configuration differs')
if result_52d.fit_audit['seed'] != int(
    M04_ITER100_FEATURE_SCREENING_CONFIG['random_state']
):
    raise ValueError('current result_52d seed differs')
for scaler_attribute in ('mean_', 'scale_'):
    if not np.array_equal(
        getattr(result_52d.scaler, scaler_attribute),
        getattr(prepared_52d.scaler, scaler_attribute),
    ):
        raise ValueError(
            f'current result_52d scaler {scaler_attribute} differs'
        )

current_52d_reference_report = pd.DataFrame([{
    'representation_id': result_52d.experiment_id,
    'comparison_reference': 'current_reproducible_52D',
    'accuracy': float(result_52d.metrics['accuracy']),
    'historical_AF09_accuracy': 0.868970,
    'historical_status': 'historical_non_reproduced_result',
    'historical_accuracy_used_as_gate': False,
    'max_iter': result_52d.model_config['max_iter'],
    'seed': result_52d.fit_audit['seed'],
    'train_rows': result_52d.train_shape[0],
    'validation_rows': result_52d.evaluation_shape[0],
    'scaler_fit_scope': result_52d.fit_audit['scaler_fit_scope'],
}])
display(current_52d_reference_report)

,representation_id,comparison_reference,accuracy,historical_AF09_accuracy,historical_status,historical_accuracy_used_as_gate,max_iter,seed,train_rows,validation_rows,scaler_fit_scope
0,AF09_52D_M04_ITER100_REFERENCE,current_reproducible_52D,0.869532,0.86897,historical_non_reproduced_result,False,100,42,711467,177891,internal_train_only


In [13]:
# 3. Fit 54D from the validated current 52D comparison reference
required_52d_54d_objects = (
    'result_52d',
    'prepared_52d',
    'prepared_54d',
)
missing_52d_54d_objects = [
    name for name in required_52d_54d_objects if name not in globals()
]
if missing_52d_54d_objects:
    raise RuntimeError(
        'Run the cache preparation and current 52D reference cell first. '
        f'Missing objects: {missing_52d_54d_objects}'
    )
if tuple(result_52d.feature_names) != tuple(prepared_52d.feature_names):
    raise ValueError('current 52D feature names/order differ')
if tuple(prepared_54d.feature_names[:52]) != tuple(
    prepared_52d.feature_names
):
    raise ValueError('54D does not preserve the exact current 52D prefix')
if not np.array_equal(result_52d.y_true, prepared_52d.y_evaluation):
    raise ValueError('current result_52d validation labels differ')
if result_52d.fit_audit['scaler_fit_scope'] != 'internal_train_only':
    raise ValueError('current result_52d scaler policy differs')
for scaler_attribute in ('mean_', 'scale_'):
    if not np.array_equal(
        getattr(result_52d.scaler, scaler_attribute),
        getattr(prepared_52d.scaler, scaler_attribute),
    ):
        raise ValueError(
            f'current result_52d scaler {scaler_attribute} differs'
        )
for prepared_attribute in (
    'train_idx',
    'validation_idx',
    'training_record_keys',
    'evaluation_record_keys',
):
    if not np.array_equal(
        getattr(prepared_52d, prepared_attribute),
        getattr(prepared_54d, prepared_attribute),
    ):
        raise ValueError(
            f'52D/54D prepared {prepared_attribute} differs'
        )
for audit_name, prepared_values in (
    ('train_idx', prepared_52d.train_idx),
    ('validation_idx', prepared_52d.validation_idx),
    ('training_record_keys', prepared_52d.training_record_keys),
    ('validation_record_keys', prepared_52d.evaluation_record_keys),
):
    if not np.array_equal(
        result_52d.fit_audit[audit_name],
        prepared_values,
    ):
        raise ValueError(
            f'current result_52d {audit_name} differs from prepared_52d'
        )
if dict(result_52d.model_config) != dict(
    M04_ITER100_FEATURE_SCREENING_CONFIG
):
    raise ValueError('current result_52d model configuration differs')
if result_52d.fit_audit['seed'] != int(
    M04_ITER100_FEATURE_SCREENING_CONFIG['random_state']
):
    raise ValueError('current result_52d seed differs')

result_54d = _run_prepared_representation(
    prepared_54d,
    experiment_id='AF09_AF07_54D_M04_ITER100_CANDIDATE',
    allow_fit=True,
)
print(
    'Completed current-reference comparison models:',
    result_52d.experiment_id,
    result_54d.experiment_id,
)

Completed current-reference comparison models: AF09_52D_M04_ITER100_REFERENCE AF09_AF07_54D_M04_ITER100_CANDIDATE


In [14]:
# 4. Core same-run 52D versus 54D comparison and paired transitions
required_comparison_objects = ('result_52d', 'result_54d')
missing_comparison_objects = [
    name for name in required_comparison_objects if name not in globals()
]
if missing_comparison_objects:
    raise RuntimeError(
        'Run the current 52D reference and 54D fit cells first. '
        f'Missing objects: {missing_comparison_objects}'
    )

if dict(result_52d.model_config) != dict(result_54d.model_config):
    raise ValueError('52D/54D model configurations differ')
for audit_name in (
    'train_idx',
    'validation_idx',
    'training_record_keys',
    'validation_record_keys',
):
    if not np.array_equal(
        result_52d.fit_audit[audit_name],
        result_54d.fit_audit[audit_name],
    ):
        raise ValueError(f'52D/54D {audit_name} alignment differs')
if not np.array_equal(result_52d.y_true, result_54d.y_true):
    raise ValueError('52D/54D validation labels differ')

truth_52d_54d = np.asarray(result_52d.y_true, dtype=np.int64)
prediction_52d = np.asarray(result_52d.y_pred, dtype=np.int64)
prediction_54d = np.asarray(result_54d.y_pred, dtype=np.int64)
correct_52d = prediction_52d == truth_52d_54d
correct_54d = prediction_54d == truth_52d_54d

comparison_52d_54d = pd.DataFrame([
    {
        'representation_id': '52D_AF09_REFERENCE',
        'dimension': 52,
        'accuracy': float(result_52d.metrics['accuracy']),
        'accuracy_delta_vs_52D': 0.0,
        'confusion_matrix': result_52d.metrics['confusion_matrix'],
        'recall_speaker_0': float(
            result_52d.metrics['recall_speaker_0']
        ),
        'recall_0_delta_vs_52D': 0.0,
        'recall_speaker_1': float(
            result_52d.metrics['recall_speaker_1']
        ),
        'recall_1_delta_vs_52D': 0.0,
        'correct_count': int(result_52d.metrics['correct_count']),
        'correct_count_delta_vs_52D': 0,
        '52D_wrong_to_54D_correct': np.nan,
        '52D_correct_to_54D_wrong': np.nan,
        'automatic_promotion': False,
    },
    {
        'representation_id': '54D_AF09_AF07_CANDIDATE',
        'dimension': 54,
        'accuracy': float(result_54d.metrics['accuracy']),
        'accuracy_delta_vs_52D': float(
            result_54d.metrics['accuracy']
            - result_52d.metrics['accuracy']
        ),
        'confusion_matrix': result_54d.metrics['confusion_matrix'],
        'recall_speaker_0': float(
            result_54d.metrics['recall_speaker_0']
        ),
        'recall_0_delta_vs_52D': float(
            result_54d.metrics['recall_speaker_0']
            - result_52d.metrics['recall_speaker_0']
        ),
        'recall_speaker_1': float(
            result_54d.metrics['recall_speaker_1']
        ),
        'recall_1_delta_vs_52D': float(
            result_54d.metrics['recall_speaker_1']
            - result_52d.metrics['recall_speaker_1']
        ),
        'correct_count': int(result_54d.metrics['correct_count']),
        'correct_count_delta_vs_52D': int(
            result_54d.metrics['correct_count']
            - result_52d.metrics['correct_count']
        ),
        '52D_wrong_to_54D_correct': int(np.sum(
            (~correct_52d) & correct_54d
        )),
        '52D_correct_to_54D_wrong': int(np.sum(
            correct_52d & (~correct_54d)
        )),
        'automatic_promotion': False,
    },
])
display(comparison_52d_54d)

,representation_id,dimension,accuracy,accuracy_delta_vs_52D,confusion_matrix,recall_speaker_0,recall_0_delta_vs_52D,recall_speaker_1,recall_1_delta_vs_52D,correct_count,correct_count_delta_vs_52D,52D_wrong_to_54D_correct,52D_correct_to_54D_wrong,automatic_promotion
0,52D_AF09_REFERENCE,52,0.869532,0.000000,"[[71066, 14285], [8924, 83616]]",0.832632,0.000000,0.903566,0.000000,154682,0,NaN,NaN,False
1,54D_AF09_AF07_CANDIDATE,54,0.869639,0.000107,"[[70011, 15340], [7850, 84690]]",0.820272,-0.012361,0.915172,0.011606,154701,19,3699.0,3680.0,False


In [15]:
# 5. Fixed-duration performance with deltas against current 52D
required_duration_objects = (
    'truth_52d_54d',
    'prediction_52d',
    'prediction_54d',
    'result_52d',
)
missing_duration_objects = [
    name for name in required_duration_objects if name not in globals()
]
if missing_duration_objects:
    raise RuntimeError(
        'Run the 52D/54D core comparison cell first. '
        f'Missing objects: {missing_duration_objects}'
    )

evaluation_tracking_52d_54d = e05_cache.tracking.iloc[
    result_52d.fit_audit['validation_idx']
].reset_index(drop=True)
evaluation_keys_52d_54d = evaluation_tracking_52d_54d[
    'record_key'
].astype(str).to_numpy()
if not np.array_equal(
    evaluation_keys_52d_54d,
    np.asarray(
        result_52d.fit_audit['validation_record_keys']
    ).astype(str),
):
    raise ValueError('diagnostic tracking and 52D record_key order differ')

duration_sec_52d_54d = (
    pd.to_numeric(
        evaluation_tracking_52d_54d['endAt_ms'],
        errors='raise',
    ).to_numpy(dtype=np.float64)
    - pd.to_numeric(
        evaluation_tracking_52d_54d['startAt_ms'],
        errors='raise',
    ).to_numpy(dtype=np.float64)
) / 1_000.0
duration_order_52d_54d = (
    '<0.5s',
    '0.5-1.0s',
    '1-2s',
    '2-5s',
    '>=5s',
)
duration_slice_52d_54d = pd.cut(
    duration_sec_52d_54d,
    bins=[-np.inf, 0.5, 1.0, 2.0, 5.0, np.inf],
    labels=duration_order_52d_54d,
    right=False,
).astype(str)

duration_metric_map_52d_54d = {}
for representation_id, prediction in (
    ('52D', prediction_52d),
    ('54D', prediction_54d),
):
    duration_metric_map_52d_54d[representation_id] = {}
    for duration_name in duration_order_52d_54d:
        selected = duration_slice_52d_54d == duration_name
        if not np.any(selected):
            metrics = {
                'sample_count': 0,
                'accuracy': np.nan,
                'confusion_matrix': [[0, 0], [0, 0]],
                'recall_speaker_0': np.nan,
                'recall_speaker_1': np.nan,
                'correct_count': 0,
            }
        else:
            metrics = calculate_prediction_metrics(
                truth_52d_54d[selected],
                prediction[selected],
            )
        duration_metric_map_52d_54d[
            representation_id
        ][duration_name] = metrics

fixed_duration_rows_52d_54d = []
for duration_name in duration_order_52d_54d:
    reference_metrics = duration_metric_map_52d_54d[
        '52D'
    ][duration_name]
    for representation_id in ('52D', '54D'):
        metrics = duration_metric_map_52d_54d[
            representation_id
        ][duration_name]
        fixed_duration_rows_52d_54d.append({
            'duration_slice': duration_name,
            'representation_id': representation_id,
            'count': int(metrics['sample_count']),
            'accuracy': metrics['accuracy'],
            'accuracy_delta_vs_52D': (
                metrics['accuracy'] - reference_metrics['accuracy']
            ),
            'confusion_matrix': metrics['confusion_matrix'],
            'recall_speaker_0': metrics['recall_speaker_0'],
            'recall_0_delta_vs_52D': (
                metrics['recall_speaker_0']
                - reference_metrics['recall_speaker_0']
            ),
            'recall_speaker_1': metrics['recall_speaker_1'],
            'recall_1_delta_vs_52D': (
                metrics['recall_speaker_1']
                - reference_metrics['recall_speaker_1']
            ),
        })

fixed_duration_52d_54d = pd.DataFrame(
    fixed_duration_rows_52d_54d
)
display(fixed_duration_52d_54d)

,duration_slice,representation_id,count,accuracy,accuracy_delta_vs_52D,confusion_matrix,recall_speaker_0,recall_0_delta_vs_52D,recall_speaker_1,recall_1_delta_vs_52D
0,<0.5s,52D,14076,0.863527,0.000000,"[[5963, 765], [1156, 6192]]",0.886296,0.000000,0.842678,0.000000
1,<0.5s,54D,14076,0.862390,-0.001137,"[[5605, 1123], [814, 6534]]",0.833086,-0.053210,0.889222,0.046543
2,0.5-1.0s,52D,43199,0.825598,0.000000,"[[14958, 4635], [2899, 20707]]",0.763436,0.000000,0.877192,0.000000
3,0.5-1.0s,54D,43199,0.826177,0.000579,"[[14438, 5155], [2354, 21252]]",0.736896,-0.026540,0.900280,0.023087
4,1-2s,52D,58320,0.847154,0.000000,"[[25813, 6219], [2695, 23593]]",0.805850,0.000000,0.897482,0.000000
5,1-2s,54D,58320,0.847222,0.000069,"[[25692, 6340], [2570, 23718]]",0.802073,-0.003777,0.902237,0.004755
6,2-5s,52D,51015,0.915221,0.000000,"[[20972, 2457], [1868, 25718]]",0.895130,0.000000,0.932284,0.000000
7,2-5s,54D,51015,0.914986,-0.000235,"[[20931, 2498], [1839, 25747]]",0.893380,-0.001750,0.933336,0.001051
8,>=5s,52D,11281,0.954348,0.000000,"[[3360, 209], [306, 7406]]",0.941440,0.000000,0.960322,0.000000
9,>=5s,54D,11281,0.955944,0.001596,"[[3345, 224], [273, 7439]]",0.937237,-0.004203,0.964601,0.004279


In [16]:
# 6. Overlap/no-overlap and high-confidence FP/FN diagnostics
required_overlap_confidence_objects = (
    'evaluation_tracking_52d_54d',
    'truth_52d_54d',
    'prediction_52d',
    'prediction_54d',
    'correct_52d',
    'correct_54d',
    'result_52d',
    'result_54d',
)
missing_overlap_confidence_objects = [
    name
    for name in required_overlap_confidence_objects
    if name not in globals()
]
if missing_overlap_confidence_objects:
    raise RuntimeError(
        'Run the core comparison and fixed-duration cells first. '
        f'Missing objects: {missing_overlap_confidence_objects}'
    )

overlap_work_52d_54d = evaluation_tracking_52d_54d.loc[
    :, ['call_id', 'startAt_ms', 'endAt_ms']
].copy()
overlap_work_52d_54d['_row'] = np.arange(
    len(overlap_work_52d_54d),
    dtype=np.int64,
)
overlap_work_52d_54d['startAt_ms'] = pd.to_numeric(
    overlap_work_52d_54d['startAt_ms'],
    errors='raise',
)
overlap_work_52d_54d['endAt_ms'] = pd.to_numeric(
    overlap_work_52d_54d['endAt_ms'],
    errors='raise',
)
ordered_overlap_52d_54d = overlap_work_52d_54d.sort_values(
    ['call_id', 'startAt_ms', 'endAt_ms', '_row'],
    kind='stable',
)
previous_max_end_52d_54d = ordered_overlap_52d_54d.groupby(
    'call_id',
    sort=False,
)['endAt_ms'].transform(lambda values: values.cummax().shift(1))
next_start_52d_54d = ordered_overlap_52d_54d.groupby(
    'call_id',
    sort=False,
)['startAt_ms'].shift(-1)
ordered_overlap_52d_54d['_overlap'] = (
    (
        ordered_overlap_52d_54d['startAt_ms']
        < previous_max_end_52d_54d
    )
    | (
        next_start_52d_54d
        < ordered_overlap_52d_54d['endAt_ms']
    )
).fillna(False)
overlap_mask_52d_54d = ordered_overlap_52d_54d.sort_values(
    '_row',
    kind='stable',
)['_overlap'].to_numpy(dtype=bool)

overlap_metric_map_52d_54d = {}
for representation_id, prediction in (
    ('52D', prediction_52d),
    ('54D', prediction_54d),
):
    overlap_metric_map_52d_54d[representation_id] = {}
    for slice_name, selected in (
        ('overlap', overlap_mask_52d_54d),
        ('no_overlap', ~overlap_mask_52d_54d),
    ):
        overlap_metric_map_52d_54d[
            representation_id
        ][slice_name] = calculate_prediction_metrics(
            truth_52d_54d[selected],
            prediction[selected],
        )

overlap_rows_52d_54d = []
for slice_name in ('overlap', 'no_overlap'):
    reference_metrics = overlap_metric_map_52d_54d[
        '52D'
    ][slice_name]
    for representation_id in ('52D', '54D'):
        metrics = overlap_metric_map_52d_54d[
            representation_id
        ][slice_name]
        overlap_rows_52d_54d.append({
            'slice': slice_name,
            'representation_id': representation_id,
            'count': int(metrics['sample_count']),
            'accuracy': metrics['accuracy'],
            'accuracy_delta_vs_52D': (
                metrics['accuracy'] - reference_metrics['accuracy']
            ),
            'confusion_matrix': metrics['confusion_matrix'],
            'recall_speaker_0': metrics['recall_speaker_0'],
            'recall_0_delta_vs_52D': (
                metrics['recall_speaker_0']
                - reference_metrics['recall_speaker_0']
            ),
            'recall_speaker_1': metrics['recall_speaker_1'],
            'recall_1_delta_vs_52D': (
                metrics['recall_speaker_1']
                - reference_metrics['recall_speaker_1']
            ),
        })

overlap_52d_54d = pd.DataFrame(overlap_rows_52d_54d)

HIGH_CONFIDENCE_THRESHOLD_52D_54D = 0.90
probabilities_52d = np.asarray(
    result_52d.probabilities,
    dtype=np.float64,
)
probabilities_54d = np.asarray(
    result_54d.probabilities,
    dtype=np.float64,
)
confidence_52d = probabilities_52d.max(axis=1)
confidence_54d = probabilities_54d.max(axis=1)
high_confidence_error_52d = (
    (~correct_52d)
    & (confidence_52d >= HIGH_CONFIDENCE_THRESHOLD_52D_54D)
)
high_confidence_error_54d = (
    (~correct_54d)
    & (confidence_54d >= HIGH_CONFIDENCE_THRESHOLD_52D_54D)
)

high_confidence_52d_54d = pd.DataFrame([
    {
        'representation_id': '52D',
        'threshold': HIGH_CONFIDENCE_THRESHOLD_52D_54D,
        'high_confidence_error_count': int(np.sum(
            high_confidence_error_52d
        )),
        'high_confidence_FP_count': int(np.sum(
            high_confidence_error_52d
            & (truth_52d_54d == 0)
            & (prediction_52d == 1)
        )),
        'high_confidence_FN_count': int(np.sum(
            high_confidence_error_52d
            & (truth_52d_54d == 1)
            & (prediction_52d == 0)
        )),
        '52D_high_confidence_error_fixed_by_54D': np.nan,
        '54D_high_confidence_regression_from_52D': np.nan,
    },
    {
        'representation_id': '54D',
        'threshold': HIGH_CONFIDENCE_THRESHOLD_52D_54D,
        'high_confidence_error_count': int(np.sum(
            high_confidence_error_54d
        )),
        'high_confidence_FP_count': int(np.sum(
            high_confidence_error_54d
            & (truth_52d_54d == 0)
            & (prediction_54d == 1)
        )),
        'high_confidence_FN_count': int(np.sum(
            high_confidence_error_54d
            & (truth_52d_54d == 1)
            & (prediction_54d == 0)
        )),
        '52D_high_confidence_error_fixed_by_54D': int(np.sum(
            high_confidence_error_52d & correct_54d
        )),
        '54D_high_confidence_regression_from_52D': int(np.sum(
            correct_52d & high_confidence_error_54d
        )),
    },
])

display(overlap_52d_54d)
display(high_confidence_52d_54d)
print('Automatic reference promotion: disabled; user decision required.')

,slice,representation_id,count,accuracy,accuracy_delta_vs_52D,confusion_matrix,recall_speaker_0,recall_0_delta_vs_52D,recall_speaker_1,recall_1_delta_vs_52D
0,overlap,52D,78581,0.827846,0.000000,"[[30704, 7721], [5807, 34349]]",0.799063,0.000000,0.855389,0.000000
1,overlap,54D,78581,0.827770,-0.000076,"[[30069, 8356], [5178, 34978]]",0.782537,-0.016526,0.871053,0.015664
2,no_overlap,52D,99310,0.902517,0.000000,"[[40362, 6564], [3117, 49267]]",0.860120,0.000000,0.940497,0.000000
3,no_overlap,54D,99310,0.902769,0.000252,"[[39942, 6984], [2672, 49712]]",0.851170,-0.008950,0.948992,0.008495


,representation_id,threshold,high_confidence_error_count,high_confidence_FP_count,high_confidence_FN_count,52D_high_confidence_error_fixed_by_54D,54D_high_confidence_regression_from_52D
0,52D,0.9,2590,1223,1367,NaN,NaN
1,54D,0.9,2647,1366,1281,6.0,11.0


Automatic reference promotion: disabled; user decision required.


In [17]:
# Self-contained diagnosis of the historical AF09 path versus current 52D
import numpy as np
import pandas as pd

from dcc_data.mission2_acoustic_feature_expansion import (
    BLOCK_BY_ID,
    FROZEN_49D_FEATURE_NAMES,
    NEXT_CANDIDATE_BLOCK_IDS,
    ROUND1_BLOCK_IDS,
    ROUND2_BLOCK_IDS,
    assemble_single_block_matrix,
    load_candidate_feature_cache,
    prepare_frozen_49d_single_block_screening,
    run_frozen_49d_screening_candidate,
)
from dcc_data.mission2_model_experiments import EXPECTED_INTERNAL_SPLIT

AF09_DIAGNOSTIC_BLOCK_ID = 'AF09_MULTIBAND_ENVELOPE_COHERENCE'
HISTORICAL_AF09_ACCURACY = 0.868970
ACCURACY_ATOL = 5e-7

required_diagnostic_objects = (
    'prepared_52d',
    'result_52d',
    'e05_cache',
    'utterance_manifest',
    'CANDIDATE_CACHE_ROOT',
)
missing_diagnostic_objects = [
    name for name in required_diagnostic_objects if name not in globals()
]
if missing_diagnostic_objects:
    raise RuntimeError(
        'Run the existing 52D preparation/reproduction cells first. '
        f'Missing objects: {missing_diagnostic_objects}'
    )


def _as_python_scalar(value):
    return value.item() if isinstance(value, np.generic) else value


def _sequence_detail(item, legacy_values, current_values):
    legacy = np.asarray(legacy_values)
    current = np.asarray(current_values)
    shape_equal = legacy.shape == current.shape
    if shape_equal:
        mismatch_positions = np.flatnonzero(legacy != current)
        difference_count = int(len(mismatch_positions))
        first_position = (
            int(mismatch_positions[0]) if difference_count else None
        )
    else:
        difference_count = np.nan
        first_position = None
    return {
        'item': item,
        'legacy_shape': legacy.shape,
        'current_shape': current.shape,
        'array_equal': bool(
            shape_equal and difference_count == 0
        ),
        'difference_count': difference_count,
        'first_difference_index': first_position,
        'legacy_first_value': (
            _as_python_scalar(legacy[first_position])
            if first_position is not None else None
        ),
        'current_first_value': (
            _as_python_scalar(current[first_position])
            if first_position is not None else None
        ),
    }


def _numeric_detail(item, legacy_values, current_values):
    legacy = np.asarray(legacy_values)
    current = np.asarray(current_values)
    shape_equal = legacy.shape == current.shape
    if not shape_equal:
        return {
            'item': item,
            'legacy_shape': legacy.shape,
            'current_shape': current.shape,
            'legacy_dtype': str(legacy.dtype),
            'current_dtype': str(current.dtype),
            'array_equal': False,
            'allclose_rtol_1e-5_atol_1e-8': False,
            'difference_count': np.nan,
            'max_absolute_difference': np.nan,
            'first_difference_index': None,
            'max_difference_index': None,
            'legacy_at_max_difference': None,
            'current_at_max_difference': None,
        }

    legacy_flat = legacy.reshape(-1)
    current_flat = current.reshape(-1)
    chunk_size = 1_000_000
    array_equal = True
    allclose = True
    difference_count = 0
    first_difference_flat = None
    max_difference_flat = None
    max_absolute_difference = 0.0

    for start in range(0, legacy_flat.size, chunk_size):
        stop = min(start + chunk_size, legacy_flat.size)
        legacy_chunk = legacy_flat[start:stop]
        current_chunk = current_flat[start:stop]
        exact_mismatch = legacy_chunk != current_chunk
        chunk_difference_count = int(np.count_nonzero(exact_mismatch))
        if chunk_difference_count:
            array_equal = False
            difference_count += chunk_difference_count
            if first_difference_flat is None:
                first_difference_flat = (
                    start + int(np.flatnonzero(exact_mismatch)[0])
                )
        if not np.all(np.isclose(
            legacy_chunk,
            current_chunk,
            rtol=1e-5,
            atol=1e-8,
            equal_nan=False,
        )):
            allclose = False

        absolute_difference = np.abs(
            legacy_chunk.astype(np.float64)
            - current_chunk.astype(np.float64)
        )
        if absolute_difference.size:
            local_index = int(np.argmax(absolute_difference))
            local_max = float(absolute_difference[local_index])
            if local_max > max_absolute_difference:
                max_absolute_difference = local_max
                max_difference_flat = start + local_index

    def _index(flat_index):
        if flat_index is None:
            return None
        return tuple(
            int(index)
            for index in np.unravel_index(flat_index, legacy.shape)
        )

    max_difference_index = _index(max_difference_flat)
    return {
        'item': item,
        'legacy_shape': legacy.shape,
        'current_shape': current.shape,
        'legacy_dtype': str(legacy.dtype),
        'current_dtype': str(current.dtype),
        'array_equal': array_equal,
        'allclose_rtol_1e-5_atol_1e-8': allclose,
        'difference_count': difference_count,
        'max_absolute_difference': max_absolute_difference,
        'first_difference_index': _index(first_difference_flat),
        'max_difference_index': max_difference_index,
        'legacy_at_max_difference': (
            _as_python_scalar(legacy[max_difference_index])
            if max_difference_index is not None else None
        ),
        'current_at_max_difference': (
            _as_python_scalar(current[max_difference_index])
            if max_difference_index is not None else None
        ),
    }


def _emit_first_difference(
    stage,
    details,
    *,
    fit_executed,
    rerun_result=None,
):
    rerun_accuracy = (
        float(rerun_result.metrics['accuracy'])
        if rerun_result is not None else np.nan
    )
    summary = pd.DataFrame([{
        'first_mismatch_stage': stage,
        'historical_AF09_accuracy': HISTORICAL_AF09_ACCURACY,
        'current_52D_accuracy': float(result_52d.metrics['accuracy']),
        'AF09_runner_rerun_accuracy': rerun_accuracy,
        'AF09_runner_fit_executed': fit_executed,
    }])
    display(summary)
    if details is not None and not details.empty:
        display(details)
    if not fit_executed:
        print(
            'AF09 runner fit skipped because preparation already differs.'
        )
    print('AF07/AF08/49D/54D were not fitted by this cell.')
    return {
        'summary': summary,
        'details': details,
        'af09_runner_rerun': rerun_result,
    }


def _run_self_contained_af09_52d_diagnostic():
    cache_af02_diagnostic = load_candidate_feature_cache(
        manifest=utterance_manifest,
        cache_root=CANDIDATE_CACHE_ROOT,
        split='Training',
        block_ids=ROUND1_BLOCK_IDS,
    )
    cache_af04_diagnostic = load_candidate_feature_cache(
        manifest=utterance_manifest,
        cache_root=CANDIDATE_CACHE_ROOT,
        split='Training',
        block_ids=ROUND2_BLOCK_IDS,
    )
    cache_af09_diagnostic = load_candidate_feature_cache(
        manifest=utterance_manifest,
        cache_root=CANDIDATE_CACHE_ROOT,
        split='Training',
        block_ids=NEXT_CANDIDATE_BLOCK_IDS,
    )

    canonical_record_keys = np.asarray(
        e05_cache.validated.record_keys
    ).astype(str)
    canonical_call_ids = (
        e05_cache.tracking['call_id'].astype(str).to_numpy()
    )
    cache_alignment = pd.DataFrame([
        _sequence_detail(
            f'{cache_name}.record_key',
            canonical_record_keys,
            cache_frame['record_key'].astype(str).to_numpy(),
        )
        for cache_name, cache_frame in (
            ('AF02_cache', cache_af02_diagnostic),
            ('AF04_cache', cache_af04_diagnostic),
            ('AF09_cache', cache_af09_diagnostic),
        )
    ] + [
        _sequence_detail(
            f'{cache_name}.call_id',
            canonical_call_ids,
            cache_frame['call_id'].astype(str).to_numpy(),
        )
        for cache_name, cache_frame in (
            ('AF02_cache', cache_af02_diagnostic),
            ('AF04_cache', cache_af04_diagnostic),
            ('AF09_cache', cache_af09_diagnostic),
        )
    ])
    if not bool(cache_alignment['array_equal'].all()):
        return _emit_first_difference(
            'cache_row_alignment',
            cache_alignment.loc[
                ~cache_alignment['array_equal']
            ].reset_index(drop=True),
            fit_executed=False,
        )

    af02_names = BLOCK_BY_ID['AF02_PITCH_VOICING'].feature_names
    af04_names = BLOCK_BY_ID['AF04_CPP'].feature_names
    af09_names = BLOCK_BY_ID[
        AF09_DIAGNOSTIC_BLOCK_ID
    ].feature_names

    legacy_candidate_frame = cache_af02_diagnostic.loc[
        :, ['record_key', 'call_id', *af02_names]
    ].copy()
    for source_frame, feature_names in (
        (cache_af04_diagnostic, af04_names),
        (cache_af09_diagnostic, af09_names),
    ):
        for feature_name in feature_names:
            legacy_candidate_frame[feature_name] = source_frame[
                feature_name
            ].to_numpy(dtype=np.float32)

    legacy_prepared_map = prepare_frozen_49d_single_block_screening(
        X_e05=e05_cache.validated.X,
        y=e05_cache.validated.y,
        record_keys=e05_cache.validated.record_keys,
        call_ids=canonical_call_ids,
        candidate_frame=legacy_candidate_frame,
        expected_split_contract=EXPECTED_INTERNAL_SPLIT,
        block_ids=(AF09_DIAGNOSTIC_BLOCK_ID,),
    )
    legacy_prepared_af09 = legacy_prepared_map[
        AF09_DIAGNOSTIC_BLOCK_ID
    ]

    split_details = pd.DataFrame([
        _numeric_detail(
            'train_idx',
            legacy_prepared_af09.train_idx,
            prepared_52d.train_idx,
        ),
        _numeric_detail(
            'validation_idx',
            legacy_prepared_af09.validation_idx,
            prepared_52d.validation_idx,
        ),
    ])
    if not bool(split_details['array_equal'].all()):
        return _emit_first_difference(
            'split_indices',
            split_details.loc[
                ~split_details['array_equal']
            ].reset_index(drop=True),
            fit_executed=False,
        )

    record_key_details = pd.DataFrame([
        _sequence_detail(
            'training_record_keys',
            legacy_prepared_af09.training_record_keys.astype(str),
            prepared_52d.training_record_keys.astype(str),
        ),
        _sequence_detail(
            'validation_record_keys',
            legacy_prepared_af09.evaluation_record_keys.astype(str),
            prepared_52d.evaluation_record_keys.astype(str),
        ),
    ])
    if not bool(record_key_details['array_equal'].all()):
        return _emit_first_difference(
            'record_key_alignment',
            record_key_details.loc[
                ~record_key_details['array_equal']
            ].reset_index(drop=True),
            fit_executed=False,
        )

    feature_details = pd.DataFrame([
        _sequence_detail(
            'feature_names_and_order',
            tuple(legacy_prepared_af09.feature_names),
            tuple(prepared_52d.feature_names),
        )
    ])
    if not bool(feature_details['array_equal'].all()):
        return _emit_first_difference(
            'feature_names_or_order',
            feature_details,
            fit_executed=False,
        )

    raw_matrix_rows = [{
        'item': 'raw_E05_42D_shared_input',
        'legacy_shape': e05_cache.validated.X.shape,
        'current_shape': e05_cache.validated.X.shape,
        'legacy_dtype': str(e05_cache.validated.X.dtype),
        'current_dtype': str(e05_cache.validated.X.dtype),
        'array_equal': True,
        'allclose_rtol_1e-5_atol_1e-8': True,
        'difference_count': 0,
        'max_absolute_difference': 0.0,
        'first_difference_index': None,
        'max_difference_index': None,
        'legacy_at_max_difference': None,
        'current_at_max_difference': None,
    }]
    current_block_sources = {
        'AF02_PITCH_VOICING': cache_af02_diagnostic,
        'AF04_CPP': cache_af04_diagnostic,
        AF09_DIAGNOSTIC_BLOCK_ID: cache_af09_diagnostic,
    }
    for block_id in (
        'AF02_PITCH_VOICING',
        'AF04_CPP',
        AF09_DIAGNOSTIC_BLOCK_ID,
    ):
        block = BLOCK_BY_ID[block_id]
        legacy_e05_plus_block, legacy_names = (
            assemble_single_block_matrix(
                X_e05=e05_cache.validated.X,
                y=e05_cache.validated.y,
                record_keys=e05_cache.validated.record_keys,
                call_ids=canonical_call_ids,
                candidate_frame=legacy_candidate_frame,
                block_id=block_id,
            )
        )
        if legacy_names[-block.output_dimension:] != block.feature_names:
            raise AssertionError(
                f'{block_id} legacy raw feature order changed'
            )
        legacy_block = legacy_e05_plus_block[
            :, -block.output_dimension:
        ]
        current_block = current_block_sources[block_id].loc[
            :, block.feature_names
        ].to_numpy(dtype=np.float32)
        raw_matrix_rows.append(_numeric_detail(
            f'raw_{block_id}',
            legacy_block,
            current_block,
        ))
        del legacy_e05_plus_block, legacy_block, current_block

    raw_matrix_details = pd.DataFrame(raw_matrix_rows)
    if not bool(raw_matrix_details['array_equal'].all()):
        return _emit_first_difference(
            'raw_feature_matrix',
            raw_matrix_details.loc[
                ~raw_matrix_details['array_equal']
            ].reset_index(drop=True),
            fit_executed=False,
        )

    scaler_details = pd.DataFrame([
        _numeric_detail(
            'scaler.mean_',
            legacy_prepared_af09.scaler.mean_,
            prepared_52d.scaler.mean_,
        ),
        _numeric_detail(
            'scaler.scale_',
            legacy_prepared_af09.scaler.scale_,
            prepared_52d.scaler.scale_,
        ),
    ])
    if not bool(scaler_details['array_equal'].all()):
        return _emit_first_difference(
            'scaler_statistics',
            scaler_details.loc[
                ~scaler_details['array_equal']
            ].reset_index(drop=True),
            fit_executed=False,
        )

    scaled_matrix_details = pd.DataFrame([
        _numeric_detail(
            'X_train_scaled',
            legacy_prepared_af09.X_train_scaled,
            prepared_52d.X_train_scaled,
        ),
        _numeric_detail(
            'X_evaluation_scaled',
            legacy_prepared_af09.X_evaluation_scaled,
            prepared_52d.X_evaluation_scaled,
        ),
    ])
    if not bool(scaled_matrix_details['array_equal'].all()):
        return _emit_first_difference(
            'scaled_feature_matrix',
            scaled_matrix_details.loc[
                ~scaled_matrix_details['array_equal']
            ].reset_index(drop=True),
            fit_executed=False,
        )

    af09_runner_rerun = run_frozen_49d_screening_candidate(
        legacy_prepared_af09,
        block_id=AF09_DIAGNOSTIC_BLOCK_ID,
        allow_fit=True,
    )

    legacy_config = dict(af09_runner_rerun.model_config)
    current_config = dict(result_52d.model_config)
    config_keys = sorted(set(legacy_config) | set(current_config))
    config_difference_rows = [
        {
            'item': f'model_config.{config_key}',
            'legacy_AF09_runner': legacy_config.get(config_key),
            'current_52D': current_config.get(config_key),
        }
        for config_key in config_keys
        if (
            config_key not in legacy_config
            or config_key not in current_config
            or legacy_config[config_key] != current_config[config_key]
        )
    ]
    legacy_seed = af09_runner_rerun.fit_audit['seed']
    current_seed = result_52d.fit_audit['seed']
    if legacy_seed != current_seed:
        config_difference_rows.append({
            'item': 'seed',
            'legacy_AF09_runner': legacy_seed,
            'current_52D': current_seed,
        })
    if config_difference_rows:
        return _emit_first_difference(
            'model_config_or_seed',
            pd.DataFrame(config_difference_rows),
            fit_executed=True,
            rerun_result=af09_runner_rerun,
        )

    truth_detail = pd.DataFrame([
        _numeric_detail(
            'y_true',
            af09_runner_rerun.y_true,
            result_52d.y_true,
        )
    ])
    if not bool(truth_detail['array_equal'].all()):
        return _emit_first_difference(
            'evaluation_labels',
            truth_detail,
            fit_executed=True,
            rerun_result=af09_runner_rerun,
        )

    legacy_prediction = np.asarray(
        af09_runner_rerun.y_pred, dtype=np.int64
    )
    current_prediction = np.asarray(result_52d.y_pred, dtype=np.int64)
    prediction_detail = _numeric_detail(
        'predictions',
        legacy_prediction,
        current_prediction,
    )
    legacy_accuracy = float(
        af09_runner_rerun.metrics['accuracy']
    )
    current_accuracy = float(result_52d.metrics['accuracy'])
    legacy_confusion = af09_runner_rerun.metrics[
        'confusion_matrix'
    ]
    current_confusion = result_52d.metrics['confusion_matrix']
    model_outputs_equal = (
        prediction_detail['array_equal']
        and legacy_accuracy == current_accuracy
        and legacy_confusion == current_confusion
    )
    if not model_outputs_equal:
        disagreement_positions = np.flatnonzero(
            legacy_prediction != current_prediction
        )
        disagreement_keys = tuple(
            prepared_52d.evaluation_record_keys[
                disagreement_positions[:20]
            ].astype(str)
        )
        result_difference = pd.DataFrame([
            {
                'result': 'AF09_runner_rerun',
                'accuracy': legacy_accuracy,
                'confusion_matrix': legacy_confusion,
                'prediction_disagreement_count': (
                    prediction_detail['difference_count']
                ),
                'first_20_disagreement_record_keys': disagreement_keys,
            },
            {
                'result': 'current_52D',
                'accuracy': current_accuracy,
                'confusion_matrix': current_confusion,
                'prediction_disagreement_count': (
                    prediction_detail['difference_count']
                ),
                'first_20_disagreement_record_keys': disagreement_keys,
            },
        ])
        return _emit_first_difference(
            'model_fit_output',
            result_difference,
            fit_executed=True,
            rerun_result=af09_runner_rerun,
        )

    historical_reproduced = bool(np.isclose(
        legacy_accuracy,
        HISTORICAL_AF09_ACCURACY,
        rtol=0.0,
        atol=ACCURACY_ATOL,
    ))
    if not historical_reproduced:
        historical_difference = pd.DataFrame([
            {
                'result': 'historical_saved_AF09',
                'accuracy': HISTORICAL_AF09_ACCURACY,
                'accuracy_delta_vs_historical': 0.0,
                'confusion_matrix': 'not available in current kernel',
                'prediction_disagreement_vs_current_52D': (
                    'not available in current kernel'
                ),
            },
            {
                'result': 'AF09_runner_rerun',
                'accuracy': legacy_accuracy,
                'accuracy_delta_vs_historical': (
                    legacy_accuracy - HISTORICAL_AF09_ACCURACY
                ),
                'confusion_matrix': legacy_confusion,
                'prediction_disagreement_vs_current_52D': 0,
            },
            {
                'result': 'current_52D',
                'accuracy': current_accuracy,
                'accuracy_delta_vs_historical': (
                    current_accuracy - HISTORICAL_AF09_ACCURACY
                ),
                'confusion_matrix': current_confusion,
                'prediction_disagreement_vs_current_52D': 0,
            },
        ])
        return _emit_first_difference(
            'historical_saved_AF09_result',
            historical_difference,
            fit_executed=True,
            rerun_result=af09_runner_rerun,
        )

    no_difference = pd.DataFrame([{
        'legacy_AF09_runner_accuracy': legacy_accuracy,
        'current_52D_accuracy': current_accuracy,
        'historical_AF09_accuracy': HISTORICAL_AF09_ACCURACY,
        'prediction_disagreement_count': 0,
        'confusion_matrix': legacy_confusion,
    }])
    return _emit_first_difference(
        'no_mismatch_detected',
        no_difference,
        fit_executed=True,
        rerun_result=af09_runner_rerun,
    )


af09_52d_mismatch_diagnostic = (
    _run_self_contained_af09_52d_diagnostic()
)

,first_mismatch_stage,historical_AF09_accuracy,current_52D_accuracy,AF09_runner_rerun_accuracy,AF09_runner_fit_executed
0,historical_saved_AF09_result,0.86897,0.869532,0.869532,True


,result,accuracy,accuracy_delta_vs_historical,confusion_matrix,prediction_disagreement_vs_current_52D
0,historical_saved_AF09,0.868970,0.000000,not available in current kernel,not available in current kernel
1,AF09_runner_rerun,0.869532,0.000562,"[[71066, 14285], [8924, 83616]]",0
2,current_52D,0.869532,0.000562,"[[71066, 14285], [8924, 83616]]",0


AF07/AF08/49D/54D were not fitted by this cell.


### 52D vs 54D 결과

AF07을 추가한 54D는 Accuracy가 +0.0107%p로 거의 차이가 없었다.

하지만 119대원 Recall은 -1.24%p 떨어지고,
신고자 Recall은 +1.16%p 올라 예측이 신고자 쪽으로 치우쳤다.

이 패턴은 duration과 overlap에서도 반복됐고,
high-confidence error도 2590 → 2647로 증가했다.

→ AF07은 추가하지 않고 DROP.
→ AF09까지 포함한 52D를 현재 reference로 유지한다.

| Feature                               |      차원 | 초보자식 의미                            | 쉽게 말하면                         |
| ------------------------------------- | ------: | ---------------------------------- | ------------------------------ |
| **MFCC mean**                         |     20D | 음성의 평균적인 음색·주파수 모양                 | “이 발화는 전체적으로 어떤 목소리 느낌인가?”     |
| **MFCC std**                          |     20D | 음색·주파수 구조가 시간에 따라 얼마나 변하는지         | “말하는 동안 목소리 특성이 얼마나 흔들리나?”     |
| **Spectral Rolloff**                  |      1D | 에너지가 높은 주파수까지 얼마나 퍼지는지             | “소리가 날카로운 편인가?”                |
| **ZCR**                               |      1D | 파형이 얼마나 자주 +/−를 오가는지               | “소리가 얼마나 거칠고 잡음성 있나?”          |
| **AF02 Pitch / Voicing**              |      5D | 목소리 높이, 높이 변화, 사람이 실제로 말하는 정도, 주기성 | “목소리 높이와 발성 패턴은 어떤가?”          |
| **AF04 CPP**                          |      2D | 음성이 얼마나 또렷하고 주기적인지                 | “사람 목소리가 얼마나 선명한가?”            |
| **AF09 Multiband Envelope Coherence** |      3D | 여러 주파수 대역이 시간에 따라 같이 움직이는 정도       | “소리의 여러 주파수 부분이 얼마나 같이 움직이나?”  |
| **총합**                                | **52D** | 한 발화를 52개의 숫자로 요약                  | 모델은 이 52개 숫자를 보고 119대원/신고자를 구분 |


## Current 52D reference — MFCC axis reduction

현재 재현 가능한 52D (Accuracy = 0.869532)를 비교 기준으로 유지한다. 과거 0.868970은 historical_non_reproduced_result이며 이 실험의 gate나 비교 기준으로 사용하지 않는다.

이 구간은 기존 52D matrix의 MFCC40 열을 subset으로 재조합한다. waveform 또는 acoustic feature를 다시 추출하지 않는다. mfcc_01–mfcc_13은 MFCC coefficient 0–12에 해당한다.

- **REF_MFCC20 (52D):** mean 20 + std 20 + 동일 non-MFCC 12D
- **MFCC13_BOTH (38D):** mean 13 + std 13 + 동일 non-MFCC 12D
- **MFCC20_MEAN_MFCC13_STD (45D):** mean 20 + std 13 + 동일 non-MFCC 12D
- **MFCC13_MEAN_MFCC20_STD (45D):** mean 13 + std 20 + 동일 non-MFCC 12D

모든 candidate는 현재 52D와 동일한 internal split, label/record order, M04 configuration, seed=42, max_iter=100을 사용한다. scaler는 representation별 internal-train rows에만 fit한다. 결과는 자동 승격하지 않는다.

In [18]:
# 1. Audit current 52D and prepare MFCC-column subsets without extraction
from dcc_data.mission2_acoustic_feature_expansion import (
    BLOCK_BY_ID,
    M04_ITER100_FEATURE_SCREENING_CONFIG,
    _prepare_representation,
    _run_prepared_representation,
)
from dcc_data.mission2_model_experiments import (
    E05_EXTRA_FEATURE_NAMES,
    MFCC40_FEATURE_NAMES,
    calculate_prediction_metrics,
)

required_mfcc_ablation_objects = (
    'result_52d',
    'prepared_52d',
    'X_52d',
    'feature_names_52d',
    'e05_cache',
)
missing_mfcc_ablation_objects = [
    name for name in required_mfcc_ablation_objects
    if name not in globals()
]
if missing_mfcc_ablation_objects:
    raise RuntimeError(
        'Run the existing 52D preparation and current-reference cells first. '
        f'Missing objects: {missing_mfcc_ablation_objects}'
    )

CURRENT_52D_REFERENCE_ACCURACY = 0.869532
if not np.isclose(
    float(result_52d.metrics['accuracy']),
    CURRENT_52D_REFERENCE_ACCURACY,
    rtol=0.0,
    atol=5e-7,
):
    raise RuntimeError(
        'STOP: current result_52d does not reproduce Accuracy=0.869532. '
        'Do not run the MFCC ablation fits.'
    )
if dict(result_52d.model_config) != dict(
    M04_ITER100_FEATURE_SCREENING_CONFIG
):
    raise RuntimeError('STOP: current 52D model configuration differs')
if result_52d.model_config['max_iter'] != 100:
    raise RuntimeError('STOP: current 52D max_iter is not 100')
if result_52d.fit_audit['seed'] != 42:
    raise RuntimeError('STOP: current 52D seed is not 42')
if result_52d.fit_audit['scaler_fit_scope'] != 'internal_train_only':
    raise RuntimeError('STOP: current 52D scaler policy differs')
if int(prepared_52d.scaler.n_samples_seen_) != len(
    prepared_52d.train_idx
):
    raise RuntimeError('STOP: 52D scaler was not fit on train rows only')
if tuple(result_52d.feature_names) != tuple(prepared_52d.feature_names):
    raise RuntimeError('STOP: 52D result/prepared feature order differs')
if not np.array_equal(result_52d.y_true, prepared_52d.y_evaluation):
    raise RuntimeError('STOP: 52D result/prepared labels differ')
for audit_name, prepared_values in (
    ('train_idx', prepared_52d.train_idx),
    ('validation_idx', prepared_52d.validation_idx),
    ('training_record_keys', prepared_52d.training_record_keys),
    ('validation_record_keys', prepared_52d.evaluation_record_keys),
):
    if not np.array_equal(
        result_52d.fit_audit[audit_name],
        prepared_values,
    ):
        raise RuntimeError(
            f'STOP: 52D result/prepared {audit_name} differs'
        )
for scaler_attribute in ('mean_', 'scale_'):
    if not np.array_equal(
        getattr(result_52d.scaler, scaler_attribute),
        getattr(prepared_52d.scaler, scaler_attribute),
    ):
        raise RuntimeError(
            f'STOP: 52D scaler {scaler_attribute} differs'
        )

mfcc20_mean_names = tuple(MFCC40_FEATURE_NAMES[:20])
mfcc20_std_names = tuple(MFCC40_FEATURE_NAMES[20:])
mfcc13_mean_names = mfcc20_mean_names[:13]
mfcc13_std_names = mfcc20_std_names[:13]
non_mfcc_52d_feature_names = (
    *E05_EXTRA_FEATURE_NAMES,
    *BLOCK_BY_ID['AF02_PITCH_VOICING'].feature_names,
    *BLOCK_BY_ID['AF04_CPP'].feature_names,
    *BLOCK_BY_ID['AF09_MULTIBAND_ENVELOPE_COHERENCE'].feature_names,
)
expected_feature_names_52d = (
    *mfcc20_mean_names,
    *mfcc20_std_names,
    *non_mfcc_52d_feature_names,
)
if tuple(feature_names_52d) != expected_feature_names_52d:
    raise RuntimeError(
        'STOP: current 52D order is not '
        'MFCC20 mean/std + rolloff + zcr + AF02 + AF04 + AF09'
    )

X_52d_mfcc_source = np.asarray(X_52d, dtype=np.float32)
if X_52d_mfcc_source.shape != (
    len(e05_cache.validated.y),
    len(expected_feature_names_52d),
):
    raise RuntimeError('STOP: current 52D matrix shape differs')
if not np.isfinite(X_52d_mfcc_source).all():
    raise RuntimeError('STOP: current 52D matrix contains NaN or inf')
if not np.array_equal(
    X_52d_mfcc_source[:, :40],
    np.asarray(e05_cache.validated.X, dtype=np.float32)[:, :40],
):
    raise RuntimeError('STOP: current 52D MFCC40 prefix differs from cache')

MFCC_ABLATION_IDS = (
    'REF_MFCC20',
    'MFCC13_BOTH',
    'MFCC20_MEAN_MFCC13_STD',
    'MFCC13_MEAN_MFCC20_STD',
)
mfcc_ablation_feature_names = {
    'REF_MFCC20': expected_feature_names_52d,
    'MFCC13_BOTH': (
        *mfcc13_mean_names,
        *mfcc13_std_names,
        *non_mfcc_52d_feature_names,
    ),
    'MFCC20_MEAN_MFCC13_STD': (
        *mfcc20_mean_names,
        *mfcc13_std_names,
        *non_mfcc_52d_feature_names,
    ),
    'MFCC13_MEAN_MFCC20_STD': (
        *mfcc13_mean_names,
        *mfcc20_std_names,
        *non_mfcc_52d_feature_names,
    ),
}
expected_mfcc_ablation_dimensions = {
    'REF_MFCC20': 52,
    'MFCC13_BOTH': 38,
    'MFCC20_MEAN_MFCC13_STD': 45,
    'MFCC13_MEAN_MFCC20_STD': 45,
}
source_column_by_name = {
    name: index for index, name in enumerate(expected_feature_names_52d)
}
mfcc_ablation_matrices = {}
for representation_id in MFCC_ABLATION_IDS:
    names = mfcc_ablation_feature_names[representation_id]
    if len(names) != expected_mfcc_ablation_dimensions[representation_id]:
        raise AssertionError(
            f'{representation_id} dimension contract failed'
        )
    if tuple(names[-12:]) != non_mfcc_52d_feature_names:
        raise AssertionError(
            f'{representation_id} non-MFCC order changed'
        )
    column_indices = [
        source_column_by_name[name] for name in names
    ]
    matrix = X_52d_mfcc_source[:, column_indices].astype(
        np.float32,
        copy=False,
    )
    if matrix.shape[1] != len(names) or not np.isfinite(matrix).all():
        raise AssertionError(
            f'{representation_id} matrix contract failed'
        )
    mfcc_ablation_matrices[representation_id] = matrix

mfcc_ablation_prepared = {'REF_MFCC20': prepared_52d}
for representation_id in MFCC_ABLATION_IDS[1:]:
    mfcc_ablation_prepared[representation_id] = _prepare_representation(
        representation_id=representation_id,
        X=mfcc_ablation_matrices[representation_id],
        y=np.asarray(e05_cache.validated.y, dtype=np.int64),
        record_keys=np.asarray(
            e05_cache.validated.record_keys
        ).astype(str),
        feature_names=mfcc_ablation_feature_names[representation_id],
        train_idx=prepared_52d.train_idx,
        validation_idx=prepared_52d.validation_idx,
        split_audit=prepared_52d.split_audit,
    )

for representation_id, prepared in mfcc_ablation_prepared.items():
    if len(prepared.feature_names) != (
        expected_mfcc_ablation_dimensions[representation_id]
    ):
        raise AssertionError(
            f'{representation_id} prepared dimension differs'
        )
    for attribute in (
        'train_idx',
        'validation_idx',
        'training_record_keys',
        'evaluation_record_keys',
        'y_train',
        'y_evaluation',
    ):
        if not np.array_equal(
            getattr(prepared, attribute),
            getattr(prepared_52d, attribute),
        ):
            raise AssertionError(
                f'{representation_id} {attribute} alignment differs'
            )
    if int(prepared.scaler.n_samples_seen_) != len(
        prepared.train_idx
    ):
        raise AssertionError(
            f'{representation_id} scaler fit scope differs'
        )

mfcc_ablation_preparation_audit = pd.DataFrame([
    {
        'representation_id': representation_id,
        'dimension': len(prepared.feature_names),
        'train_rows': len(prepared.train_idx),
        'validation_rows': len(prepared.validation_idx),
        'seed': 42,
        'max_iter': 100,
        'scaler_fit_rows': int(prepared.scaler.n_samples_seen_),
        'scaler_policy': 'internal_train_only',
        'same_rows_keys_labels_as_52D': True,
        'non_MFCC_12D_preserved': (
            tuple(prepared.feature_names[-12:])
            == non_mfcc_52d_feature_names
        ),
        'source': (
            'existing_current_52D'
            if representation_id == 'REF_MFCC20'
            else 'existing_MFCC40_column_subset'
        ),
    }
    for representation_id, prepared
    in mfcc_ablation_prepared.items()
])
display(mfcc_ablation_preparation_audit)
print('No waveform or feature extraction was performed by this cell.')

,representation_id,dimension,train_rows,validation_rows,seed,max_iter,scaler_fit_rows,scaler_policy,same_rows_keys_labels_as_52D,non_MFCC_12D_preserved,source
0,REF_MFCC20,52,711467,177891,42,100,711467,internal_train_only,True,True,existing_current_52D
1,MFCC13_BOTH,38,711467,177891,42,100,711467,internal_train_only,True,True,existing_MFCC40_column_subset
2,MFCC20_MEAN_MFCC13_STD,45,711467,177891,42,100,711467,internal_train_only,True,True,existing_MFCC40_column_subset
3,MFCC13_MEAN_MFCC20_STD,45,711467,177891,42,100,711467,internal_train_only,True,True,existing_MFCC40_column_subset


No waveform or feature extraction was performed by this cell.


In [19]:
# 2. Fit only the three reduced-MFCC candidates; reuse current 52D result
if 'mfcc_ablation_prepared' not in globals():
    raise RuntimeError('Run the MFCC ablation preparation cell first.')

mfcc_ablation_results = {'REF_MFCC20': result_52d}
for representation_id in MFCC_ABLATION_IDS[1:]:
    mfcc_ablation_results[representation_id] = (
        _run_prepared_representation(
            mfcc_ablation_prepared[representation_id],
            experiment_id=(
                f'{representation_id}_M04_ITER100_CANDIDATE'
            ),
            allow_fit=True,
        )
    )

for representation_id, result in mfcc_ablation_results.items():
    prepared = mfcc_ablation_prepared[representation_id]
    if dict(result.model_config) != dict(result_52d.model_config):
        raise AssertionError(
            f'{representation_id} model configuration differs'
        )
    if result.fit_audit['seed'] != result_52d.fit_audit['seed']:
        raise AssertionError(f'{representation_id} seed differs')
    if tuple(result.feature_names) != tuple(prepared.feature_names):
        raise AssertionError(
            f'{representation_id} feature order differs'
        )
    if not np.array_equal(result.y_true, result_52d.y_true):
        raise AssertionError(
            f'{representation_id} validation labels differ'
        )
    for audit_name in (
        'train_idx',
        'validation_idx',
        'training_record_keys',
        'validation_record_keys',
    ):
        if not np.array_equal(
            result.fit_audit[audit_name],
            result_52d.fit_audit[audit_name],
        ):
            raise AssertionError(
                f'{representation_id} {audit_name} differs'
            )

print(
    'Completed MFCC reduction candidates:',
    ', '.join(MFCC_ABLATION_IDS[1:]),
)
print('REF_MFCC20 reused result_52d; it was not refit.')
print('Automatic reference promotion: disabled; user decision required.')

Completed MFCC reduction candidates: MFCC13_BOTH, MFCC20_MEAN_MFCC13_STD, MFCC13_MEAN_MFCC20_STD
REF_MFCC20 reused result_52d; it was not refit.
Automatic reference promotion: disabled; user decision required.


In [20]:
# 3. Overall metrics and paired transitions against current 52D
if 'mfcc_ablation_results' not in globals():
    raise RuntimeError('Run the MFCC ablation fit cell first.')

mfcc_ablation_truth = np.asarray(
    result_52d.y_true,
    dtype=np.int64,
)
mfcc_ablation_predictions = {
    representation_id: np.asarray(result.y_pred, dtype=np.int64)
    for representation_id, result in mfcc_ablation_results.items()
}
mfcc_ablation_correct = {
    representation_id: (
        prediction == mfcc_ablation_truth
    )
    for representation_id, prediction
    in mfcc_ablation_predictions.items()
}
reference_mfcc_ablation_metrics = result_52d.metrics
reference_mfcc_ablation_correct = (
    mfcc_ablation_correct['REF_MFCC20']
)

mfcc_ablation_comparison = pd.DataFrame([
    {
        'representation_id': representation_id,
        'dimension': len(result.feature_names),
        'accuracy': float(result.metrics['accuracy']),
        'accuracy_delta_vs_52D': float(
            result.metrics['accuracy']
            - reference_mfcc_ablation_metrics['accuracy']
        ),
        'confusion_matrix': result.metrics['confusion_matrix'],
        'recall_speaker_0': float(
            result.metrics['recall_speaker_0']
        ),
        'recall_0_delta_vs_52D': float(
            result.metrics['recall_speaker_0']
            - reference_mfcc_ablation_metrics['recall_speaker_0']
        ),
        'recall_speaker_1': float(
            result.metrics['recall_speaker_1']
        ),
        'recall_1_delta_vs_52D': float(
            result.metrics['recall_speaker_1']
            - reference_mfcc_ablation_metrics['recall_speaker_1']
        ),
        'correct_count': int(result.metrics['correct_count']),
        'correct_count_delta_vs_52D': int(
            result.metrics['correct_count']
            - reference_mfcc_ablation_metrics['correct_count']
        ),
        '52D_wrong_to_candidate_correct': (
            np.nan
            if representation_id == 'REF_MFCC20'
            else int(np.sum(
                (~reference_mfcc_ablation_correct)
                & mfcc_ablation_correct[representation_id]
            ))
        ),
        '52D_correct_to_candidate_wrong': (
            np.nan
            if representation_id == 'REF_MFCC20'
            else int(np.sum(
                reference_mfcc_ablation_correct
                & (~mfcc_ablation_correct[representation_id])
            ))
        ),
        'automatic_promotion': False,
    }
    for representation_id, result
    in mfcc_ablation_results.items()
])
display(mfcc_ablation_comparison)

,representation_id,dimension,accuracy,accuracy_delta_vs_52D,confusion_matrix,recall_speaker_0,recall_0_delta_vs_52D,recall_speaker_1,recall_1_delta_vs_52D,correct_count,correct_count_delta_vs_52D,52D_wrong_to_candidate_correct,52D_correct_to_candidate_wrong,automatic_promotion
0,REF_MFCC20,52,0.869532,0.000000,"[[71066, 14285], [8924, 83616]]",0.832632,0.000000,0.903566,0.000000,154682,0,NaN,NaN,False
1,MFCC13_BOTH,38,0.863793,-0.005739,"[[72173, 13178], [11052, 81488]]",0.845602,0.012970,0.880571,-0.022995,153661,-1021,5501.0,6522.0,False
2,MFCC20_MEAN_MFCC13_STD,45,0.868717,-0.000815,"[[70749, 14602], [8752, 83788]]",0.828918,-0.003714,0.905425,0.001859,154537,-145,3296.0,3441.0,False
3,MFCC13_MEAN_MFCC20_STD,45,0.863175,-0.006358,"[[73335, 12016], [12324, 80216]]",0.859217,0.026584,0.866825,-0.036741,153551,-1131,6922.0,8053.0,False


In [21]:
# 4. Fixed-duration performance
if 'mfcc_ablation_predictions' not in globals():
    raise RuntimeError('Run the MFCC overall comparison cell first.')

mfcc_ablation_tracking = e05_cache.tracking.iloc[
    result_52d.fit_audit['validation_idx']
].reset_index(drop=True)
mfcc_ablation_tracking_keys = mfcc_ablation_tracking[
    'record_key'
].astype(str).to_numpy()
if not np.array_equal(
    mfcc_ablation_tracking_keys,
    np.asarray(
        result_52d.fit_audit['validation_record_keys']
    ).astype(str),
):
    raise RuntimeError(
        'STOP: diagnostic tracking and 52D record_key order differ'
    )

mfcc_ablation_duration_sec = (
    pd.to_numeric(
        mfcc_ablation_tracking['endAt_ms'],
        errors='raise',
    ).to_numpy(dtype=np.float64)
    - pd.to_numeric(
        mfcc_ablation_tracking['startAt_ms'],
        errors='raise',
    ).to_numpy(dtype=np.float64)
) / 1_000.0
MFCC_ABLATION_DURATION_ORDER = (
    '<0.5s',
    '0.5-1.0s',
    '1-2s',
    '2-5s',
    '>=5s',
)
mfcc_ablation_duration_slice = pd.cut(
    mfcc_ablation_duration_sec,
    bins=[-np.inf, 0.5, 1.0, 2.0, 5.0, np.inf],
    labels=MFCC_ABLATION_DURATION_ORDER,
    right=False,
).astype(str)

def _mfcc_ablation_slice_metrics(selected, prediction):
    if not np.any(selected):
        return {
            'sample_count': 0,
            'accuracy': np.nan,
            'confusion_matrix': [[0, 0], [0, 0]],
            'recall_speaker_0': np.nan,
            'recall_speaker_1': np.nan,
        }
    return calculate_prediction_metrics(
        mfcc_ablation_truth[selected],
        prediction[selected],
    )

mfcc_ablation_duration_metrics = {}
for representation_id in MFCC_ABLATION_IDS:
    prediction = mfcc_ablation_predictions[representation_id]
    mfcc_ablation_duration_metrics[representation_id] = {
        duration_name: _mfcc_ablation_slice_metrics(
            mfcc_ablation_duration_slice == duration_name,
            prediction,
        )
        for duration_name in MFCC_ABLATION_DURATION_ORDER
    }

mfcc_ablation_duration_rows = []
for duration_name in MFCC_ABLATION_DURATION_ORDER:
    reference_metrics = mfcc_ablation_duration_metrics[
        'REF_MFCC20'
    ][duration_name]
    for representation_id in MFCC_ABLATION_IDS:
        metrics = mfcc_ablation_duration_metrics[
            representation_id
        ][duration_name]
        mfcc_ablation_duration_rows.append({
            'duration_slice': duration_name,
            'representation_id': representation_id,
            'count': int(metrics['sample_count']),
            'accuracy': metrics['accuracy'],
            'accuracy_delta_vs_52D': (
                metrics['accuracy']
                - reference_metrics['accuracy']
            ),
            'confusion_matrix': metrics['confusion_matrix'],
            'recall_speaker_0': metrics['recall_speaker_0'],
            'recall_0_delta_vs_52D': (
                metrics['recall_speaker_0']
                - reference_metrics['recall_speaker_0']
            ),
            'recall_speaker_1': metrics['recall_speaker_1'],
            'recall_1_delta_vs_52D': (
                metrics['recall_speaker_1']
                - reference_metrics['recall_speaker_1']
            ),
        })

mfcc_ablation_fixed_duration = pd.DataFrame(
    mfcc_ablation_duration_rows
)
display(mfcc_ablation_fixed_duration)

,duration_slice,representation_id,count,accuracy,accuracy_delta_vs_52D,confusion_matrix,recall_speaker_0,recall_0_delta_vs_52D,recall_speaker_1,recall_1_delta_vs_52D
0,<0.5s,REF_MFCC20,14076,0.863527,0.000000,"[[5963, 765], [1156, 6192]]",0.886296,0.000000,0.842678,0.000000
1,<0.5s,MFCC13_BOTH,14076,0.858127,-0.005399,"[[5842, 886], [1111, 6237]]",0.868312,-0.017985,0.848802,0.006124
2,<0.5s,MFCC20_MEAN_MFCC13_STD,14076,0.867931,0.004405,"[[5955, 773], [1086, 6262]]",0.885107,-0.001189,0.852205,0.009526
3,<0.5s,MFCC13_MEAN_MFCC20_STD,14076,0.863029,-0.000497,"[[5901, 827], [1101, 6247]]",0.877081,-0.009215,0.850163,0.007485
4,0.5-1.0s,REF_MFCC20,43199,0.825598,0.000000,"[[14958, 4635], [2899, 20707]]",0.763436,0.000000,0.877192,0.000000
5,0.5-1.0s,MFCC13_BOTH,43199,0.815274,-0.010324,"[[16528, 3065], [4915, 18691]]",0.843567,0.080131,0.791790,-0.085402
6,0.5-1.0s,MFCC20_MEAN_MFCC13_STD,43199,0.824001,-0.001597,"[[14965, 4628], [2975, 20631]]",0.763793,0.000357,0.873973,-0.003220
7,0.5-1.0s,MFCC13_MEAN_MFCC20_STD,43199,0.814139,-0.011459,"[[16918, 2675], [5354, 18252]]",0.863472,0.100036,0.773193,-0.103999
8,1-2s,REF_MFCC20,58320,0.847154,0.000000,"[[25813, 6219], [2695, 23593]]",0.805850,0.000000,0.897482,0.000000
9,1-2s,MFCC13_BOTH,58320,0.841975,-0.005178,"[[25665, 6367], [2849, 23439]]",0.801230,-0.004620,0.891624,-0.005858


In [22]:
# 5. Overlap/no-overlap and high-confidence error diagnostics
if 'mfcc_ablation_tracking' not in globals():
    raise RuntimeError('Run the MFCC fixed-duration cell first.')

mfcc_ablation_overlap_work = mfcc_ablation_tracking.loc[
    :, ['call_id', 'startAt_ms', 'endAt_ms']
].copy()
mfcc_ablation_overlap_work['_row'] = np.arange(
    len(mfcc_ablation_overlap_work),
    dtype=np.int64,
)
mfcc_ablation_overlap_work['startAt_ms'] = pd.to_numeric(
    mfcc_ablation_overlap_work['startAt_ms'],
    errors='raise',
)
mfcc_ablation_overlap_work['endAt_ms'] = pd.to_numeric(
    mfcc_ablation_overlap_work['endAt_ms'],
    errors='raise',
)
mfcc_ablation_overlap_ordered = mfcc_ablation_overlap_work.sort_values(
    ['call_id', 'startAt_ms', 'endAt_ms', '_row'],
    kind='stable',
)
mfcc_ablation_previous_max_end = (
    mfcc_ablation_overlap_ordered.groupby(
        'call_id',
        sort=False,
    )['endAt_ms'].transform(
        lambda values: values.cummax().shift(1)
    )
)
mfcc_ablation_next_start = (
    mfcc_ablation_overlap_ordered.groupby(
        'call_id',
        sort=False,
    )['startAt_ms'].shift(-1)
)
mfcc_ablation_overlap_ordered['_overlap'] = (
    (
        mfcc_ablation_overlap_ordered['startAt_ms']
        < mfcc_ablation_previous_max_end
    )
    | (
        mfcc_ablation_next_start
        < mfcc_ablation_overlap_ordered['endAt_ms']
    )
).fillna(False)
mfcc_ablation_overlap_mask = (
    mfcc_ablation_overlap_ordered.sort_values(
        '_row',
        kind='stable',
    )['_overlap'].to_numpy(dtype=bool)
)

mfcc_ablation_overlap_rows = []
for slice_name, selected in (
    ('overlap', mfcc_ablation_overlap_mask),
    ('no_overlap', ~mfcc_ablation_overlap_mask),
):
    reference_metrics = _mfcc_ablation_slice_metrics(
        selected,
        mfcc_ablation_predictions['REF_MFCC20'],
    )
    for representation_id in MFCC_ABLATION_IDS:
        metrics = _mfcc_ablation_slice_metrics(
            selected,
            mfcc_ablation_predictions[representation_id],
        )
        mfcc_ablation_overlap_rows.append({
            'slice': slice_name,
            'representation_id': representation_id,
            'count': int(metrics['sample_count']),
            'accuracy': metrics['accuracy'],
            'accuracy_delta_vs_52D': (
                metrics['accuracy']
                - reference_metrics['accuracy']
            ),
            'confusion_matrix': metrics['confusion_matrix'],
            'recall_speaker_0': metrics['recall_speaker_0'],
            'recall_0_delta_vs_52D': (
                metrics['recall_speaker_0']
                - reference_metrics['recall_speaker_0']
            ),
            'recall_speaker_1': metrics['recall_speaker_1'],
            'recall_1_delta_vs_52D': (
                metrics['recall_speaker_1']
                - reference_metrics['recall_speaker_1']
            ),
        })
mfcc_ablation_overlap = pd.DataFrame(
    mfcc_ablation_overlap_rows
)

MFCC_ABLATION_HIGH_CONFIDENCE_THRESHOLD = 0.90
mfcc_ablation_reference_high_confidence_error = None
mfcc_ablation_high_confidence_rows = []
for representation_id in MFCC_ABLATION_IDS:
    result = mfcc_ablation_results[representation_id]
    prediction = mfcc_ablation_predictions[representation_id]
    correct = mfcc_ablation_correct[representation_id]
    confidence = np.asarray(
        result.probabilities,
        dtype=np.float64,
    ).max(axis=1)
    high_confidence_error = (
        (~correct)
        & (
            confidence
            >= MFCC_ABLATION_HIGH_CONFIDENCE_THRESHOLD
        )
    )
    if representation_id == 'REF_MFCC20':
        mfcc_ablation_reference_high_confidence_error = (
            high_confidence_error
        )
    mfcc_ablation_high_confidence_rows.append({
        'representation_id': representation_id,
        'threshold': MFCC_ABLATION_HIGH_CONFIDENCE_THRESHOLD,
        'high_confidence_error_count': int(np.sum(
            high_confidence_error
        )),
        'high_confidence_error_delta_vs_52D': (
            int(np.sum(high_confidence_error))
            - int(np.sum(
                mfcc_ablation_reference_high_confidence_error
            ))
        ),
        'high_confidence_FP_count': int(np.sum(
            high_confidence_error
            & (mfcc_ablation_truth == 0)
            & (prediction == 1)
        )),
        'high_confidence_FN_count': int(np.sum(
            high_confidence_error
            & (mfcc_ablation_truth == 1)
            & (prediction == 0)
        )),
        '52D_high_confidence_error_fixed': (
            np.nan
            if representation_id == 'REF_MFCC20'
            else int(np.sum(
                mfcc_ablation_reference_high_confidence_error
                & correct
            ))
        ),
        'candidate_high_confidence_regressed_from_52D': (
            np.nan
            if representation_id == 'REF_MFCC20'
            else int(np.sum(
                reference_mfcc_ablation_correct
                & high_confidence_error
            ))
        ),
    })

mfcc_ablation_high_confidence = pd.DataFrame(
    mfcc_ablation_high_confidence_rows
)
display(mfcc_ablation_overlap)
display(mfcc_ablation_high_confidence)
print('Automatic reference promotion: disabled; user decision required.')

,slice,representation_id,count,accuracy,accuracy_delta_vs_52D,confusion_matrix,recall_speaker_0,recall_0_delta_vs_52D,recall_speaker_1,recall_1_delta_vs_52D
0,overlap,REF_MFCC20,78581,0.827846,0.000000,"[[30704, 7721], [5807, 34349]]",0.799063,0.000000,0.855389,0.000000
1,overlap,MFCC13_BOTH,78581,0.824665,-0.003181,"[[31197, 7228], [6550, 33606]]",0.811893,0.012830,0.836886,-0.018503
2,overlap,MFCC20_MEAN_MFCC13_STD,78581,0.826688,-0.001158,"[[30559, 7866], [5753, 34403]]",0.795290,-0.003774,0.856734,0.001345
3,overlap,MFCC13_MEAN_MFCC20_STD,78581,0.824461,-0.003385,"[[31583, 6842], [6952, 33204]]",0.821939,0.022876,0.826875,-0.028514
4,no_overlap,REF_MFCC20,99310,0.902517,0.000000,"[[40362, 6564], [3117, 49267]]",0.860120,0.000000,0.940497,0.000000
5,no_overlap,MFCC13_BOTH,99310,0.894754,-0.007764,"[[40976, 5950], [4502, 47882]]",0.873205,0.013084,0.914058,-0.026439
6,no_overlap,MFCC20_MEAN_MFCC13_STD,99310,0.901974,-0.000544,"[[40190, 6736], [2999, 49385]]",0.856455,-0.003665,0.942750,0.002253
7,no_overlap,MFCC13_MEAN_MFCC20_STD,99310,0.893807,-0.008710,"[[41752, 5174], [5372, 47012]]",0.889741,0.029621,0.897450,-0.043047


,representation_id,threshold,high_confidence_error_count,high_confidence_error_delta_vs_52D,high_confidence_FP_count,high_confidence_FN_count,52D_high_confidence_error_fixed,candidate_high_confidence_regressed_from_52D
0,REF_MFCC20,0.9,2590,0,1223,1367,NaN,NaN
1,MFCC13_BOTH,0.9,2655,65,1327,1328,33.0,23.0
2,MFCC20_MEAN_MFCC13_STD,0.9,2707,117,1362,1345,3.0,4.0
3,MFCC13_MEAN_MFCC20_STD,0.9,2621,31,1419,1202,21.0,19.0


Automatic reference promotion: disabled; user decision required.


### MFCC 축소 실험

현재 MFCC20 mean/std를 13개로 줄여봤지만 모든 경우 전체 Accuracy가 감소했다.

특히 MFCC mean을 20→13으로 줄였을 때 성능이 크게 떨어져,
뒤쪽 MFCC mean 정보도 모델에 중요한 것으로 보인다.

std만 20→13으로 줄인 경우 감소폭은 작았지만,
Accuracy와 high-confidence error 모두 현재 52D보다 좋지 않았다.

또한 MFCC를 줄여도 overlap 성능은 개선되지 않았다.

→ MFCC 축소 가설은 지지되지 않음.
→ 현재 MFCC20 mean/std를 그대로 유지하고 52D reference를 유지.

## 52D 기준 모델 잔여 오류 분석

현재 가장 성능이 좋은 52D 기준 모델이 어떤 상황에서 아직 틀리는지 분석한다.

발화 길이, overlap, speaker, confidence, call별 오류를 확인해서 반복되는 약점과 남은 오류 패턴을 찾고, 다음 feature 실험 방향을 정하는 것이 목적이다.

In [13]:
# 1. Resolve the reproducible 52D reference and fail fast on any contract mismatch
import numpy as np
import pandas as pd
from IPython.display import display

from dcc_data.mission2_acoustic_feature_expansion import (
    BLOCK_BY_ID,
    CANONICAL_52D_ACCURACY,
    CANONICAL_52D_CONFUSION_MATRIX,
    FROZEN_49D_FEATURE_NAMES,
    M04_ITER100_FEATURE_SCREENING_CONFIG,
    NEXT_CANDIDATE_BLOCK_IDS,
    ROUND1_BLOCK_IDS,
    ROUND2_BLOCK_IDS,
    _prepare_representation,
    load_candidate_feature_cache,
    prepare_frozen_49d_single_block_screening,
    resolve_canonical_52d_reference,
)
from dcc_data.mission2_model_experiments import (
    EXPECTED_INTERNAL_SPLIT,
    calculate_prediction_metrics,
)

RESIDUAL_AF09_BLOCK_ID = 'AF09_MULTIBAND_ENVELOPE_COHERENCE'
RESIDUAL_EXPECTED_VALIDATION_ROWS = 177_891
RESIDUAL_EXPECTED_ACCURACY = CANONICAL_52D_ACCURACY
RESIDUAL_EXPECTED_CONFUSION = [
    list(row) for row in CANONICAL_52D_CONFUSION_MATRIX
]
RESIDUAL_ACCURACY_ATOL = 5e-7

if 'e05_cache' not in globals():
    raise RuntimeError(
        'Residual analysis requires the existing validated e05_cache. '
        'Run the earlier E05 cache-load cell; no extraction is needed.'
    )

if 'result_52d' in globals() and 'prepared_52d' in globals():
    required_canonical_sources = (
        'X_52d', 'labels_52d_54d', 'canonical_record_keys',
        'train_idx_52d_54d', 'validation_idx_52d_54d',
    )
    missing_canonical_sources = [
        name for name in required_canonical_sources if name not in globals()
    ]
    if missing_canonical_sources:
        raise RuntimeError(
            'STOP: existing 52D objects lack independent canonical sources: '
            f'{missing_canonical_sources}'
        )
    residual_prepared_52d = prepared_52d
    residual_52d_resolution = resolve_canonical_52d_reference(
        residual_prepared_52d, existing_result=result_52d,
        canonical_matrix=X_52d, canonical_labels=labels_52d_54d,
        canonical_record_keys=canonical_record_keys,
        canonical_train_idx=train_idx_52d_54d,
        canonical_validation_idx=validation_idx_52d_54d,
        expected_split_contract=EXPECTED_INTERNAL_SPLIT, allow_refit=True,
    )
    residual_result_52d = residual_52d_resolution.result
    residual_reference_source = residual_52d_resolution.action
else:
    required_cache_objects = (
        'e05_cache',
        'utterance_manifest',
        'CANDIDATE_CACHE_ROOT',
    )
    missing_cache_objects = [
        name for name in required_cache_objects
        if name not in globals()
    ]
    if missing_cache_objects:
        raise RuntimeError(
            '52D objects are absent, and cache-only reconstruction '
            f'is missing: {missing_cache_objects}'
        )

    residual_cache_af02 = load_candidate_feature_cache(
        manifest=utterance_manifest,
        cache_root=CANDIDATE_CACHE_ROOT,
        split='Training',
        block_ids=ROUND1_BLOCK_IDS,
    )
    residual_cache_af04 = load_candidate_feature_cache(
        manifest=utterance_manifest,
        cache_root=CANDIDATE_CACHE_ROOT,
        split='Training',
        block_ids=ROUND2_BLOCK_IDS,
    )
    residual_cache_af09 = load_candidate_feature_cache(
        manifest=utterance_manifest,
        cache_root=CANDIDATE_CACHE_ROOT,
        split='Training',
        block_ids=NEXT_CANDIDATE_BLOCK_IDS,
    )

    residual_canonical_keys = np.asarray(
        e05_cache.validated.record_keys
    ).astype(str)
    residual_canonical_calls = (
        e05_cache.tracking['call_id'].astype(str).to_numpy()
    )
    for cache_name, cache_frame in (
        ('AF02', residual_cache_af02),
        ('AF04', residual_cache_af04),
        ('AF09', residual_cache_af09),
    ):
        if not np.array_equal(
            cache_frame['record_key'].astype(str).to_numpy(),
            residual_canonical_keys,
        ):
            raise RuntimeError(
                f'STOP: {cache_name} cache record_key order differs'
            )
        if not np.array_equal(
            cache_frame['call_id'].astype(str).to_numpy(),
            residual_canonical_calls,
        ):
            raise RuntimeError(
                f'STOP: {cache_name} cache call_id order differs'
            )

    residual_af02_names = BLOCK_BY_ID[
        'AF02_PITCH_VOICING'
    ].feature_names
    residual_af04_names = BLOCK_BY_ID['AF04_CPP'].feature_names
    residual_af09_names = BLOCK_BY_ID[
        RESIDUAL_AF09_BLOCK_ID
    ].feature_names
    residual_candidate_frame = residual_cache_af02.loc[
        :, ['record_key', 'call_id', *residual_af02_names]
    ].copy()
    for source_frame, feature_names in (
        (residual_cache_af04, residual_af04_names),
        (residual_cache_af09, residual_af09_names),
    ):
        for feature_name in feature_names:
            residual_candidate_frame[feature_name] = source_frame[
                feature_name
            ].to_numpy(dtype=np.float32)

    residual_prepared_map = (
        prepare_frozen_49d_single_block_screening(
            X_e05=e05_cache.validated.X,
            y=e05_cache.validated.y,
            record_keys=e05_cache.validated.record_keys,
            call_ids=residual_canonical_calls,
            candidate_frame=residual_candidate_frame,
            expected_split_contract=EXPECTED_INTERNAL_SPLIT,
            block_ids=(RESIDUAL_AF09_BLOCK_ID,),
        )
    )
    residual_prepared_52d = residual_prepared_map[
        RESIDUAL_AF09_BLOCK_ID
    ]
    residual_X_52d = np.concatenate(
        [
            np.asarray(e05_cache.validated.X, dtype=np.float32),
            residual_candidate_frame.loc[:, [
                *residual_af02_names, *residual_af04_names,
                *residual_af09_names,
            ]].to_numpy(dtype=np.float32),
        ],
        axis=1,
    ).astype(np.float32, copy=False)
    residual_labels_52d = np.asarray(
        e05_cache.validated.y, dtype=np.int64
    )
    residual_prepared_52d = _prepare_representation(
        representation_id='AF09_52D_REFERENCE', X=residual_X_52d,
        y=residual_labels_52d, record_keys=residual_canonical_keys,
        feature_names=(
            *FROZEN_49D_FEATURE_NAMES, *residual_af09_names,
        ),
        train_idx=residual_prepared_52d.train_idx,
        validation_idx=residual_prepared_52d.validation_idx,
        split_audit=residual_prepared_52d.split_audit,
    )
    residual_52d_resolution = resolve_canonical_52d_reference(
        residual_prepared_52d, existing_result=globals().get('result_52d'),
        canonical_matrix=residual_X_52d,
        canonical_labels=residual_labels_52d,
        canonical_record_keys=residual_canonical_keys,
        canonical_train_idx=residual_prepared_52d.train_idx,
        canonical_validation_idx=residual_prepared_52d.validation_idx,
        expected_split_contract=EXPECTED_INTERNAL_SPLIT, allow_refit=True,
    )
    residual_result_52d = residual_52d_resolution.result
    residual_reference_source = residual_52d_resolution.action

residual_y_true = np.asarray(
    residual_result_52d.y_true,
    dtype=np.int64,
)
residual_prediction = np.asarray(
    residual_result_52d.y_pred,
    dtype=np.int64,
)
residual_probabilities = np.asarray(
    residual_result_52d.probabilities,
    dtype=np.float64,
)
residual_recomputed_metrics = calculate_prediction_metrics(
    residual_y_true,
    residual_prediction,
)
residual_expected_feature_names = (
    *FROZEN_49D_FEATURE_NAMES,
    *BLOCK_BY_ID[RESIDUAL_AF09_BLOCK_ID].feature_names,
)
residual_train_idx = np.asarray(
    residual_prepared_52d.train_idx,
    dtype=np.int64,
)
residual_validation_idx = np.asarray(
    residual_prepared_52d.validation_idx,
    dtype=np.int64,
)
residual_training_keys = np.asarray(
    residual_prepared_52d.training_record_keys
).astype(str)
residual_validation_keys = np.asarray(
    residual_prepared_52d.evaluation_record_keys
).astype(str)
residual_all_keys = np.asarray(
    e05_cache.validated.record_keys
).astype(str)
residual_all_labels = np.asarray(
    e05_cache.validated.y,
    dtype=np.int64,
)
residual_tracking_keys = (
    e05_cache.tracking.iloc[residual_validation_idx]['record_key']
    .astype(str)
    .to_numpy()
)

residual_split_alignment_ok = all((
    np.array_equal(
        residual_result_52d.fit_audit['train_idx'],
        residual_train_idx,
    ),
    np.array_equal(
        residual_result_52d.fit_audit['validation_idx'],
        residual_validation_idx,
    ),
    np.array_equal(
        np.asarray(
            residual_result_52d.fit_audit['training_record_keys']
        ).astype(str),
        residual_training_keys,
    ),
    np.array_equal(
        np.asarray(
            residual_result_52d.fit_audit['validation_record_keys']
        ).astype(str),
        residual_validation_keys,
    ),
    np.array_equal(
        residual_training_keys,
        residual_all_keys[residual_train_idx],
    ),
    np.array_equal(
        residual_validation_keys,
        residual_all_keys[residual_validation_idx],
    ),
    np.array_equal(
        residual_prepared_52d.y_train,
        residual_all_labels[residual_train_idx],
    ),
    np.array_equal(
        residual_prepared_52d.y_evaluation,
        residual_all_labels[residual_validation_idx],
    ),
    np.array_equal(
        residual_y_true,
        residual_prepared_52d.y_evaluation,
    ),
    np.array_equal(
        residual_validation_keys,
        residual_tracking_keys,
    ),
    len(np.intersect1d(
        residual_train_idx,
        residual_validation_idx,
    )) == 0,
))

residual_contract_rows = [
    {
        'check': 'reference_source',
        'expected': (
            'existing objects, otherwise cache-only 52D reconstruction'
        ),
        'observed': residual_reference_source,
        'pass': True,
    },
    {
        'check': 'validation_rows',
        'expected': RESIDUAL_EXPECTED_VALIDATION_ROWS,
        'observed': len(residual_y_true),
        'pass': (
            len(residual_y_true)
            == RESIDUAL_EXPECTED_VALIDATION_ROWS
        ),
    },
    {
        'check': 'accuracy',
        'expected': RESIDUAL_EXPECTED_ACCURACY,
        'observed': residual_recomputed_metrics['accuracy'],
        'pass': bool(np.isclose(
            residual_recomputed_metrics['accuracy'],
            RESIDUAL_EXPECTED_ACCURACY,
            rtol=0.0,
            atol=RESIDUAL_ACCURACY_ATOL,
        )),
    },
    {
        'check': 'confusion_matrix',
        'expected': RESIDUAL_EXPECTED_CONFUSION,
        'observed': residual_recomputed_metrics[
            'confusion_matrix'
        ],
        'pass': (
            residual_recomputed_metrics['confusion_matrix']
            == RESIDUAL_EXPECTED_CONFUSION
        ),
    },
    {
        'check': 'validation_record_key_unique',
        'expected': True,
        'observed': bool(pd.Index(
            residual_validation_keys
        ).is_unique),
        'pass': bool(pd.Index(residual_validation_keys).is_unique),
    },
    {
        'check': 'train_validation_row_key_label_alignment',
        'expected': True,
        'observed': residual_split_alignment_ok,
        'pass': residual_split_alignment_ok,
    },
    {
        'check': '52D_feature_order',
        'expected': residual_expected_feature_names,
        'observed': tuple(residual_result_52d.feature_names),
        'pass': (
            tuple(residual_result_52d.feature_names)
            == residual_expected_feature_names
            == tuple(residual_prepared_52d.feature_names)
        ),
    },
    {
        'check': 'M04_max_iter_100_seed_42',
        'expected': {
            **dict(M04_ITER100_FEATURE_SCREENING_CONFIG),
        },
        'observed': dict(residual_result_52d.model_config),
        'pass': (
            dict(residual_result_52d.model_config)
            == dict(M04_ITER100_FEATURE_SCREENING_CONFIG)
            and residual_result_52d.fit_audit['seed'] == 42
        ),
    },
    {
        'check': 'internal_train_only_scaler',
        'expected': len(residual_train_idx),
        'observed': int(
            residual_prepared_52d.scaler.n_samples_seen_
        ),
        'pass': (
            residual_result_52d.fit_audit[
                'scaler_fit_scope'
            ] == 'internal_train_only'
            and int(
                residual_prepared_52d.scaler.n_samples_seen_
            ) == len(residual_train_idx)
        ),
    },
    {
        'check': 'probability_contract',
        'expected': (
            RESIDUAL_EXPECTED_VALIDATION_ROWS,
            2,
        ),
        'observed': residual_probabilities.shape,
        'pass': (
            residual_probabilities.shape
            == (RESIDUAL_EXPECTED_VALIDATION_ROWS, 2)
            and np.isfinite(residual_probabilities).all()
            and np.allclose(
                residual_probabilities.sum(axis=1),
                1.0,
            )
            and np.array_equal(
                np.asarray(
                    residual_result_52d.model.classes_,
                    dtype=np.int64,
                ),
                np.array([0, 1], dtype=np.int64),
            )
            and np.array_equal(
                residual_prediction,
                np.argmax(
                    residual_probabilities,
                    axis=1,
                ).astype(np.int64),
            )
        ),
    },
]
residual_52d_contract_audit = pd.DataFrame(
    residual_contract_rows
)
display(residual_52d_contract_audit)

failed_residual_contract = residual_52d_contract_audit.loc[
    ~residual_52d_contract_audit['pass']
].reset_index(drop=True)
if not failed_residual_contract.empty:
    print('52D residual analysis stopped: contract mismatch')
    display(failed_residual_contract)
    raise RuntimeError(
        'STOP: current 52D reference does not match the frozen '
        'residual-analysis contract.'
    )

print(
    '52D residual-analysis contract passed:',
    residual_reference_source,
)

,check,expected,observed,pass
0,reference_source,"existing objects, otherwise cache-only 52D rec...",reused_canonical,True
1,validation_rows,177891,177891,True
2,accuracy,0.869532,0.869532,True
3,confusion_matrix,"[[71066, 14285], [8924, 83616]]","[[71066, 14285], [8924, 83616]]",True
4,validation_record_key_unique,True,True,True
5,train_validation_row_key_label_alignment,True,True,True
6,52D_feature_order,"(mfcc_01_mean, mfcc_02_mean, mfcc_03_mean, mfc...","(mfcc_01_mean, mfcc_02_mean, mfcc_03_mean, mfc...",True
7,M04_max_iter_100_seed_42,"{'hidden_layer_sizes': (64,), 'activation': 'r...","{'hidden_layer_sizes': (64,), 'activation': 'r...",True
8,internal_train_only_scaler,711467,711467,True
9,probability_contract,"(177891, 2)","(177891, 2)",True


52D residual-analysis contract passed: reused_canonical


In [14]:
# 2. Build the row-level residual ledger and overall error summary
residual_tracking = e05_cache.tracking.iloc[
    residual_validation_idx
].reset_index(drop=True).copy()
if not np.array_equal(
    residual_tracking['record_key'].astype(str).to_numpy(),
    residual_validation_keys,
):
    raise RuntimeError(
        'STOP: tracking rows differ from validation record_key order'
    )

residual_duration_sec = (
    pd.to_numeric(
        residual_tracking['endAt_ms'],
        errors='raise',
    ).to_numpy(dtype=np.float64)
    - pd.to_numeric(
        residual_tracking['startAt_ms'],
        errors='raise',
    ).to_numpy(dtype=np.float64)
) / 1_000.0
RESIDUAL_DURATION_ORDER = (
    '<0.5s',
    '0.5-1.0s',
    '1-2s',
    '2-5s',
    '>=5s',
)
residual_duration_slice = pd.cut(
    residual_duration_sec,
    bins=[-np.inf, 0.5, 1.0, 2.0, 5.0, np.inf],
    labels=RESIDUAL_DURATION_ORDER,
    right=False,
    ordered=True,
)

residual_overlap_work = residual_tracking.loc[
    :, ['call_id', 'startAt_ms', 'endAt_ms']
].copy()
residual_overlap_work['_row'] = np.arange(
    len(residual_overlap_work),
    dtype=np.int64,
)
residual_overlap_work['call_id'] = residual_overlap_work[
    'call_id'
].astype(str)
residual_overlap_work['startAt_ms'] = pd.to_numeric(
    residual_overlap_work['startAt_ms'],
    errors='raise',
)
residual_overlap_work['endAt_ms'] = pd.to_numeric(
    residual_overlap_work['endAt_ms'],
    errors='raise',
)
residual_overlap_ordered = residual_overlap_work.sort_values(
    ['call_id', 'startAt_ms', 'endAt_ms', '_row'],
    kind='stable',
)
residual_previous_max_end = residual_overlap_ordered.groupby(
    'call_id',
    sort=False,
)['endAt_ms'].transform(
    lambda values: values.cummax().shift(1)
)
residual_next_start = residual_overlap_ordered.groupby(
    'call_id',
    sort=False,
)['startAt_ms'].shift(-1)
residual_overlap_ordered['_overlap'] = (
    (
        residual_overlap_ordered['startAt_ms']
        < residual_previous_max_end
    )
    | (
        residual_next_start
        < residual_overlap_ordered['endAt_ms']
    )
).fillna(False)
residual_overlap = residual_overlap_ordered.sort_values(
    '_row',
    kind='stable',
)['_overlap'].to_numpy(dtype=bool)

residual_probability_0 = residual_probabilities[:, 0]
residual_probability_1 = residual_probabilities[:, 1]
residual_confidence = residual_probabilities.max(axis=1)
residual_correct = residual_prediction == residual_y_true
residual_error_type = np.full(
    len(residual_y_true),
    'CORRECT',
    dtype=object,
)
residual_error_type[
    (residual_y_true == 0) & (residual_prediction == 1)
] = 'FP'
residual_error_type[
    (residual_y_true == 1) & (residual_prediction == 0)
] = 'FN'
residual_near_boundary = (
    (residual_probability_1 >= 0.45)
    & (residual_probability_1 <= 0.55)
)
residual_high_confidence_error = (
    (~residual_correct)
    & (residual_confidence >= 0.90)
)

residual_error_ledger = pd.DataFrame({
    'record_key': residual_validation_keys,
    'call_id': residual_tracking['call_id'].astype(str).to_numpy(),
    'startAt_ms': pd.to_numeric(
        residual_tracking['startAt_ms'],
        errors='raise',
    ).to_numpy(dtype=np.int64),
    'endAt_ms': pd.to_numeric(
        residual_tracking['endAt_ms'],
        errors='raise',
    ).to_numpy(dtype=np.int64),
    'y_true': residual_y_true,
    'prediction': residual_prediction,
    'probability_0': residual_probability_0,
    'probability_1': residual_probability_1,
    'confidence': residual_confidence,
    'correct': residual_correct,
    'outcome': np.where(
        residual_correct,
        'correct',
        'error',
    ),
    'error_type': residual_error_type,
    'duration_sec': residual_duration_sec,
    'duration_slice': residual_duration_slice,
    'overlap': residual_overlap,
    'high_confidence_error': residual_high_confidence_error,
    'near_boundary': residual_near_boundary,
    'boundary_distance': np.abs(
        residual_probability_1 - 0.5
    ),
})
if len(residual_error_ledger) != RESIDUAL_EXPECTED_VALIDATION_ROWS:
    raise AssertionError('residual ledger row count changed')
if not residual_error_ledger['record_key'].is_unique:
    raise AssertionError('residual ledger record_key is not unique')
if residual_error_ledger.isna().loc[
    :, [
        'record_key',
        'call_id',
        'y_true',
        'prediction',
        'probability_0',
        'probability_1',
        'confidence',
        'duration_sec',
        'duration_slice',
        'overlap',
    ]
].any().any():
    raise AssertionError('residual ledger contains required nulls')

def _residual_metric_row(slice_name, selected):
    selected = np.asarray(selected, dtype=bool)
    truth = residual_y_true[selected]
    prediction = residual_prediction[selected]
    count = int(np.sum(selected))
    true_negative = int(np.sum(
        (truth == 0) & (prediction == 0)
    ))
    false_positive = int(np.sum(
        (truth == 0) & (prediction == 1)
    ))
    false_negative = int(np.sum(
        (truth == 1) & (prediction == 0)
    ))
    true_positive = int(np.sum(
        (truth == 1) & (prediction == 1)
    ))
    error_count = false_positive + false_negative
    speaker_0_count = true_negative + false_positive
    speaker_1_count = true_positive + false_negative
    return {
        'slice': slice_name,
        'count': count,
        'error_count': error_count,
        'error_rate': (
            error_count / count if count else np.nan
        ),
        'accuracy': (
            (true_negative + true_positive) / count
            if count else np.nan
        ),
        'recall_speaker_0': (
            true_negative / speaker_0_count
            if speaker_0_count else np.nan
        ),
        'recall_speaker_1': (
            true_positive / speaker_1_count
            if speaker_1_count else np.nan
        ),
        'FP': false_positive,
        'FN': false_negative,
        'confusion_matrix': [
            [true_negative, false_positive],
            [false_negative, true_positive],
        ],
    }

residual_overall_summary = pd.DataFrame([
    _residual_metric_row(
        'all_validation',
        np.ones(len(residual_error_ledger), dtype=bool),
    )
])
display(residual_overall_summary)
print(
    'Residual ledger:',
    residual_error_ledger.shape,
    '| unique record_key:',
    residual_error_ledger['record_key'].is_unique,
)
display(residual_error_ledger.head(20))

,slice,count,error_count,error_rate,accuracy,recall_speaker_0,recall_speaker_1,FP,FN,confusion_matrix
0,all_validation,177891,23209,0.130468,0.869532,0.832632,0.903566,14285,8924,"[[71066, 14285], [8924, 83616]]"


Residual ledger: (177891, 18) | unique record_key: True


,record_key,call_id,startAt_ms,endAt_ms,y_true,prediction,probability_0,probability_1,confidence,correct,outcome,error_type,duration_sec,duration_slice,overlap,high_confidence_error,near_boundary,boundary_distance
0,Training|651e464d69a4f266f0626820_20220101|0,651e464d69a4f266f0626820_20220101,502,2072,0,0,0.999526,0.000474,0.999526,True,correct,CORRECT,1.570,1-2s,False,False,False,0.499526
1,Training|651e464d69a4f266f0626820_20220101|1,651e464d69a4f266f0626820_20220101,2422,5220,1,1,0.005933,0.994067,0.994067,True,correct,CORRECT,2.798,2-5s,False,False,False,0.494067
2,Training|651e464d69a4f266f0626820_20220101|2,651e464d69a4f266f0626820_20220101,5729,6390,0,0,0.986182,0.013818,0.986182,True,correct,CORRECT,0.661,0.5-1.0s,True,False,False,0.486182
3,Training|651e464d69a4f266f0626820_20220101|3,651e464d69a4f266f0626820_20220101,6291,6828,1,1,0.326688,0.673312,0.673312,True,correct,CORRECT,0.537,0.5-1.0s,True,False,False,0.173312
4,Training|651e464d69a4f266f0626820_20220101|4,651e464d69a4f266f0626820_20220101,7228,7646,0,0,0.881058,0.118942,0.881058,True,correct,CORRECT,0.418,<0.5s,False,False,False,0.381058
5,Training|651e464d69a4f266f0626820_20220101|5,651e464d69a4f266f0626820_20220101,7648,10558,1,1,0.041991,0.958009,0.958009,True,correct,CORRECT,2.910,2-5s,True,False,False,0.458009
6,Training|651e464d69a4f266f0626820_20220101|6,651e464d69a4f266f0626820_20220101,10421,12209,0,0,0.997097,0.002903,0.997097,True,correct,CORRECT,1.788,1-2s,True,False,False,0.497097
7,Training|651e464d69a4f266f0626820_20220101|7,651e464d69a4f266f0626820_20220101,12490,13568,1,1,0.014510,0.985490,0.985490,True,correct,CORRECT,1.078,1-2s,False,False,False,0.485490
8,Training|651e464d69a4f266f0626820_20220101|8,651e464d69a4f266f0626820_20220101,13963,18164,0,0,0.985272,0.014728,0.985272,True,correct,CORRECT,4.201,2-5s,True,False,False,0.485272
9,Training|651e464d69a4f266f0626820_20220101|9,651e464d69a4f266f0626820_20220101,15158,15485,1,1,0.055339,0.944661,0.944661,True,correct,CORRECT,0.327,<0.5s,True,False,False,0.444661


In [26]:
# 3. Duration and overlap residual summaries
residual_duration_rows = []
for duration_name in RESIDUAL_DURATION_ORDER:
    duration_mask = (
        residual_error_ledger['duration_slice'].astype(str).to_numpy()
        == duration_name
    )
    duration_row = _residual_metric_row(
        duration_name,
        duration_mask,
    )
    duration_row['duration_slice'] = duration_row.pop('slice')
    residual_duration_rows.append(duration_row)
residual_duration_summary = pd.DataFrame(
    residual_duration_rows
)

residual_overlap_rows = []
for overlap_name, overlap_mask in (
    ('overlap', residual_overlap),
    ('no_overlap', ~residual_overlap),
):
    overlap_row = _residual_metric_row(
        overlap_name,
        overlap_mask,
    )
    overlap_row['overlap_slice'] = overlap_row.pop('slice')
    residual_overlap_rows.append(overlap_row)
residual_overlap_summary = pd.DataFrame(
    residual_overlap_rows
)

display(residual_duration_summary)
display(residual_overlap_summary)

,count,error_count,error_rate,accuracy,recall_speaker_0,recall_speaker_1,FP,FN,confusion_matrix,duration_slice
0,14076,1921,0.136473,0.863527,0.886296,0.842678,765,1156,"[[5963, 765], [1156, 6192]]",<0.5s
1,43199,7534,0.174402,0.825598,0.763436,0.877192,4635,2899,"[[14958, 4635], [2899, 20707]]",0.5-1.0s
2,58320,8914,0.152846,0.847154,0.805850,0.897482,6219,2695,"[[25813, 6219], [2695, 23593]]",1-2s
3,51015,4325,0.084779,0.915221,0.895130,0.932284,2457,1868,"[[20972, 2457], [1868, 25718]]",2-5s
4,11281,515,0.045652,0.954348,0.941440,0.960322,209,306,"[[3360, 209], [306, 7406]]",>=5s


,count,error_count,error_rate,accuracy,recall_speaker_0,recall_speaker_1,FP,FN,confusion_matrix,overlap_slice
0,78581,13528,0.172154,0.827846,0.799063,0.855389,7721,5807,"[[30704, 7721], [5807, 34349]]",overlap
1,99310,9681,0.097483,0.902517,0.860120,0.940497,6564,3117,"[[40362, 6564], [3117, 49267]]",no_overlap


In [27]:
# 4. Interaction slices with an explicit small-sample marker
RESIDUAL_MIN_INTERACTION_SAMPLE_COUNT = 100
residual_error_ledger['overlap_slice'] = pd.Categorical(
    np.where(
        residual_error_ledger['overlap'],
        'overlap',
        'no_overlap',
    ),
    categories=['overlap', 'no_overlap'],
    ordered=True,
)

def _build_residual_interaction_table(group_columns):
    grouped = residual_error_ledger.groupby(
        list(group_columns),
        observed=False,
        dropna=False,
        sort=True,
    )
    table = grouped.agg(
        count=('record_key', 'size'),
        error_count=('correct', lambda values: int(
            np.sum(~values.to_numpy(dtype=bool))
        )),
    ).reset_index()
    table['error_rate'] = np.where(
        table['count'] > 0,
        table['error_count'] / table['count'],
        np.nan,
    )
    table['accuracy'] = 1.0 - table['error_rate']
    table['sample_too_small'] = (
        table['count'] < RESIDUAL_MIN_INTERACTION_SAMPLE_COUNT
    )
    table['small_sample_threshold'] = (
        RESIDUAL_MIN_INTERACTION_SAMPLE_COUNT
    )
    return table

residual_speaker_duration = _build_residual_interaction_table(
    ('y_true', 'duration_slice')
)
residual_speaker_overlap = _build_residual_interaction_table(
    ('y_true', 'overlap_slice')
)
residual_duration_overlap = _build_residual_interaction_table(
    ('duration_slice', 'overlap_slice')
)
residual_speaker_duration_overlap = (
    _build_residual_interaction_table(
        ('y_true', 'duration_slice', 'overlap_slice')
    )
)

print('speaker × duration')
display(residual_speaker_duration)
print('speaker × overlap')
display(residual_speaker_overlap)
print('duration × overlap')
display(residual_duration_overlap)
print('speaker × duration × overlap')
display(residual_speaker_duration_overlap)

speaker × duration


,y_true,duration_slice,count,error_count,error_rate,accuracy,sample_too_small,small_sample_threshold
0,0,<0.5s,6728,765,0.113704,0.886296,False,100
1,0,0.5-1.0s,19593,4635,0.236564,0.763436,False,100
2,0,1-2s,32032,6219,0.194150,0.805850,False,100
3,0,2-5s,23429,2457,0.104870,0.895130,False,100
4,0,>=5s,3569,209,0.058560,0.941440,False,100
5,1,<0.5s,7348,1156,0.157322,0.842678,False,100
6,1,0.5-1.0s,23606,2899,0.122808,0.877192,False,100
7,1,1-2s,26288,2695,0.102518,0.897482,False,100
8,1,2-5s,27586,1868,0.067716,0.932284,False,100
9,1,>=5s,7712,306,0.039678,0.960322,False,100


speaker × overlap


,y_true,overlap_slice,count,error_count,error_rate,accuracy,sample_too_small,small_sample_threshold
0,0,overlap,38425,7721,0.200937,0.799063,False,100
1,0,no_overlap,46926,6564,0.139880,0.860120,False,100
2,1,overlap,40156,5807,0.144611,0.855389,False,100
3,1,no_overlap,52384,3117,0.059503,0.940497,False,100


duration × overlap


,duration_slice,overlap_slice,count,error_count,error_rate,accuracy,sample_too_small,small_sample_threshold
0,<0.5s,overlap,4037,959,0.237553,0.762447,False,100
1,<0.5s,no_overlap,10039,962,0.095826,0.904174,False,100
2,0.5-1.0s,overlap,17847,4396,0.246316,0.753684,False,100
3,0.5-1.0s,no_overlap,25352,3138,0.123777,0.876223,False,100
4,1-2s,overlap,25653,5156,0.200990,0.799010,False,100
5,1-2s,no_overlap,32667,3758,0.115040,0.884960,False,100
6,2-5s,overlap,25298,2728,0.107835,0.892165,False,100
7,2-5s,no_overlap,25717,1597,0.062099,0.937901,False,100
8,>=5s,overlap,5746,289,0.050296,0.949704,False,100
9,>=5s,no_overlap,5535,226,0.040831,0.959169,False,100


speaker × duration × overlap


,y_true,duration_slice,overlap_slice,count,error_count,error_rate,accuracy,sample_too_small,small_sample_threshold
0,0,<0.5s,overlap,1909,405,0.212153,0.787847,False,100
1,0,<0.5s,no_overlap,4819,360,0.074704,0.925296,False,100
2,0,0.5-1.0s,overlap,7976,2587,0.324348,0.675652,False,100
3,0,0.5-1.0s,no_overlap,11617,2048,0.176293,0.823707,False,100
4,0,1-2s,overlap,13517,3145,0.232670,0.767330,False,100
5,0,1-2s,no_overlap,18515,3074,0.166028,0.833972,False,100
6,0,2-5s,overlap,12904,1474,0.114228,0.885772,False,100
7,0,2-5s,no_overlap,10525,983,0.093397,0.906603,False,100
8,0,>=5s,overlap,2119,110,0.051911,0.948089,False,100
9,0,>=5s,no_overlap,1450,99,0.068276,0.931724,False,100


In [28]:
# 5-6. Confidence slices and high-confidence error detail
RESIDUAL_CONFIDENCE_ORDER = (
    '<0.60',
    '0.60-0.70',
    '0.70-0.80',
    '0.80-0.90',
    '>=0.90',
)
residual_error_ledger['confidence_slice'] = pd.cut(
    residual_error_ledger['confidence'],
    bins=[-np.inf, 0.60, 0.70, 0.80, 0.90, np.inf],
    labels=RESIDUAL_CONFIDENCE_ORDER,
    right=False,
    ordered=True,
)

residual_confidence_rows = []
for confidence_name in RESIDUAL_CONFIDENCE_ORDER:
    confidence_mask = (
        residual_error_ledger['confidence_slice']
        .astype(str)
        .to_numpy()
        == confidence_name
    )
    confidence_row = _residual_metric_row(
        confidence_name,
        confidence_mask,
    )
    residual_confidence_rows.append({
        'confidence_slice': confidence_name,
        'count': confidence_row['count'],
        'accuracy': confidence_row['accuracy'],
        'FP': confidence_row['FP'],
        'FN': confidence_row['FN'],
    })
residual_confidence_summary = pd.DataFrame(
    residual_confidence_rows
)

residual_focus_specs = (
    (
        'high_confidence_error',
        'confidence >= 0.90 and prediction is wrong',
        residual_error_ledger[
            'high_confidence_error'
        ].to_numpy(dtype=bool),
    ),
    (
        'low_confidence_correct',
        'confidence < 0.60 and prediction is correct',
        (
            (residual_confidence < 0.60)
            & residual_correct
        ),
    ),
    (
        'near_boundary_all',
        'probability_1 is between 0.45 and 0.55 inclusive',
        residual_near_boundary,
    ),
    (
        'near_boundary_error',
        'near boundary and prediction is wrong',
        residual_near_boundary & (~residual_correct),
    ),
    (
        'near_boundary_correct',
        'near boundary and prediction is correct',
        residual_near_boundary & residual_correct,
    ),
)
residual_confidence_focus_rows = []
for focus_name, definition, focus_mask in residual_focus_specs:
    focus_row = _residual_metric_row(
        focus_name,
        focus_mask,
    )
    residual_confidence_focus_rows.append({
        'focus': focus_name,
        'definition': definition,
        **{
            key: value
            for key, value in focus_row.items()
            if key != 'slice'
        },
    })
residual_confidence_focus_summary = pd.DataFrame(
    residual_confidence_focus_rows
)

residual_high_confidence_errors = residual_error_ledger.loc[
    residual_error_ledger['high_confidence_error']
].copy()
residual_high_confidence_errors['error_meaning'] = (
    residual_high_confidence_errors['error_type'].map({
        'FP': (
            'actual speaker 0 (119 responder), '
            'strongly predicted speaker 1 (caller)'
        ),
        'FN': (
            'actual speaker 1 (caller), '
            'strongly predicted speaker 0 (119 responder)'
        ),
    })
)

def _high_confidence_group_count(group_columns):
    if residual_high_confidence_errors.empty:
        return pd.DataFrame(
            columns=[*group_columns, 'count', 'rate_within_high_conf_errors']
        )
    grouped = (
        residual_high_confidence_errors.groupby(
            list(group_columns),
            observed=False,
            dropna=False,
            sort=True,
        )
        .size()
        .rename('count')
        .reset_index()
    )
    grouped['rate_within_high_conf_errors'] = (
        grouped['count']
        / len(residual_high_confidence_errors)
    )
    return grouped

residual_high_confidence_by_error_type = (
    _high_confidence_group_count(('error_type',))
)
if not residual_high_confidence_by_error_type.empty:
    residual_high_confidence_by_error_type['meaning'] = (
        residual_high_confidence_by_error_type['error_type'].map({
            'FP': (
                'actual speaker 0 -> predicted speaker 1'
            ),
            'FN': (
                'actual speaker 1 -> predicted speaker 0'
            ),
        })
    )
residual_high_confidence_by_speaker = (
    _high_confidence_group_count(('y_true',))
)
residual_high_confidence_by_duration = (
    _high_confidence_group_count(('duration_slice',))
)
residual_high_confidence_by_overlap = (
    _high_confidence_group_count(('overlap_slice',))
)
residual_high_confidence_joint = (
    _high_confidence_group_count((
        'error_type',
        'y_true',
        'duration_slice',
        'overlap_slice',
    ))
)

display(residual_confidence_summary)
display(residual_confidence_focus_summary)
print('high-confidence error by FP/FN')
display(residual_high_confidence_by_error_type)
print('high-confidence error by speaker')
display(residual_high_confidence_by_speaker)
print('high-confidence error by duration')
display(residual_high_confidence_by_duration)
print('high-confidence error by overlap')
display(residual_high_confidence_by_overlap)
print('high-confidence error joint detail')
display(residual_high_confidence_joint)

,confidence_slice,count,accuracy,FP,FN
0,<0.60,18758,0.549152,5804,2653
1,0.60-0.70,15144,0.635301,3673,1850
2,0.70-0.80,14227,0.745554,2055,1565
3,0.80-0.90,20141,0.850107,1530,1489
4,>=0.90,109621,0.976373,1223,1367


,focus,definition,count,error_count,error_rate,accuracy,recall_speaker_0,recall_speaker_1,FP,FN,confusion_matrix
0,high_confidence_error,confidence >= 0.90 and prediction is wrong,2590,2590,1.000000,0.000000,0.0000,0.000000,1223,1367,"[[0, 1223], [1367, 0]]"
1,low_confidence_correct,confidence < 0.60 and prediction is correct,10301,0,0.000000,1.000000,1.0000,1.000000,0,0,"[[2867, 0], [0, 7434]]"
2,near_boundary_all,probability_1 is between 0.45 and 0.55 inclusive,8006,3766,0.470397,0.529603,0.3936,0.649436,2274,1492,"[[1476, 2274], [1492, 2764]]"
3,near_boundary_error,near boundary and prediction is wrong,3766,3766,1.000000,0.000000,0.0000,0.000000,2274,1492,"[[0, 2274], [1492, 0]]"
4,near_boundary_correct,near boundary and prediction is correct,4240,0,0.000000,1.000000,1.0000,1.000000,0,0,"[[1476, 0], [0, 2764]]"


high-confidence error by FP/FN


,error_type,count,rate_within_high_conf_errors,meaning
0,FN,1367,0.527799,actual speaker 1 -> predicted speaker 0
1,FP,1223,0.472201,actual speaker 0 -> predicted speaker 1


high-confidence error by speaker


...

,duration_slice,count,rate_within_high_conf_errors
0,<0.5s,291,0.112355
1,0.5-1.0s,810,0.312741
2,1-2s,913,0.352510
3,2-5s,484,0.186873
4,>=5s,92,0.035521


high-confidence error by overlap


,overlap_slice,count,rate_within_high_conf_errors
0,overlap,1758,0.678764
1,no_overlap,832,0.321236


high-confidence error joint detail


,error_type,y_true,duration_slice,overlap_slice,count,rate_within_high_conf_errors
0,FN,1,<0.5s,overlap,52,0.020077
1,FN,1,<0.5s,no_overlap,80,0.030888
2,FN,1,0.5-1.0s,overlap,276,0.106564
3,FN,1,0.5-1.0s,no_overlap,141,0.054440
4,FN,1,1-2s,overlap,362,0.139768
5,FN,1,1-2s,no_overlap,113,0.043629
6,FN,1,2-5s,overlap,196,0.075676
7,FN,1,2-5s,no_overlap,101,0.038996
8,FN,1,>=5s,overlap,19,0.007336
9,FN,1,>=5s,no_overlap,27,0.010425


In [29]:
# 7. Call-level concentration (analysis metadata only; never a model feature)
residual_call_error_concentration = (
    residual_error_ledger.groupby(
        'call_id',
        sort=False,
        dropna=False,
    )
    .agg(
        utterance_count=('record_key', 'size'),
        error_count=('correct', lambda values: int(
            np.sum(~values.to_numpy(dtype=bool))
        )),
    )
    .reset_index()
)
residual_call_error_concentration['error_rate'] = (
    residual_call_error_concentration['error_count']
    / residual_call_error_concentration['utterance_count']
)
residual_call_error_concentration = (
    residual_call_error_concentration.sort_values(
        ['error_count', 'error_rate', 'utterance_count', 'call_id'],
        ascending=[False, False, False, True],
        kind='stable',
    ).reset_index(drop=True)
)
residual_calls_by_error_rate = (
    residual_call_error_concentration.sort_values(
        ['error_rate', 'error_count', 'utterance_count', 'call_id'],
        ascending=[False, False, False, True],
        kind='stable',
    ).reset_index(drop=True)
)

residual_total_errors = int(
    residual_error_ledger['outcome'].eq('error').sum()
)
residual_call_concentration_summary = pd.DataFrame([{
    'call_count': len(residual_call_error_concentration),
    'calls_with_error': int(
        residual_call_error_concentration[
            'error_count'
        ].gt(0).sum()
    ),
    'total_error_count': residual_total_errors,
    'top_1_call_error_share': (
        residual_call_error_concentration.head(1)[
            'error_count'
        ].sum() / residual_total_errors
    ),
    'top_5_call_error_share': (
        residual_call_error_concentration.head(5)[
            'error_count'
        ].sum() / residual_total_errors
    ),
    'top_10_call_error_share': (
        residual_call_error_concentration.head(10)[
            'error_count'
        ].sum() / residual_total_errors
    ),
    'top_20_call_error_share': (
        residual_call_error_concentration.head(20)[
            'error_count'
        ].sum() / residual_total_errors
    ),
    'median_call_error_rate': (
        residual_call_error_concentration['error_rate'].median()
    ),
    'p90_call_error_rate': (
        residual_call_error_concentration['error_rate'].quantile(0.90)
    ),
    'p95_call_error_rate': (
        residual_call_error_concentration['error_rate'].quantile(0.95)
    ),
    'p99_call_error_rate': (
        residual_call_error_concentration['error_rate'].quantile(0.99)
    ),
    'max_call_error_rate': (
        residual_call_error_concentration['error_rate'].max()
    ),
}])

display(residual_call_concentration_summary)
print('Top 20 calls by error count')
display(residual_call_error_concentration.head(20))
print('Top 20 calls by error rate')
display(residual_calls_by_error_rate.head(20))
print('call_id is used only for residual analysis and sampling.')

,call_count,calls_with_error,total_error_count,top_1_call_error_share,top_5_call_error_share,top_10_call_error_share,top_20_call_error_share,median_call_error_rate,p90_call_error_rate,p95_call_error_rate,p99_call_error_rate,max_call_error_rate
0,5840,5439,23209,0.001379,0.005687,0.010298,0.01857,0.114286,0.230769,0.285714,0.410722,0.722222


Top 20 calls by error count


,call_id,utterance_count,error_count,error_rate
0,651e53da81ad054c47959829_20220202,62,32,0.516129
1,651e5494386c2a48273e4b7b_20220305,62,27,0.435484
2,651e4baacf5885969118c2ad_20220504,65,27,0.415385
3,651e5500835c8a556d7e2bfc_20220404,57,23,0.403509
4,651e539031ce2800caf02bc8_20220104,59,23,0.389831
5,651e4f9dddc447a00994573c_20220904,64,22,0.343750
6,651e5272f98a6671a3fee584_20221207,64,22,0.343750
7,651e54cf6032e904f5dcea18_20220402,43,21,0.488372
8,651e4c8ad24efd3f59d016c4_20220607,44,21,0.477273
9,651e5494386c2a48273e4baf_20220305,50,21,0.420000


Top 20 calls by error rate


,call_id,utterance_count,error_count,error_rate
0,651e4e712e98ee712096903c_20220704,18,13,0.722222
1,651e4f2166d8da7dcba97469_20220806,23,13,0.565217
2,651e4c3e8c3bda5f9fa38af4_20220601,16,9,0.562500
3,651e4c8ad24efd3f59d0172f_20220607,20,11,0.550000
4,651e5258be98814e96ca2a0b_20221205,20,11,0.550000
5,651e51d797589376a420aff5_20221201,13,7,0.538462
6,651e4b1829d82455557a238d_20220502,28,15,0.535714
7,651e4932d392f1f6bd520f74_20220102,36,19,0.527778
8,651e4bfaf12d88de38d497f9_20220507,19,10,0.526316
9,651e494aaf1b6ab3ae0e56b6_20220105,27,14,0.518519


call_id is used only for residual analysis and sampling.


In [30]:
# 8. Deterministic manual-listening sample tables (maximum 20 per category)
residual_call_stats_by_id = (
    residual_call_error_concentration.set_index('call_id')
)
residual_error_ledger['call_utterance_count'] = (
    residual_error_ledger['call_id'].map(
        residual_call_stats_by_id['utterance_count']
    )
)
residual_error_ledger['call_error_count'] = (
    residual_error_ledger['call_id'].map(
        residual_call_stats_by_id['error_count']
    )
)
residual_error_ledger['call_error_rate'] = (
    residual_error_ledger['call_id'].map(
        residual_call_stats_by_id['error_rate']
    )
)

def _deterministic_listening_sample(
    category,
    selected,
    sort_columns,
    ascending,
    limit=20,
):
    sample = residual_error_ledger.loc[
        np.asarray(selected, dtype=bool)
    ].copy()
    sample = sample.sort_values(
        [*sort_columns, 'record_key'],
        ascending=[*ascending, True],
        kind='stable',
    ).head(limit)
    sample.insert(0, 'sample_category', category)
    sample.insert(
        1,
        'sample_rank',
        np.arange(1, len(sample) + 1),
    )
    return sample

residual_listening_sample_frames = [
    _deterministic_listening_sample(
        'high_confidence_FP',
        (
            residual_error_ledger['high_confidence_error']
            & residual_error_ledger['error_type'].eq('FP')
        ),
        ['confidence'],
        [False],
    ),
    _deterministic_listening_sample(
        'high_confidence_FN',
        (
            residual_error_ledger['high_confidence_error']
            & residual_error_ledger['error_type'].eq('FN')
        ),
        ['confidence'],
        [False],
    ),
    _deterministic_listening_sample(
        '0.5-1.0s_FP',
        (
            residual_error_ledger['duration_slice']
            .astype(str)
            .eq('0.5-1.0s')
            & residual_error_ledger['error_type'].eq('FP')
        ),
        ['confidence'],
        [False],
    ),
    _deterministic_listening_sample(
        '0.5-1.0s_FN',
        (
            residual_error_ledger['duration_slice']
            .astype(str)
            .eq('0.5-1.0s')
            & residual_error_ledger['error_type'].eq('FN')
        ),
        ['confidence'],
        [False],
    ),
    _deterministic_listening_sample(
        'overlap_FP',
        (
            residual_error_ledger['overlap']
            & residual_error_ledger['error_type'].eq('FP')
        ),
        ['confidence'],
        [False],
    ),
    _deterministic_listening_sample(
        'overlap_FN',
        (
            residual_error_ledger['overlap']
            & residual_error_ledger['error_type'].eq('FN')
        ),
        ['confidence'],
        [False],
    ),
    _deterministic_listening_sample(
        '<0.5s_error',
        (
            residual_error_ledger['duration_slice']
            .astype(str)
            .eq('<0.5s')
            & residual_error_ledger['outcome'].eq('error')
        ),
        ['confidence'],
        [False],
    ),
    _deterministic_listening_sample(
        'near_boundary_error',
        (
            residual_error_ledger['near_boundary']
            & residual_error_ledger['outcome'].eq('error')
        ),
        ['boundary_distance', 'confidence'],
        [True, True],
    ),
    _deterministic_listening_sample(
        'near_boundary_correct',
        (
            residual_error_ledger['near_boundary']
            & residual_error_ledger['outcome'].eq('correct')
        ),
        ['boundary_distance', 'confidence'],
        [True, True],
    ),
    _deterministic_listening_sample(
        'highest_confidence_error',
        residual_error_ledger['outcome'].eq('error'),
        ['confidence'],
        [False],
    ),
    _deterministic_listening_sample(
        'high_error_rate_call_errors',
        residual_error_ledger['outcome'].eq('error'),
        [
            'call_error_rate',
            'call_error_count',
            'call_utterance_count',
            'confidence',
        ],
        [False, False, False, False],
    ),
]
residual_manual_listening_samples = pd.concat(
    residual_listening_sample_frames,
    ignore_index=True,
)
residual_manual_listening_columns = [
    'sample_category',
    'sample_rank',
    'record_key',
    'call_id',
    'startAt_ms',
    'endAt_ms',
    'y_true',
    'prediction',
    'probability_0',
    'probability_1',
    'confidence',
    'outcome',
    'error_type',
    'duration_sec',
    'duration_slice',
    'overlap',
    'high_confidence_error',
    'near_boundary',
    'call_utterance_count',
    'call_error_count',
    'call_error_rate',
]
residual_manual_listening_samples = (
    residual_manual_listening_samples.loc[
        :, residual_manual_listening_columns
    ]
)
residual_manual_listening_counts = (
    residual_manual_listening_samples.groupby(
        'sample_category',
        sort=False,
    )
    .size()
    .rename('sample_count')
    .reset_index()
)

display(residual_manual_listening_counts)
with pd.option_context(
    'display.max_rows',
    len(residual_manual_listening_samples),
):
    display(residual_manual_listening_samples)
print(
    'Sampling is deterministic: stable sort plus record_key; '
    'no random sampling is used.'
)

,sample_category,sample_count
0,high_confidence_FP,20
1,high_confidence_FN,20
2,0.5-1.0s_FP,20
3,0.5-1.0s_FN,20
4,overlap_FP,20
5,overlap_FN,20
6,<0.5s_error,20
7,near_boundary_error,20
8,near_boundary_correct,20
9,highest_confidence_error,20


,sample_category,sample_rank,record_key,call_id,startAt_ms,endAt_ms,y_true,prediction,probability_0,probability_1,...,outcome,error_type,duration_sec,duration_slice,overlap,high_confidence_error,near_boundary,call_utterance_count,call_error_count,call_error_rate
0,high_confidence_FP,1,Training|651e4e712e98ee712096903c_20220704|11,651e4e712e98ee712096903c_20220704,24423,30624,0,1,0.000071,0.999929,...,error,FP,6.201,>=5s,False,True,False,18,13,0.722222
1,high_confidence_FP,2,Training|651e51b884f721c55a6dc70b_20221106|12,651e51b884f721c55a6dc70b_20221106,25531,26241,0,1,0.000094,0.999906,...,error,FP,0.710,0.5-1.0s,False,True,False,30,7,0.233333
2,high_confidence_FP,3,Training|651e4f2166d8da7dcba9729f_20220806|20,651e4f2166d8da7dcba9729f_20220806,53284,54475,0,1,0.000145,0.999855,...,error,FP,1.191,1-2s,True,True,False,42,5,0.119048
3,high_confidence_FP,4,Training|651e4a1fb7503416e5f1659e_20220302|2,651e4a1fb7503416e5f1659e_20220302,7812,14504,0,1,0.000167,0.999833,...,error,FP,6.692,>=5s,False,True,False,28,4,0.142857
4,high_confidence_FP,5,Training|651e4dfb2795332cde28f0d5_20220703|18,651e4dfb2795332cde28f0d5_20220703,48958,50917,0,1,0.000201,0.999799,...,error,FP,1.959,1-2s,True,True,False,37,7,0.189189
5,high_confidence_FP,6,Training|651e4f2166d8da7dcba9766d_20220806|16,651e4f2166d8da7dcba9766d_20220806,32223,32778,0,1,0.000224,0.999776,...,error,FP,0.555,0.5-1.0s,False,True,False,54,12,0.222222
6,high_confidence_FP,7,Training|651e4a6ede6495f4e9d36cc9_20220406|20,651e4a6ede6495f4e9d36cc9_20220406,57246,61876,0,1,0.000319,0.999681,...,error,FP,4.630,2-5s,False,True,False,27,1,0.037037
7,high_confidence_FP,8,Training|651e553ba8f14bddba4c130a_20220603|7,651e553ba8f14bddba4c130a_20220603,19745,20807,0,1,0.000412,0.999588,...,error,FP,1.062,1-2s,True,True,False,19,2,0.105263
8,high_confidence_FP,9,Training|651e5013d7e77fe70c0b00d5_20221002|10,651e5013d7e77fe70c0b00d5_20221002,17176,17876,0,1,0.000418,0.999582,...,error,FP,0.700,0.5-1.0s,True,True,False,53,7,0.132075
9,high_confidence_FP,10,Training|651e551aaa666596ee75fd1d_20220405|19,651e551aaa666596ee75fd1d_20220405,37901,39511,0,1,0.000517,0.999483,...,error,FP,1.610,1-2s,True,True,False,32,2,0.062500


Sampling is deterministic: stable sort plus record_key; no random sampling is used.


In [31]:
# 9-10. Assemble an objective weak-slice table without assigning severity
residual_slice_registry = []

def _register_residual_slice(slice_type, slice_name, selected):
    residual_slice_registry.append({
        'slice_type': slice_type,
        'slice': slice_name,
        'selected': np.asarray(selected, dtype=bool),
    })

for duration_name in RESIDUAL_DURATION_ORDER:
    _register_residual_slice(
        'duration',
        duration_name,
        (
            residual_error_ledger['duration_slice']
            .astype(str)
            .to_numpy()
            == duration_name
        ),
    )
for overlap_name, overlap_mask in (
    ('overlap', residual_overlap),
    ('no_overlap', ~residual_overlap),
):
    _register_residual_slice(
        'overlap',
        overlap_name,
        overlap_mask,
    )
for confidence_name in RESIDUAL_CONFIDENCE_ORDER:
    _register_residual_slice(
        'confidence',
        confidence_name,
        (
            residual_error_ledger['confidence_slice']
            .astype(str)
            .to_numpy()
            == confidence_name
        ),
    )

for speaker, duration_name in (
    residual_error_ledger.loc[
        :, ['y_true', 'duration_slice']
    ].drop_duplicates().itertuples(index=False, name=None)
):
    _register_residual_slice(
        'speaker_x_duration',
        f'speaker={speaker} | duration={duration_name}',
        (
            residual_error_ledger['y_true'].eq(speaker).to_numpy()
            & residual_error_ledger['duration_slice']
            .astype(str)
            .eq(str(duration_name))
            .to_numpy()
        ),
    )
for speaker, overlap_name in (
    residual_error_ledger.loc[
        :, ['y_true', 'overlap_slice']
    ].drop_duplicates().itertuples(index=False, name=None)
):
    _register_residual_slice(
        'speaker_x_overlap',
        f'speaker={speaker} | overlap={overlap_name}',
        (
            residual_error_ledger['y_true'].eq(speaker).to_numpy()
            & residual_error_ledger['overlap_slice']
            .astype(str)
            .eq(str(overlap_name))
            .to_numpy()
        ),
    )
for duration_name, overlap_name in (
    residual_error_ledger.loc[
        :, ['duration_slice', 'overlap_slice']
    ].drop_duplicates().itertuples(index=False, name=None)
):
    _register_residual_slice(
        'duration_x_overlap',
        f'duration={duration_name} | overlap={overlap_name}',
        (
            residual_error_ledger['duration_slice']
            .astype(str)
            .eq(str(duration_name))
            .to_numpy()
            & residual_error_ledger['overlap_slice']
            .astype(str)
            .eq(str(overlap_name))
            .to_numpy()
        ),
    )
for speaker, duration_name, overlap_name in (
    residual_error_ledger.loc[
        :, ['y_true', 'duration_slice', 'overlap_slice']
    ].drop_duplicates().itertuples(index=False, name=None)
):
    _register_residual_slice(
        'speaker_x_duration_x_overlap',
        (
            f'speaker={speaker} | duration={duration_name} '
            f'| overlap={overlap_name}'
        ),
        (
            residual_error_ledger['y_true'].eq(speaker).to_numpy()
            & residual_error_ledger['duration_slice']
            .astype(str)
            .eq(str(duration_name))
            .to_numpy()
            & residual_error_ledger['overlap_slice']
            .astype(str)
            .eq(str(overlap_name))
            .to_numpy()
        ),
    )

residual_all_slice_rows = []
for registered_slice in residual_slice_registry:
    metric_row = _residual_metric_row(
        registered_slice['slice'],
        registered_slice['selected'],
    )
    residual_all_slice_rows.append({
        'slice_type': registered_slice['slice_type'],
        **metric_row,
        'severity': 'not_scored',
    })
residual_all_slice_summary = pd.DataFrame(
    residual_all_slice_rows
)

residual_overall_error_rate = float(
    residual_overall_summary.loc[0, 'error_rate']
)
residual_weak_slice_summary = (
    residual_all_slice_summary.loc[
        (
            residual_all_slice_summary['count']
            >= RESIDUAL_MIN_INTERACTION_SAMPLE_COUNT
        )
        & (
            residual_all_slice_summary['error_rate']
            > residual_overall_error_rate
        ),
        [
            'slice_type',
            'slice',
            'count',
            'error_count',
            'accuracy',
            'error_rate',
            'recall_speaker_0',
            'recall_speaker_1',
            'FP',
            'FN',
            'severity',
        ],
    ]
    .sort_values(
        ['error_rate', 'error_count', 'count', 'slice_type', 'slice'],
        ascending=[False, False, False, True, True],
        kind='stable',
    )
    .reset_index(drop=True)
)

residual_weak_slice_reference = pd.DataFrame([{
    'slice_type': 'reference',
    'slice': 'all_validation',
    'count': int(residual_overall_summary.loc[0, 'count']),
    'error_count': int(
        residual_overall_summary.loc[0, 'error_count']
    ),
    'accuracy': float(
        residual_overall_summary.loc[0, 'accuracy']
    ),
    'error_rate': residual_overall_error_rate,
    'recall_speaker_0': float(
        residual_overall_summary.loc[
            0, 'recall_speaker_0'
        ]
    ),
    'recall_speaker_1': float(
        residual_overall_summary.loc[
            0, 'recall_speaker_1'
        ]
    ),
    'FP': int(residual_overall_summary.loc[0, 'FP']),
    'FN': int(residual_overall_summary.loc[0, 'FN']),
    'severity': 'not_scored',
}])
residual_weak_slice_summary = pd.concat(
    [
        residual_weak_slice_reference,
        residual_weak_slice_summary,
    ],
    ignore_index=True,
)

print(
    'Weak-slice inclusion rule: count >=',
    RESIDUAL_MIN_INTERACTION_SAMPLE_COUNT,
    'and error_rate > overall error_rate.',
)
print(
    'severity is intentionally not scored; '
    'use count and error_rate for user interpretation.'
)
display(residual_weak_slice_summary)

Weak-slice inclusion rule: count >= 100 and error_rate > overall error_rate.
severity is intentionally not scored; use count and error_rate for user interpretation.


,slice_type,slice,count,error_count,accuracy,error_rate,recall_speaker_0,recall_speaker_1,FP,FN,severity
0,reference,all_validation,177891,23209,0.869532,0.130468,0.832632,0.903566,14285,8924,not_scored
1,confidence,<0.60,18758,8457,0.549152,0.450848,0.330642,0.736988,5804,2653,not_scored
2,confidence,0.60-0.70,15144,5523,0.635301,0.364699,0.440177,0.784458,3673,1850,not_scored
3,speaker_x_duration_x_overlap,speaker=0 | duration=0.5-1.0s | overlap=overlap,7976,2587,0.675652,0.324348,0.675652,NaN,2587,0,not_scored
4,speaker_x_duration_x_overlap,speaker=1 | duration=<0.5s | overlap=overlap,2128,554,0.739662,0.260338,NaN,0.739662,0,554,not_scored
5,confidence,0.70-0.80,14227,3620,0.745554,0.254446,0.664161,0.806981,2055,1565,not_scored
6,duration_x_overlap,duration=0.5-1.0s | overlap=overlap,17847,4396,0.753684,0.246316,0.675652,0.816736,2587,1809,not_scored
7,duration_x_overlap,duration=<0.5s | overlap=overlap,4037,959,0.762447,0.237553,0.787847,0.739662,405,554,not_scored
8,speaker_x_duration,speaker=0 | duration=0.5-1.0s,19593,4635,0.763436,0.236564,0.763436,NaN,4635,0,not_scored
9,speaker_x_duration_x_overlap,speaker=0 | duration=1-2s | overlap=overlap,13517,3145,0.767330,0.232670,0.767330,NaN,3145,0,not_scored


In [32]:
# 10-sample Manual Listening
from pathlib import Path
from IPython.display import Audio, Markdown, display

manual_listening_source_candidates = (
    'residual_ledger',
    'residual_error_ledger',
    'listening_sample_table',
    'residual_manual_listening_samples',
)
manual_listening_required_columns = {
    'record_key',
    'y_true',
    'prediction',
    'probability_1',
    'confidence',
    'duration_sec',
    'overlap',
    'startAt_ms',
    'endAt_ms',
}
manual_listening_source_name = next(
    (
        name
        for name in manual_listening_source_candidates
        if name in globals()
        and isinstance(globals()[name], pd.DataFrame)
        and manual_listening_required_columns.issubset(
            globals()[name].columns
        )
    ),
    None,
)
if manual_listening_source_name is None:
    raise RuntimeError(
        'Run an existing residual ledger/listening sample cell with '
        f'these columns first: {sorted(manual_listening_required_columns)}. '
        'No model training or feature extraction is needed.'
    )
for required_name in (
    'RAW_DATA_ROOT',
    'training_manifest',
    'read_wav_frames',
):
    if required_name not in globals():
        raise RuntimeError(
            f'Missing existing audio resolver object: {required_name}. '
            'Run the earlier setup/cache-load cells only.'
        )

manual_listening_ledger = (
    globals()[manual_listening_source_name].copy()
)
manual_listening_ledger['record_key'] = (
    manual_listening_ledger['record_key'].astype(str)
)
manual_listening_ledger = (
    manual_listening_ledger.sort_values(
        'record_key',
        kind='stable',
    )
    .drop_duplicates('record_key', keep='first')
    .reset_index(drop=True)
)
for numeric_column in (
    'y_true',
    'prediction',
    'probability_1',
    'confidence',
    'duration_sec',
    'startAt_ms',
    'endAt_ms',
):
    manual_listening_ledger[numeric_column] = pd.to_numeric(
        manual_listening_ledger[numeric_column],
        errors='raise',
    )
manual_listening_ledger['overlap'] = (
    manual_listening_ledger['overlap'].astype(bool)
)
if not set(
    manual_listening_ledger['y_true'].astype(int).unique()
) <= {0, 1}:
    raise RuntimeError('Manual-listening y_true is not binary')
if not set(
    manual_listening_ledger['prediction'].astype(int).unique()
) <= {0, 1}:
    raise RuntimeError('Manual-listening prediction is not binary')
if not np.isfinite(
    manual_listening_ledger.loc[
        :,
        [
            'probability_1',
            'confidence',
            'duration_sec',
            'startAt_ms',
            'endAt_ms',
        ],
    ].to_numpy(dtype=np.float64)
).all():
    raise RuntimeError(
        'Manual-listening source contains non-finite values'
    )

manual_listening_ledger['error_type'] = np.select(
    (
        (
            manual_listening_ledger['y_true'].eq(0)
            & manual_listening_ledger['prediction'].eq(1)
        ),
        (
            manual_listening_ledger['y_true'].eq(1)
            & manual_listening_ledger['prediction'].eq(0)
        ),
    ),
    ('FP', 'FN'),
    default='CORRECT',
)
manual_listening_ledger['boundary_distance'] = np.abs(
    manual_listening_ledger['probability_1'] - 0.5
)

selected_manual_keys = set()
manual_selection_frames = []
manual_selection_audit_rows = []

def _select_manual_listening_rows(
    *,
    purpose,
    selected,
    requested_count,
    sort_columns,
    ascending,
):
    available = manual_listening_ledger.loc[
        np.asarray(selected, dtype=bool)
        & ~manual_listening_ledger['record_key'].isin(
            selected_manual_keys
        )
    ].copy()
    chosen = (
        available.sort_values(
            [*sort_columns, 'record_key'],
            ascending=[*ascending, True],
            kind='stable',
        )
        .head(requested_count)
        .copy()
    )
    manual_selection_audit_rows.append({
        'sample_purpose': purpose,
        'eligible_after_prior_exclusions': len(available),
        'requested_count': requested_count,
        'selected_count': len(chosen),
    })
    if len(chosen) == requested_count:
        chosen.insert(0, 'sample_purpose', purpose)
        manual_selection_frames.append(chosen)
        selected_manual_keys.update(
            chosen['record_key'].astype(str).tolist()
        )

duration_05_to_10 = (
    manual_listening_ledger['duration_sec'].ge(0.5)
    & manual_listening_ledger['duration_sec'].lt(1.0)
)
false_positive = manual_listening_ledger[
    'error_type'
].eq('FP')
high_confidence_error = (
    manual_listening_ledger['error_type'].ne('CORRECT')
    & manual_listening_ledger['confidence'].ge(0.90)
)
near_boundary_error = (
    manual_listening_ledger['error_type'].ne('CORRECT')
    & manual_listening_ledger['probability_1'].between(
        0.45,
        0.55,
        inclusive='both',
    )
)

_select_manual_listening_rows(
    purpose='FP_0.5-1.0s_overlap',
    selected=(
        false_positive
        & duration_05_to_10
        & manual_listening_ledger['overlap']
    ),
    requested_count=4,
    sort_columns=['confidence'],
    ascending=[False],
)
_select_manual_listening_rows(
    purpose='FP_0.5-1.0s_no_overlap',
    selected=(
        false_positive
        & duration_05_to_10
        & ~manual_listening_ledger['overlap']
    ),
    requested_count=2,
    sort_columns=['confidence'],
    ascending=[False],
)
_select_manual_listening_rows(
    purpose='high_confidence_FN',
    selected=(
        manual_listening_ledger['error_type'].eq('FN')
        & manual_listening_ledger['confidence'].ge(0.90)
    ),
    requested_count=2,
    sort_columns=['confidence'],
    ascending=[False],
)
_select_manual_listening_rows(
    purpose='<0.5s_high_confidence_error',
    selected=(
        high_confidence_error
        & manual_listening_ledger['duration_sec'].lt(0.5)
    ),
    requested_count=1,
    sort_columns=['confidence'],
    ascending=[False],
)
_select_manual_listening_rows(
    purpose='near_boundary_error',
    selected=near_boundary_error,
    requested_count=1,
    sort_columns=['boundary_distance'],
    ascending=[True],
)

manual_selection_audit = pd.DataFrame(
    manual_selection_audit_rows
)
display(manual_selection_audit)
if not manual_selection_audit[
    'selected_count'
].eq(
    manual_selection_audit['requested_count']
).all():
    raise RuntimeError(
        'STOP: at least one manual-listening category cannot '
        'supply its requested unique rows.'
    )

ten_sample_listening_table = pd.concat(
    manual_selection_frames,
    ignore_index=True,
)
ten_sample_listening_table.insert(
    0,
    'listening_no',
    np.arange(1, len(ten_sample_listening_table) + 1),
)
if len(ten_sample_listening_table) != 10:
    raise AssertionError(
        '10-sample listening table does not contain exactly 10 rows'
    )
if not ten_sample_listening_table['record_key'].is_unique:
    raise AssertionError(
        '10-sample listening record_key values are not unique'
    )

ten_sample_listening_table['y_true_role'] = (
    ten_sample_listening_table['y_true'].astype(int).map({
        0: '119대원',
        1: '신고자',
    })
)
ten_sample_listening_table['prediction_role'] = (
    ten_sample_listening_table['prediction'].astype(int).map({
        0: '119대원',
        1: '신고자',
    })
)
ten_sample_listening_table['label_guide'] = (
    '0 = 119대원, 1 = 신고자'
)

manual_audio_manifest = training_manifest.loc[
    :,
    [
        'record_key',
        'audio_path',
        'crop_start_frame',
        'crop_end_frame',
    ],
].copy()
manual_audio_manifest['record_key'] = (
    manual_audio_manifest['record_key'].astype(str)
)
if not manual_audio_manifest['record_key'].is_unique:
    raise RuntimeError(
        'Training audio manifest record_key is not unique'
    )
ten_sample_listening_table = (
    ten_sample_listening_table.merge(
        manual_audio_manifest,
        on='record_key',
        how='left',
        validate='one_to_one',
        sort=False,
    )
)
if ten_sample_listening_table[
    ['audio_path', 'crop_start_frame', 'crop_end_frame']
].isna().any().any():
    raise RuntimeError(
        'At least one selected row has no raw WAV crop mapping'
    )
if not np.array_equal(
    ten_sample_listening_table['listening_no'].to_numpy(),
    np.arange(1, 11),
):
    raise AssertionError(
        'Listening order changed during audio-path alignment'
    )

listening_sample_table_10 = ten_sample_listening_table.loc[
    :,
    [
        'listening_no',
        'sample_purpose',
        'record_key',
        'y_true',
        'y_true_role',
        'prediction',
        'prediction_role',
        'label_guide',
        'confidence',
        'duration_sec',
        'overlap',
        'error_type',
        'startAt_ms',
        'endAt_ms',
    ],
].copy()
print(
    'Manual listening source:',
    manual_listening_source_name,
)
print('Label guide: 0 = 119대원, 1 = 신고자')
display(listening_sample_table_10)

raw_wav_root = Path(RAW_DATA_ROOT).resolve()
for sample_row in ten_sample_listening_table.itertuples(
    index=False
):
    raw_wav_path = (
        raw_wav_root / str(sample_row.audio_path)
    ).resolve()
    if (
        raw_wav_path != raw_wav_root
        and raw_wav_root not in raw_wav_path.parents
    ):
        raise RuntimeError(
            f'Raw WAV path escaped RAW_DATA_ROOT: {raw_wav_path}'
        )
    if not raw_wav_path.is_file():
        raise FileNotFoundError(raw_wav_path)

    crop_start_frame = int(sample_row.crop_start_frame)
    crop_end_frame = int(sample_row.crop_end_frame)
    audio_segment = read_wav_frames(
        raw_wav_path,
        crop_start_frame,
        crop_end_frame,
    )
    crop_duration_sec = (
        crop_end_frame - crop_start_frame
    ) / audio_segment.sample_rate
    if not np.isclose(
        crop_duration_sec,
        float(sample_row.duration_sec),
        rtol=0.0,
        atol=(1.0 / audio_segment.sample_rate) + 1e-9,
    ):
        raise RuntimeError(
            'Selected WAV crop duration differs from ledger duration: '
            f'{sample_row.record_key}'
        )

    display(Markdown(
        f'### Listening {int(sample_row.listening_no)} '
        f'— {sample_row.sample_purpose}\n\n'
        f'- record_key: {sample_row.record_key}\n'
        f'- y_true: {int(sample_row.y_true)} '
        f'({sample_row.y_true_role})\n'
        f'- prediction: {int(sample_row.prediction)} '
        f'({sample_row.prediction_role})\n'
        f'- confidence: {float(sample_row.confidence):.6f}\n'
        f'- duration_sec: {float(sample_row.duration_sec):.3f}\n'
        f'- overlap: {bool(sample_row.overlap)}\n'
        f'- error_type: {sample_row.error_type}\n'
        f'- interval: [{int(sample_row.startAt_ms)}, '
        f'{int(sample_row.endAt_ms)}) ms'
    ))
    display(Audio(
        audio_segment.waveform.mean(axis=0),
        rate=audio_segment.sample_rate,
    ))
    display(Markdown(
        '**들리는 사람 발화:**  \n\n'
        '**두 화자 겹침 정도:**  \n\n'
        '**잡음/무음/비언어음:**  \n\n'
        '**발화 잘림:**  \n\n'
        '**사람 기준 화자 판단 난이도:**  \n\n'
        '**특이사항:**  '
    ))

,sample_purpose,eligible_after_prior_exclusions,requested_count,selected_count
0,FP_0.5-1.0s_overlap,2587,4,4
1,FP_0.5-1.0s_no_overlap,2048,2,2
2,high_confidence_FN,1367,2,2
3,<0.5s_high_confidence_error,291,1,1
4,near_boundary_error,3766,1,1


Manual listening source: residual_error_ledger
Label guide: 0 = 119대원, 1 = 신고자


,listening_no,sample_purpose,record_key,y_true,y_true_role,prediction,prediction_role,label_guide,confidence,duration_sec,overlap,error_type,startAt_ms,endAt_ms
0,1,FP_0.5-1.0s_overlap,Training|651e5013d7e77fe70c0b00d5_20221002|10,0,119대원,1,신고자,"0 = 119대원, 1 = 신고자",0.999582,0.700,True,FP,17176,17876
1,2,FP_0.5-1.0s_overlap,Training|651e4dfb2795332cde28f278_20220703|42,0,119대원,1,신고자,"0 = 119대원, 1 = 신고자",0.999406,0.631,True,FP,65766,66397
2,3,FP_0.5-1.0s_overlap,Training|651e4fcea6a4bdae3d84c4ac_20220907|16,0,119대원,1,신고자,"0 = 119대원, 1 = 신고자",0.999271,0.645,True,FP,45182,45827
3,4,FP_0.5-1.0s_overlap,Training|651e5272f98a6671a3fee5a0_20221207|13,0,119대원,1,신고자,"0 = 119대원, 1 = 신고자",0.999238,0.909,True,FP,24270,25179
4,5,FP_0.5-1.0s_no_overlap,Training|651e51b884f721c55a6dc70b_20221106|12,0,119대원,1,신고자,"0 = 119대원, 1 = 신고자",0.999906,0.710,False,FP,25531,26241
5,6,FP_0.5-1.0s_no_overlap,Training|651e4f2166d8da7dcba9766d_20220806|16,0,119대원,1,신고자,"0 = 119대원, 1 = 신고자",0.999776,0.555,False,FP,32223,32778
6,7,high_confidence_FN,Training|651e50ebf615d4163143bef5_20221101|13,1,신고자,0,119대원,"0 = 119대원, 1 = 신고자",0.999833,0.746,True,FN,24708,25454
7,8,high_confidence_FN,Training|651e551aaa666596ee75fe1c_20220405|26,1,신고자,0,119대원,"0 = 119대원, 1 = 신고자",0.999829,4.771,False,FN,65586,70357
8,9,<0.5s_high_confidence_error,Training|651e4d9be84a30cdf982d595_20220702|5,1,신고자,0,119대원,"0 = 119대원, 1 = 신고자",0.999242,0.363,False,FN,10669,11032
9,10,near_boundary_error,Training|651e51b884f721c55a6dc6be_20221106|22,1,신고자,0,119대원,"0 = 119대원, 1 = 신고자",0.500014,1.148,True,FN,73088,74236


### Listening 1 — FP_0.5-1.0s_overlap

- record_key: Training|651e5013d7e77fe70c0b00d5_20221002|10
- y_true: 0 (119대원)
- prediction: 1 (신고자)
- confidence: 0.999582
- duration_sec: 0.700
- overlap: True
- error_type: FP
- interval: [17176, 17876) ms

**들리는 사람 발화:**  

**두 화자 겹침 정도:**  

**잡음/무음/비언어음:**  

**발화 잘림:**  

**사람 기준 화자 판단 난이도:**  

**특이사항:**  

...

**들리는 사람 발화:**  

**두 화자 겹침 정도:**  

**잡음/무음/비언어음:**  

**발화 잘림:**  

**사람 기준 화자 판단 난이도:**  

**특이사항:**  

### Listening 9 — <0.5s_high_confidence_error

- record_key: Training|651e4d9be84a30cdf982d595_20220702|5
- y_true: 1 (신고자)
- prediction: 0 (119대원)
- confidence: 0.999242
- duration_sec: 0.363
- overlap: False
- error_type: FN
- interval: [10669, 11032) ms

**들리는 사람 발화:**  

**두 화자 겹침 정도:**  

**잡음/무음/비언어음:**  

**발화 잘림:**  

**사람 기준 화자 판단 난이도:**  

**특이사항:**  

### Listening 10 — near_boundary_error

- record_key: Training|651e51b884f721c55a6dc6be_20221106|22
- y_true: 1 (신고자)
- prediction: 0 (119대원)
- confidence: 0.500014
- duration_sec: 1.148
- overlap: True
- error_type: FN
- interval: [73088, 74236) ms

**들리는 사람 발화:**  

**두 화자 겹침 정도:**  

**잡음/무음/비언어음:**  

**발화 잘림:**  

**사람 기준 화자 판단 난이도:**  

**특이사항:**  

| 우선적으로 볼 slice                |  count | error rate |   error 수 | 의미                  |
| ---------------------------- | -----: | ---------: | --------: | ------------------- |
| **119대원 × 0.5~1초 × overlap** |  7,976 | **32.43%** |     2,587 | 가장 심한 핵심 약점         |
| 신고자 × <0.5초 × overlap        |  2,128 | **26.03%** |       554 | 초단발화 overlap 약점     |
| **0.5~1초 × overlap 전체**      | 17,847 | **24.63%** |     4,396 | 규모도 크고 오류율도 높음      |
| **119대원 × 1~2초 × overlap**   | 13,517 | **23.27%** |     3,145 | 두 번째 큰 119 약점       |
| <0.5초 × overlap 전체           |  4,037 |     23.76% |       959 | 짧은 overlap 약점       |
| 1~2초 × overlap 전체            | 25,653 | **20.10%** | **5,156** | 오류 절대 개수가 매우 큼      |
| 119대원 × overlap 전체           | 38,425 | **20.09%** | **7,721** | 119 + overlap 전반 문제 |
| 119대원 × 0.5~1초 전체            | 19,593 |     23.66% |     4,635 | overlap 없어도 약함      |


### Manual Listening - 10 Samples

Residual error 10개를 직접 청취했다.

대부분 사람이 들어도 화자 판단이 어려웠으며,
특히 발화가 짧거나 일부 잘린 sample이 반복적으로 관찰됐다.
"어이", "예", "예예", "에"처럼 화자 구분에 사용할 정보가 매우 적은 발화도 있었다.

또한 일부 sample은 발화 자체는 존재하지만
내용을 알아듣기 어려웠다.

반면 명확한 두 화자 동시 발화는 일부 sample에서만 관찰되어,
이번 10개 청취만으로 overlap 오류를 단순히
'두 목소리가 강하게 섞여서 발생한다'고 설명하기는 어렵다.

추가로 1개 sample은 청취상 119대원으로 판단되지만
y_true가 신고자로 되어 있어 annotation / segment alignment 확인 후보로 표시했다.

→ Manual inspection에서는
'짧음, 발화 잘림, 낮은 정보량, 청취 자체의 어려움'이
반복적으로 관찰된 주요 패턴이었다.

## Next Round — 52D residual-target acoustic blocks (AF10/AF11/AF12)

현재 재현 가능한 52D (Accuracy=0.869532, CM=[[71066,14285],[8924,83616]])만 reference로 사용한다. AF10/AF11/AF12는 각각 독립적으로 52D 뒤에 붙이며 조합하지 않는다. Feature extraction과 M04-100 fitting은 별도 기본-off gate를 통과해야 한다. Official Validation은 읽거나 사용하지 않는다.

- AF10 4D: current-utterance frame RMS activity와 crop boundary
- AF11 3D: frame normalized power-spectrum entropy dynamics
- AF12 3D: canonical MFCC20 early/middle/late contrast
- overlap/call metadata는 진단과 GroupSplit에만 사용하며 모델 feature가 아니다.


In [15]:
import importlib
import dcc_data.mission2_acoustic_feature_expansion as residual_round_module

expected_residual_round_module_path = (
    REPO_ROOT / 'src' / 'dcc_data' / 'mission2_acoustic_feature_expansion.py'
).resolve()
observed_residual_round_module_path = Path(
    residual_round_module.__file__
).resolve()
if observed_residual_round_module_path != expected_residual_round_module_path:
    raise RuntimeError(
        'STOP: imported acoustic expansion module is not the current repo file: '
        f'{observed_residual_round_module_path}'
    )
if not hasattr(residual_round_module, 'RESIDUAL_TARGET_BLOCK_IDS'):
    residual_round_module = importlib.reload(residual_round_module)
if not hasattr(residual_round_module, 'RESIDUAL_TARGET_BLOCK_IDS'):
    raise RuntimeError('STOP: AF10/AF11/AF12 source is not loaded')

RESIDUAL_TARGET_BLOCK_IDS = residual_round_module.RESIDUAL_TARGET_BLOCK_IDS
BLOCK_BY_ID = residual_round_module.BLOCK_BY_ID
candidate_manifest_table = residual_round_module.candidate_manifest_table
_prepare_representation = residual_round_module._prepare_representation
_run_prepared_representation = residual_round_module._run_prepared_representation
candidate_cache_paths = residual_round_module.candidate_cache_paths
load_candidate_feature_cache = residual_round_module.load_candidate_feature_cache

ALLOW_RESIDUAL_ROUND_SCREENING = False
RESIDUAL_ROUND_REFERENCE_ACCURACY = 0.869532466510391
RESIDUAL_ROUND_REFERENCE_CM = [[71_066, 14_285], [8_924, 83_616]]
RESIDUAL_ROUND_EXPECTED_BLOCK_IDS = (
    'AF10_SPEECH_ACTIVITY_BOUNDARY',
    'AF11_SPECTRAL_ENTROPY_DYNAMICS',
    'AF12_TEMPORAL_MFCC_CONTRAST',
)
if RESIDUAL_TARGET_BLOCK_IDS != RESIDUAL_ROUND_EXPECTED_BLOCK_IDS:
    raise RuntimeError('STOP: AF10/AF11/AF12 block contract changed')
residual_round_manifest = candidate_manifest_table().set_index('block_id').loc[
    list(RESIDUAL_TARGET_BLOCK_IDS)
].reset_index()
if residual_round_manifest['output_dimension'].tolist() != [4, 3, 3]:
    raise RuntimeError('STOP: AF10/AF11/AF12 dimensions changed')
display(residual_round_manifest)
print('Screening gate:', ALLOW_RESIDUAL_ROUND_SCREENING)


,block_id,feature_names,acoustic_meaning,calculation,e05_overlap,mission2_hypothesis,research_basis,rule_compliance,diagnostic_risks,expected_cost,output_dimension
0,AF10_SPEECH_ACTIVITY_BOUNDARY,"(active_frame_fraction, longest_active_run_fra...",within-utterance speech activity density and b...,centered Hann frame RMS (512/160); active thre...,"E05 retains global MFCC moments, rolloff, and ...","short, clipped, or low-information utterances ...",frame RMS is a deterministic speech-activity p...,current-utterance waveform only; no call-relat...,"gain, stationary noise, and crop padding can a...",low; one 512/160 frame-RMS pass,4
1,AF11_SPECTRAL_ENTROPY_DYNAMICS,"(spectral_entropy_mean, spectral_entropy_std, ...",frame spectral diffuseness and its within-utte...,centered Hann STFT 512/160; normalize each fra...,rolloff and MFCC moments summarize spectral lo...,"overlap, clipped speech, and low-information f...",normalized power-spectrum entropy is a bounded...,current-utterance waveform only; no annotation...,broadband noise can mimic high entropy and sil...,medium; one 512-point real FFT per frame,3
2,AF12_TEMPORAL_MFCC_CONTRAST,"(early_middle_distance, middle_late_distance, ...",coarse early/middle/late spectral-envelope change,reuse the canonical MFCC20 frame sequence; spl...,E05 MFCC mean/std discards coarse temporal loc...,short or boundary-clipped utterances may retai...,segment-level MFCC mean contrasts retain coars...,current-utterance waveform only; no absolute t...,very short utterances and padding can dominate...,medium; one canonical MFCC20 computation per u...,3


Screening gate: True


In [16]:
# One Training-only cache: each WAV is read once per call and all three blocks share the utterance load.
residual_round_parquet, residual_round_metadata = candidate_cache_paths(
    CANDIDATE_CACHE_ROOT, split='Training', block_ids=RESIDUAL_TARGET_BLOCK_IDS,
)
if not residual_round_parquet.is_file() or not residual_round_metadata.is_file():
    raise RuntimeError(
        'STOP: immutable AF10/AF11/AF12 cache is missing; extraction is disabled.'
    )
residual_round_feature_frame = load_candidate_feature_cache(
    manifest=utterance_manifest, cache_root=CANDIDATE_CACHE_ROOT,
    split='Training', block_ids=RESIDUAL_TARGET_BLOCK_IDS,
)
residual_round_feature_names = tuple(
    name for block_id in RESIDUAL_TARGET_BLOCK_IDS
    for name in BLOCK_BY_ID[block_id].feature_names
)
if len(residual_round_feature_names) != 10:
    raise AssertionError('combined cache must be 10D')
if not np.isfinite(
    residual_round_feature_frame.loc[:, residual_round_feature_names]
    .to_numpy(dtype=np.float32)
).all():
    raise ValueError('AF10/AF11/AF12 cache contains NaN or inf')
print('Combined cache:', residual_round_feature_frame.shape)


Combined cache: (889358, 12)


In [17]:
# Reconstruct the exact unscaled 52D prefix from immutable caches, then append one block only.
required_names = (
    'residual_result_52d', 'residual_prepared_52d',
    'residual_error_ledger', 'residual_round_feature_frame',
)
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        'Run the existing 52D residual contract and combined-cache cells first. '
        f'Missing: {missing_names}'
    )
if not np.isclose(
    residual_result_52d.metrics['accuracy'], RESIDUAL_ROUND_REFERENCE_ACCURACY,
    rtol=0.0, atol=5e-7,
):
    raise RuntimeError('STOP: current 52D Accuracy differs')
if residual_result_52d.metrics['confusion_matrix'] != RESIDUAL_ROUND_REFERENCE_CM:
    raise RuntimeError('STOP: current 52D confusion matrix differs')
if (
    residual_result_52d.model_config['max_iter'] != 100
    or residual_result_52d.fit_audit['seed'] != 42
    or residual_result_52d.fit_audit['scaler_fit_scope'] != 'internal_train_only'
):
    raise RuntimeError('STOP: current 52D model/scaler contract differs')

cache_af02_round = load_candidate_feature_cache(
    manifest=utterance_manifest, cache_root=CANDIDATE_CACHE_ROOT,
    split='Training', block_ids=ROUND1_BLOCK_IDS,
)
cache_af04_round = load_candidate_feature_cache(
    manifest=utterance_manifest, cache_root=CANDIDATE_CACHE_ROOT,
    split='Training', block_ids=ROUND2_BLOCK_IDS,
)
cache_af09_round = load_candidate_feature_cache(
    manifest=utterance_manifest, cache_root=CANDIDATE_CACHE_ROOT,
    split='Training', block_ids=NEXT_CANDIDATE_BLOCK_IDS,
)
round_keys = np.asarray(e05_cache.validated.record_keys).astype(str)
round_calls = e05_cache.tracking['call_id'].astype(str).to_numpy()
for cache_name, cache_frame in (
    ('AF02', cache_af02_round), ('AF04', cache_af04_round),
    ('AF09', cache_af09_round), ('AF10/11/12', residual_round_feature_frame),
):
    if not np.array_equal(cache_frame['record_key'].astype(str).to_numpy(), round_keys):
        raise RuntimeError(f'STOP: {cache_name} record_key order differs')
    if not np.array_equal(cache_frame['call_id'].astype(str).to_numpy(), round_calls):
        raise RuntimeError(f'STOP: {cache_name} call_id order differs')

af02_names_round = BLOCK_BY_ID['AF02_PITCH_VOICING'].feature_names
af04_names_round = BLOCK_BY_ID['AF04_CPP'].feature_names
af09_names_round = BLOCK_BY_ID['AF09_MULTIBAND_ENVELOPE_COHERENCE'].feature_names
feature_names_52d_round = (*FROZEN_49D_FEATURE_NAMES, *af09_names_round)
X_52d_round = np.concatenate(
    [
        np.asarray(e05_cache.validated.X, dtype=np.float32),
        cache_af02_round.loc[:, af02_names_round].to_numpy(dtype=np.float32),
        cache_af04_round.loc[:, af04_names_round].to_numpy(dtype=np.float32),
        cache_af09_round.loc[:, af09_names_round].to_numpy(dtype=np.float32),
    ],
    axis=1,
).astype(np.float32, copy=False)
if X_52d_round.shape != (len(round_keys), 52):
    raise AssertionError('reconstructed 52D matrix shape changed')
if tuple(residual_prepared_52d.feature_names) != feature_names_52d_round:
    raise RuntimeError('STOP: current 52D feature order changed')

reference_check_round = _prepare_representation(
    representation_id='AF09_52D_REFERENCE', X=X_52d_round,
    y=np.asarray(e05_cache.validated.y, dtype=np.int64),
    record_keys=round_keys, feature_names=feature_names_52d_round,
    train_idx=np.asarray(residual_prepared_52d.train_idx, dtype=np.int64),
    validation_idx=np.asarray(residual_prepared_52d.validation_idx, dtype=np.int64),
    split_audit=residual_prepared_52d.split_audit,
)
for array_name in ('X_train_scaled', 'X_evaluation_scaled'):
    if not np.array_equal(
        getattr(reference_check_round, array_name),
        getattr(residual_prepared_52d, array_name),
    ):
        raise RuntimeError(f'STOP: reconstructed 52D {array_name} differs')

prepared_residual_round = {'REF_52D': residual_prepared_52d}
for block_id in RESIDUAL_TARGET_BLOCK_IDS:
    block = BLOCK_BY_ID[block_id]
    block_matrix = residual_round_feature_frame.loc[
        :, block.feature_names
    ].to_numpy(dtype=np.float32)
    candidate_matrix = np.concatenate(
        [X_52d_round, block_matrix], axis=1,
    ).astype(np.float32, copy=False)
    if not np.array_equal(candidate_matrix[:, :52], X_52d_round):
        raise AssertionError(f'{block_id} changed the 52D prefix')
    prepared_residual_round[block_id] = _prepare_representation(
        representation_id=block_id, X=candidate_matrix,
        y=np.asarray(e05_cache.validated.y, dtype=np.int64),
        record_keys=round_keys,
        feature_names=(*feature_names_52d_round, *block.feature_names),
        train_idx=np.asarray(residual_prepared_52d.train_idx, dtype=np.int64),
        validation_idx=np.asarray(residual_prepared_52d.validation_idx, dtype=np.int64),
        split_audit=residual_prepared_52d.split_audit,
    )

expected_dimensions_round = {
    'REF_52D': 52,
    'AF10_SPEECH_ACTIVITY_BOUNDARY': 56,
    'AF11_SPECTRAL_ENTROPY_DYNAMICS': 55,
    'AF12_TEMPORAL_MFCC_CONTRAST': 55,
}
preparation_audit_round = pd.DataFrame([
    {
        'representation_id': representation_id,
        'dimension': len(prepared.feature_names),
        'expected_dimension': expected_dimensions_round[representation_id],
        'train_rows': len(prepared.train_idx),
        'validation_rows': len(prepared.validation_idx),
        'scaler_fit_rows': int(prepared.scaler.n_samples_seen_),
        'scaler_policy': 'internal_train_only',
        'preserves_52D_prefix': tuple(prepared.feature_names[:52]) == feature_names_52d_round,
        'single_candidate_only': (
            representation_id == 'REF_52D'
            or tuple(prepared.feature_names[52:])
            == BLOCK_BY_ID[representation_id].feature_names
        ),
    }
    for representation_id, prepared in prepared_residual_round.items()
])
if not (
    preparation_audit_round['dimension']
    == preparation_audit_round['expected_dimension']
).all():
    raise AssertionError('candidate dimension mismatch')
if not preparation_audit_round[
    ['preserves_52D_prefix', 'single_candidate_only']
].all().all():
    raise AssertionError('independent-block contract failed')
display(preparation_audit_round)


,representation_id,dimension,expected_dimension,train_rows,validation_rows,scaler_fit_rows,scaler_policy,preserves_52D_prefix,single_candidate_only
0,REF_52D,52,52,711467,177891,711467,internal_train_only,True,True
1,AF10_SPEECH_ACTIVITY_BOUNDARY,56,56,711467,177891,711467,internal_train_only,True,True
2,AF11_SPECTRAL_ENTROPY_DYNAMICS,55,55,711467,177891,711467,internal_train_only,True,True
3,AF12_TEMPORAL_MFCC_CONTRAST,55,55,711467,177891,711467,internal_train_only,True,True


In [27]:
# Reuse the verified 52D result; fit only three independent 52D-plus-one candidates.
if not ALLOW_RESIDUAL_ROUND_SCREENING:
    raise RuntimeError(
        'STOP: set only ALLOW_RESIDUAL_ROUND_SCREENING=True after approving '
        'the combined Training cache and preparation audit.'
    )
residual_round_results = {'REF_52D': residual_result_52d}
for block_id in RESIDUAL_TARGET_BLOCK_IDS:
    result = _run_prepared_representation(
        prepared_residual_round[block_id],
        experiment_id=block_id,
        allow_fit=ALLOW_RESIDUAL_ROUND_SCREENING,
    )
    if dict(result.model_config) != dict(residual_result_52d.model_config):
        raise RuntimeError(f'{block_id} model configuration differs')
    for audit_name in (
        'train_idx', 'validation_idx',
        'training_record_keys', 'validation_record_keys',
    ):
        if not np.array_equal(
            result.fit_audit[audit_name],
            residual_result_52d.fit_audit[audit_name],
        ):
            raise RuntimeError(f'{block_id} {audit_name} differs')
    residual_round_results[block_id] = result
print('Completed independent candidates:', tuple(residual_round_results))


Completed independent candidates: ('REF_52D', 'AF10_SPEECH_ACTIVITY_BOUNDARY', 'AF11_SPECTRAL_ENTROPY_DYNAMICS', 'AF12_TEMPORAL_MFCC_CONTRAST')


In [28]:
# Dimension, Accuracy/delta, CM, recall deltas, correct-count delta, and paired transitions.
if 'residual_round_results' not in globals():
    raise RuntimeError('Run the explicitly gated screening cell first')
truth_round = np.asarray(residual_round_results['REF_52D'].y_true, dtype=np.int64)
reference_prediction_round = np.asarray(
    residual_round_results['REF_52D'].y_pred, dtype=np.int64,
)
reference_correct_round = reference_prediction_round == truth_round
overall_rows_round = []
for representation_id, result in residual_round_results.items():
    prediction = np.asarray(result.y_pred, dtype=np.int64)
    correct = prediction == truth_round
    metrics = calculate_prediction_metrics(truth_round, prediction)
    reference_metrics = residual_round_results['REF_52D'].metrics
    overall_rows_round.append({
        'representation_id': representation_id,
        'dimension': len(result.feature_names),
        'accuracy': float(metrics['accuracy']),
        'accuracy_delta_vs_52D': float(metrics['accuracy'] - reference_metrics['accuracy']),
        'confusion_matrix': metrics['confusion_matrix'],
        'recall_speaker_0': float(metrics['recall_speaker_0']),
        'recall_0_delta_vs_52D': float(
            metrics['recall_speaker_0'] - reference_metrics['recall_speaker_0']
        ),
        'recall_speaker_1': float(metrics['recall_speaker_1']),
        'recall_1_delta_vs_52D': float(
            metrics['recall_speaker_1'] - reference_metrics['recall_speaker_1']
        ),
        'correct_count': int(metrics['correct_count']),
        'correct_count_delta_vs_52D': int(
            metrics['correct_count'] - reference_metrics['correct_count']
        ),
        '52D_wrong_to_correct': int(np.sum((~reference_correct_round) & correct)),
        '52D_correct_to_wrong': int(np.sum(reference_correct_round & (~correct))),
        'automatic_promotion': False,
    })
residual_round_overall_summary = pd.DataFrame(overall_rows_round)
display(residual_round_overall_summary)


,representation_id,dimension,accuracy,accuracy_delta_vs_52D,confusion_matrix,recall_speaker_0,recall_0_delta_vs_52D,recall_speaker_1,recall_1_delta_vs_52D,correct_count,correct_count_delta_vs_52D,52D_wrong_to_correct,52D_correct_to_wrong,automatic_promotion
0,REF_52D,52,0.869532,0.000000,"[[71066, 14285], [8924, 83616]]",0.832632,0.000000,0.903566,0.000000,154682,0,0,0,False
1,AF10_SPEECH_ACTIVITY_BOUNDARY,56,0.873074,0.003541,"[[70998, 14353], [8226, 84314]]",0.831836,-0.000797,0.911109,0.007543,155312,630,4435,3805,False
2,AF11_SPECTRAL_ENTROPY_DYNAMICS,55,0.870747,0.001214,"[[71344, 14007], [8986, 83554]]",0.835889,0.003257,0.902896,-0.000670,154898,216,3938,3722,False
3,AF12_TEMPORAL_MFCC_CONTRAST,55,0.870111,0.000579,"[[71517, 13834], [9272, 83268]]",0.837916,0.005284,0.899805,-0.003761,154785,103,3965,3862,False


In [29]:
# Duration, overlap, speaker × duration × overlap, focus slices, and high-confidence FP/FN.
validation_keys_round = residual_error_ledger['record_key'].astype(str).to_numpy()
if not np.array_equal(
    validation_keys_round,
    np.asarray(
        residual_round_results['REF_52D'].fit_audit['validation_record_keys']
    ).astype(str),
):
    raise RuntimeError('STOP: diagnostic ledger order differs from 52D')
duration_round = residual_error_ledger['duration_slice'].astype(str).to_numpy()
overlap_round = residual_error_ledger['overlap'].to_numpy(dtype=bool)

def _metric_row_round(representation_id, result, slice_type, slice_name, selected):
    selected = np.asarray(selected, dtype=bool)
    prediction = np.asarray(result.y_pred, dtype=np.int64)
    if np.any(selected):
        metrics = calculate_prediction_metrics(truth_round[selected], prediction[selected])
    else:
        metrics = {
            'sample_count': 0, 'accuracy': np.nan,
            'confusion_matrix': [[0, 0], [0, 0]],
            'recall_speaker_0': np.nan, 'recall_speaker_1': np.nan,
            'correct_count': 0,
        }
    return {
        'slice_type': slice_type, 'slice': slice_name,
        'representation_id': representation_id,
        'count': int(metrics['sample_count']),
        'accuracy': metrics['accuracy'],
        'confusion_matrix': metrics['confusion_matrix'],
        'recall_speaker_0': metrics['recall_speaker_0'],
        'recall_speaker_1': metrics['recall_speaker_1'],
        'correct_count': int(metrics['correct_count']),
    }

slice_specs_round = []
for duration_name in RESIDUAL_DURATION_ORDER:
    slice_specs_round.append(
        ('duration', duration_name, duration_round == duration_name)
    )
slice_specs_round.extend([
    ('overlap', 'overlap', overlap_round),
    ('overlap', 'no_overlap', ~overlap_round),
])
for speaker in (0, 1):
    for duration_name in RESIDUAL_DURATION_ORDER:
        for overlap_name, overlap_mask in (
            ('overlap', overlap_round), ('no_overlap', ~overlap_round),
        ):
            slice_specs_round.append((
                'speaker_x_duration_x_overlap',
                f'speaker={speaker} | duration={duration_name} | overlap={overlap_name}',
                (
                    (truth_round == speaker)
                    & (duration_round == duration_name)
                    & overlap_mask
                ),
            ))

slice_rows_round = []
for slice_type, slice_name, selected in slice_specs_round:
    reference_row = _metric_row_round(
        'REF_52D', residual_round_results['REF_52D'],
        slice_type, slice_name, selected,
    )
    for representation_id, result in residual_round_results.items():
        row = _metric_row_round(
            representation_id, result, slice_type, slice_name, selected,
        )
        row.update({
            'accuracy_delta_vs_52D': row['accuracy'] - reference_row['accuracy'],
            'recall_0_delta_vs_52D': (
                row['recall_speaker_0'] - reference_row['recall_speaker_0']
            ),
            'recall_1_delta_vs_52D': (
                row['recall_speaker_1'] - reference_row['recall_speaker_1']
            ),
            'correct_count_delta_vs_52D': (
                row['correct_count'] - reference_row['correct_count']
            ),
        })
        slice_rows_round.append(row)
residual_round_slice_summary = pd.DataFrame(slice_rows_round)

focus_specs_round = (
    (
        'speaker0_0.5-1.0s_overlap',
        (truth_round == 0) & (duration_round == '0.5-1.0s') & overlap_round,
    ),
    (
        'speaker0_1-2s_overlap',
        (truth_round == 0) & (duration_round == '1-2s') & overlap_round,
    ),
    (
        '0.5-1.0s_overlap_all_speakers',
        (duration_round == '0.5-1.0s') & overlap_round,
    ),
    ('overlap_all', overlap_round),
)
focus_rows_round = []
for focus_name, selected in focus_specs_round:
    reference_row = _metric_row_round(
        'REF_52D', residual_round_results['REF_52D'],
        'focus', focus_name, selected,
    )
    for representation_id, result in residual_round_results.items():
        row = _metric_row_round(
            representation_id, result, 'focus', focus_name, selected,
        )
        row.update({
            'accuracy_delta_vs_52D': row['accuracy'] - reference_row['accuracy'],
            'recall_0_delta_vs_52D': (
                row['recall_speaker_0'] - reference_row['recall_speaker_0']
            ),
            'recall_1_delta_vs_52D': (
                row['recall_speaker_1'] - reference_row['recall_speaker_1']
            ),
            'correct_count_delta_vs_52D': (
                row['correct_count'] - reference_row['correct_count']
            ),
        })
        focus_rows_round.append(row)
residual_round_focus_summary = pd.DataFrame(focus_rows_round)

high_confidence_rows_round = []
for representation_id, result in residual_round_results.items():
    prediction = np.asarray(result.y_pred, dtype=np.int64)
    confidence = np.max(np.asarray(result.probabilities, dtype=np.float64), axis=1)
    for error_type, selected in (
        ('FP', (confidence >= 0.90) & (truth_round == 0) & (prediction == 1)),
        ('FN', (confidence >= 0.90) & (truth_round == 1) & (prediction == 0)),
    ):
        high_confidence_rows_round.append({
            'representation_id': representation_id,
            'confidence_threshold': 0.90,
            'error_type': error_type,
            'count': int(np.sum(selected)),
        })
residual_round_high_confidence_errors = pd.DataFrame(high_confidence_rows_round)
reference_high_confidence_round = (
    residual_round_high_confidence_errors.loc[
        residual_round_high_confidence_errors['representation_id'].eq('REF_52D'),
        ['error_type', 'count'],
    ].set_index('error_type')['count']
)
residual_round_high_confidence_errors['count_delta_vs_52D'] = (
    residual_round_high_confidence_errors.apply(
        lambda row: int(row['count'])
        - int(reference_high_confidence_round.loc[row['error_type']]),
        axis=1,
    )
)

print('Duration')
display(residual_round_slice_summary.loc[
    residual_round_slice_summary['slice_type'].eq('duration')
])
print('Overlap')
display(residual_round_slice_summary.loc[
    residual_round_slice_summary['slice_type'].eq('overlap')
])
print('Speaker × duration × overlap')
display(residual_round_slice_summary.loc[
    residual_round_slice_summary['slice_type'].eq('speaker_x_duration_x_overlap')
])
print('Required focus slices')
display(residual_round_focus_summary)
print('High-confidence FP/FN')
display(residual_round_high_confidence_errors)
print(
    'STOP: no combination or automatic promotion; '
    'the user owns interpretation and the next experiment decision.'
)


Duration


,slice_type,slice,representation_id,count,accuracy,confusion_matrix,recall_speaker_0,recall_speaker_1,correct_count,accuracy_delta_vs_52D,recall_0_delta_vs_52D,recall_1_delta_vs_52D,correct_count_delta_vs_52D
0,duration,<0.5s,REF_52D,14076,0.863527,"[[5963, 765], [1156, 6192]]",0.886296,0.842678,12155,0.000000,0.000000,0.000000,0
1,duration,<0.5s,AF10_SPEECH_ACTIVITY_BOUNDARY,14076,0.872620,"[[5929, 799], [994, 6354]]",0.881243,0.864725,12283,0.009093,-0.005054,0.022047,128
2,duration,<0.5s,AF11_SPECTRAL_ENTROPY_DYNAMICS,14076,0.866226,"[[6014, 714], [1169, 6179]]",0.893876,0.840909,12193,0.002700,0.007580,-0.001769,38
3,duration,<0.5s,AF12_TEMPORAL_MFCC_CONTRAST,14076,0.866368,"[[6032, 696], [1185, 6163]]",0.896552,0.838732,12195,0.002842,0.010256,-0.003947,40
4,duration,0.5-1.0s,REF_52D,43199,0.825598,"[[14958, 4635], [2899, 20707]]",0.763436,0.877192,35665,0.000000,0.000000,0.000000,0
5,duration,0.5-1.0s,AF10_SPEECH_ACTIVITY_BOUNDARY,43199,0.828190,"[[15068, 4525], [2897, 20709]]",0.769050,0.877277,35777,0.002593,0.005614,0.000085,112
6,duration,0.5-1.0s,AF11_SPECTRAL_ENTROPY_DYNAMICS,43199,0.824533,"[[15231, 4362], [3218, 20388]]",0.777369,0.863679,35619,-0.001065,0.013934,-0.013514,-46
7,duration,0.5-1.0s,AF12_TEMPORAL_MFCC_CONTRAST,43199,0.824695,"[[15436, 4157], [3416, 20190]]",0.787832,0.855291,35626,-0.000903,0.024396,-0.021901,-39
8,duration,1-2s,REF_52D,58320,0.847154,"[[25813, 6219], [2695, 23593]]",0.805850,0.897482,49406,0.000000,0.000000,0.000000,0
9,duration,1-2s,AF10_SPEECH_ACTIVITY_BOUNDARY,58320,0.849777,"[[25760, 6272], [2489, 23799]]",0.804196,0.905318,49559,0.002623,-0.001655,0.007836,153


Overlap


,slice_type,slice,representation_id,count,accuracy,confusion_matrix,recall_speaker_0,recall_speaker_1,correct_count,accuracy_delta_vs_52D,recall_0_delta_vs_52D,recall_1_delta_vs_52D,correct_count_delta_vs_52D
20,overlap,overlap,REF_52D,78581,0.827846,"[[30704, 7721], [5807, 34349]]",0.799063,0.855389,65053,0.000000,0.000000,0.000000,0
21,overlap,overlap,AF10_SPEECH_ACTIVITY_BOUNDARY,78581,0.833102,"[[30660, 7765], [5350, 34806]]",0.797918,0.866770,65466,0.005256,-0.001145,0.011381,413
22,overlap,overlap,AF11_SPECTRAL_ENTROPY_DYNAMICS,78581,0.830290,"[[30873, 7552], [5784, 34372]]",0.803461,0.855962,65245,0.002443,0.004398,0.000573,192
23,overlap,overlap,AF12_TEMPORAL_MFCC_CONTRAST,78581,0.830201,"[[30880, 7545], [5798, 34358]]",0.803643,0.855613,65238,0.002354,0.004580,0.000224,185
24,overlap,no_overlap,REF_52D,99310,0.902517,"[[40362, 6564], [3117, 49267]]",0.860120,0.940497,89629,0.000000,0.000000,0.000000,0
25,overlap,no_overlap,AF10_SPEECH_ACTIVITY_BOUNDARY,99310,0.904702,"[[40338, 6588], [2876, 49508]]",0.859609,0.945098,89846,0.002185,-0.000511,0.004601,217
26,overlap,no_overlap,AF11_SPECTRAL_ENTROPY_DYNAMICS,99310,0.902759,"[[40471, 6455], [3202, 49182]]",0.862443,0.938874,89653,0.000242,0.002323,-0.001623,24
27,overlap,no_overlap,AF12_TEMPORAL_MFCC_CONTRAST,99310,0.901692,"[[40637, 6289], [3474, 48910]]",0.865980,0.933682,89547,-0.000826,0.005860,-0.006815,-82


Speaker × duration × overlap


...

Required focus slices


,slice_type,slice,representation_id,count,accuracy,confusion_matrix,recall_speaker_0,recall_speaker_1,correct_count,accuracy_delta_vs_52D,recall_0_delta_vs_52D,recall_1_delta_vs_52D,correct_count_delta_vs_52D
0,focus,speaker0_0.5-1.0s_overlap,REF_52D,7976,0.675652,"[[5389, 2587], [0, 0]]",0.675652,0.000000,5389,0.000000,0.000000,0.000000,0
1,focus,speaker0_0.5-1.0s_overlap,AF10_SPEECH_ACTIVITY_BOUNDARY,7976,0.684428,"[[5459, 2517], [0, 0]]",0.684428,0.000000,5459,0.008776,0.008776,0.000000,70
2,focus,speaker0_0.5-1.0s_overlap,AF11_SPECTRAL_ENTROPY_DYNAMICS,7976,0.691700,"[[5517, 2459], [0, 0]]",0.691700,0.000000,5517,0.016048,0.016048,0.000000,128
3,focus,speaker0_0.5-1.0s_overlap,AF12_TEMPORAL_MFCC_CONTRAST,7976,0.706871,"[[5638, 2338], [0, 0]]",0.706871,0.000000,5638,0.031219,0.031219,0.000000,249
4,focus,speaker0_1-2s_overlap,REF_52D,13517,0.767330,"[[10372, 3145], [0, 0]]",0.767330,0.000000,10372,0.000000,0.000000,0.000000,0
5,focus,speaker0_1-2s_overlap,AF10_SPEECH_ACTIVITY_BOUNDARY,13517,0.762595,"[[10308, 3209], [0, 0]]",0.762595,0.000000,10308,-0.004735,-0.004735,0.000000,-64
6,focus,speaker0_1-2s_overlap,AF11_SPECTRAL_ENTROPY_DYNAMICS,13517,0.769032,"[[10395, 3122], [0, 0]]",0.769032,0.000000,10395,0.001702,0.001702,0.000000,23
7,focus,speaker0_1-2s_overlap,AF12_TEMPORAL_MFCC_CONTRAST,13517,0.766072,"[[10355, 3162], [0, 0]]",0.766072,0.000000,10355,-0.001258,-0.001258,0.000000,-17
8,focus,0.5-1.0s_overlap_all_speakers,REF_52D,17847,0.753684,"[[5389, 2587], [1809, 8062]]",0.675652,0.816736,13451,0.000000,0.000000,0.000000,0
9,focus,0.5-1.0s_overlap_all_speakers,AF10_SPEECH_ACTIVITY_BOUNDARY,17847,0.759736,"[[5459, 2517], [1771, 8100]]",0.684428,0.820586,13559,0.006051,0.008776,0.003850,108


High-confidence FP/FN


,representation_id,confidence_threshold,error_type,count,count_delta_vs_52D
0,REF_52D,0.9,FP,1223,0
1,REF_52D,0.9,FN,1367,0
2,AF10_SPEECH_ACTIVITY_BOUNDARY,0.9,FP,1354,131
3,AF10_SPEECH_ACTIVITY_BOUNDARY,0.9,FN,1132,-235
4,AF11_SPECTRAL_ENTROPY_DYNAMICS,0.9,FP,1294,71
5,AF11_SPECTRAL_ENTROPY_DYNAMICS,0.9,FN,1376,9
6,AF12_TEMPORAL_MFCC_CONTRAST,0.9,FP,1152,-71
7,AF12_TEMPORAL_MFCC_CONTRAST,0.9,FN,1281,-86


STOP: no combination or automatic promotion; the user owns interpretation and the next experiment decision.


| 항목                           |           AF10 |          AF11 |          AF12 |
| ---------------------------- | -------------: | ------------: | ------------: |
| 전체 Accuracy Δ                |  **+0.3541%p** |     +0.1214%p |     +0.0579%p |
| 추가 정답                        |       **+630** |          +216 |          +103 |
| Speaker 0 Recall Δ           |      -0.0797%p |     +0.3257%p | **+0.5284%p** |
| Speaker 1 Recall Δ           |  **+0.7543%p** |     -0.0670%p |     -0.3761%p |
| Balanced Accuracy Δ          | **약 +0.337%p** |      +0.129%p |      +0.076%p |
| Overlap 전체 Δ                 |  **+0.5256%p** |     +0.2443%p |     +0.2354%p |
| No-overlap Δ                 |  **+0.2185%p** |     +0.0242%p | **-0.0826%p** |
| 핵심 `spk0 × 0.5–1s × overlap` |      +0.8776%p |     +1.6048%p | **+3.1219%p** |
| `spk0 × 1–2s × overlap`      |  **-0.4735%p** | **+0.1702%p** |     -0.1258%p |
| High-conf error 총 변화         |           -104 |       **+80** |      **-157** |


## Bounded combinations — AF10/AF11/AF12

Use the verified 52D reference and the existing immutable combined cache. Fit exactly four approved combinations; preserve the earlier single-block cells and results.

In [20]:
# 1. Validate canonical 52D and prepare exactly four residual combinations.
from dcc_data.mission2_acoustic_feature_expansion import (
    RESIDUAL_COMBINATION_SPECS,
    prepare_residual_combination_screening,
)

ALLOW_RESIDUAL_COMBINATION_SCREENING = False
RESIDUAL_COMBINATION_IDS = (
    'AF10_AF11_COMBINATION',
    'AF10_AF12_COMBINATION',
    'AF11_AF12_COMBINATION',
    'AF10_AF11_AF12_COMBINATION',
)
if tuple(RESIDUAL_COMBINATION_SPECS) != RESIDUAL_COMBINATION_IDS:
    raise RuntimeError('STOP: residual combination contract must contain exactly four entries')
required_combination_inputs = (
    'residual_result_52d', 'residual_prepared_52d',
    'residual_round_feature_frame', 'X_52d_round',
    'round_keys', 'round_calls', 'residual_error_ledger',
)
missing_combination_inputs = [
    name for name in required_combination_inputs if name not in globals()
]
if missing_combination_inputs:
    raise RuntimeError(
        'Run the completed AF10/AF11/AF12 single-experiment preparation and diagnostics first. '
        f'Missing: {missing_combination_inputs}'
    )
combination_train_idx, combination_validation_idx, combination_split_audit = (
    create_fixed_internal_split(
        round_calls,
        split_function=standalone_group_split_indices,
        expected_contract=EXPECTED_INTERNAL_SPLIT,
    )
)
combination_reference_resolution = resolve_canonical_52d_reference(
    residual_prepared_52d,
    existing_result=residual_result_52d,
    canonical_matrix=X_52d_round,
    canonical_labels=np.asarray(e05_cache.validated.y, dtype=np.int64),
    canonical_record_keys=round_keys,
    canonical_train_idx=combination_train_idx,
    canonical_validation_idx=combination_validation_idx,
    expected_split_contract=EXPECTED_INTERNAL_SPLIT,
    allow_refit=False,
)
combination_reference_result = combination_reference_resolution.result
prepared_residual_combinations = prepare_residual_combination_screening(
    prepared_52d=residual_prepared_52d,
    X_52d=X_52d_round,
    y=np.asarray(e05_cache.validated.y, dtype=np.int64),
    record_keys=round_keys,
    call_ids=round_calls,
    candidate_frame=residual_round_feature_frame,
    canonical_train_idx=combination_train_idx,
    canonical_validation_idx=combination_validation_idx,
    expected_split_contract=EXPECTED_INTERNAL_SPLIT,
)
expected_combination_dimensions = {
    'AF10_AF11_COMBINATION': 59,
    'AF10_AF12_COMBINATION': 59,
    'AF11_AF12_COMBINATION': 58,
    'AF10_AF11_AF12_COMBINATION': 62,
}
combination_preparation_audit = pd.DataFrame([
    {
        'representation_id': combination_id,
        'dimension': len(prepared.feature_names),
        'expected_dimension': expected_combination_dimensions[combination_id],
        'train_rows': len(prepared.train_idx),
        'validation_rows': len(prepared.validation_idx),
        'scaler_fit_rows': int(prepared.scaler.n_samples_seen_),
        'scaler_policy': 'internal_train_only',
        'preserves_52D_prefix': tuple(prepared.feature_names[:52]) == tuple(residual_prepared_52d.feature_names),
        'same_split': (
            np.array_equal(prepared.train_idx, residual_prepared_52d.train_idx)
            and np.array_equal(
                prepared.validation_idx, residual_prepared_52d.validation_idx
            )
        ),
    }
    for combination_id, prepared in prepared_residual_combinations.items()
])
if tuple(prepared_residual_combinations) != RESIDUAL_COMBINATION_IDS:
    raise AssertionError('prepared combination set/order changed')
if not (
    combination_preparation_audit['dimension']
    == combination_preparation_audit['expected_dimension']
).all():
    raise AssertionError('combination dimension mismatch')
if not combination_preparation_audit[['preserves_52D_prefix', 'same_split']].all().all():
    raise AssertionError('combination prefix/split contract failed')
display(combination_preparation_audit)
print('Combination screening gate:', ALLOW_RESIDUAL_COMBINATION_SCREENING)


,representation_id,dimension,expected_dimension,train_rows,validation_rows,scaler_fit_rows,scaler_policy,preserves_52D_prefix,same_split
0,AF10_AF11_COMBINATION,59,59,711467,177891,711467,internal_train_only,True,True
1,AF10_AF12_COMBINATION,59,59,711467,177891,711467,internal_train_only,True,True
2,AF11_AF12_COMBINATION,58,58,711467,177891,711467,internal_train_only,True,True
3,AF10_AF11_AF12_COMBINATION,62,62,711467,177891,711467,internal_train_only,True,True


Combination screening gate: True


In [21]:
# 2. Fit only the four approved combinations under the fixed M04-100 contract.
if not ALLOW_RESIDUAL_COMBINATION_SCREENING:
    raise RuntimeError(
        'STOP: set only ALLOW_RESIDUAL_COMBINATION_SCREENING=True after reviewing '
        'the four-row preparation audit.'
    )
residual_combination_results = {'REF_52D': combination_reference_result}
for combination_id in RESIDUAL_COMBINATION_IDS:
    result = _run_prepared_representation(
        prepared_residual_combinations[combination_id],
        experiment_id=combination_id,
        allow_fit=ALLOW_RESIDUAL_COMBINATION_SCREENING,
    )
    if dict(result.model_config) != dict(combination_reference_result.model_config):
        raise RuntimeError(f'{combination_id} model configuration differs')
    if result.model_config['max_iter'] != 100 or result.fit_audit['seed'] != 42:
        raise RuntimeError(f'{combination_id} max_iter/seed contract differs')
    if result.fit_audit['scaler_fit_scope'] != 'internal_train_only':
        raise RuntimeError(f'{combination_id} scaler scope differs')
    for audit_name in (
        'train_idx', 'validation_idx',
        'training_record_keys', 'validation_record_keys',
    ):
        if not np.array_equal(
            result.fit_audit[audit_name],
            combination_reference_result.fit_audit[audit_name],
        ):
            raise RuntimeError(f'{combination_id} {audit_name} differs')
    residual_combination_results[combination_id] = result
print('Completed bounded combinations:', tuple(residual_combination_results))


Completed bounded combinations: ('REF_52D', 'AF10_AF11_COMBINATION', 'AF10_AF12_COMBINATION', 'AF11_AF12_COMBINATION', 'AF10_AF11_AF12_COMBINATION')


In [22]:
# 3. Compare overall combination metrics against the canonical 52D reference.
if 'residual_combination_results' not in globals():
    raise RuntimeError('Run the explicitly gated combination fitting cell first')
combination_truth = np.asarray(
    residual_combination_results['REF_52D'].y_true, dtype=np.int64,
)
combination_reference_prediction = np.asarray(
    residual_combination_results['REF_52D'].y_pred, dtype=np.int64,
)
combination_reference_correct = combination_reference_prediction == combination_truth
combination_reference_metrics = calculate_prediction_metrics(
    combination_truth, combination_reference_prediction,
)
combination_overall_rows = []
for representation_id, result in residual_combination_results.items():
    prediction = np.asarray(result.y_pred, dtype=np.int64)
    correct = prediction == combination_truth
    metrics = calculate_prediction_metrics(combination_truth, prediction)
    combination_overall_rows.append({
        'representation_id': representation_id,
        'dimension': len(result.feature_names),
        'accuracy': float(metrics['accuracy']),
        'accuracy_delta_vs_52D': float(
            metrics['accuracy'] - combination_reference_metrics['accuracy']
        ),
        'confusion_matrix': metrics['confusion_matrix'],
        'recall_speaker_0': float(metrics['recall_speaker_0']),
        'recall_0_delta_vs_52D': float(
            metrics['recall_speaker_0']
            - combination_reference_metrics['recall_speaker_0']
        ),
        'recall_speaker_1': float(metrics['recall_speaker_1']),
        'recall_1_delta_vs_52D': float(
            metrics['recall_speaker_1']
            - combination_reference_metrics['recall_speaker_1']
        ),
        'correct_count': int(metrics['correct_count']),
        'correct_count_delta_vs_52D': int(
            metrics['correct_count'] - combination_reference_metrics['correct_count']
        ),
        '52D_wrong_to_correct': int(np.sum((~combination_reference_correct) & correct)),
        '52D_correct_to_wrong': int(np.sum(combination_reference_correct & (~correct))),
        'automatic_promotion': False,
    })
residual_combination_overall_summary = pd.DataFrame(combination_overall_rows)
display(residual_combination_overall_summary)


,representation_id,dimension,accuracy,accuracy_delta_vs_52D,confusion_matrix,recall_speaker_0,recall_0_delta_vs_52D,recall_speaker_1,recall_1_delta_vs_52D,correct_count,correct_count_delta_vs_52D,52D_wrong_to_correct,52D_correct_to_wrong,automatic_promotion
0,REF_52D,52,0.869532,0.000000,"[[71066, 14285], [8924, 83616]]",0.832632,0.000000,0.903566,0.000000,154682,0,0,0,False
1,AF10_AF11_COMBINATION,59,0.874558,0.005026,"[[71421, 13930], [8385, 84155]]",0.836792,0.004159,0.909391,0.005825,155576,894,4776,3882,False
2,AF10_AF12_COMBINATION,59,0.872574,0.003041,"[[71337, 14014], [8654, 83886]]",0.835807,0.003175,0.906484,0.002918,155223,541,4809,4268,False
3,AF11_AF12_COMBINATION,58,0.872107,0.002575,"[[70528, 14823], [7928, 84612]]",0.826329,-0.006303,0.914329,0.010763,155140,458,4234,3776,False
4,AF10_AF11_AF12_COMBINATION,62,0.876042,0.006510,"[[70783, 14568], [7483, 85057]]",0.829317,-0.003316,0.919138,0.015572,155840,1158,4989,3831,False


In [23]:
# 4. Compare duration, overlap, focus slices, and high-confidence FP/FN.
combination_validation_keys = residual_error_ledger['record_key'].astype(str).to_numpy()
combination_reference_keys = np.asarray(
    residual_combination_results['REF_52D'].fit_audit['validation_record_keys']
).astype(str)
if not np.array_equal(combination_validation_keys, combination_reference_keys):
    raise RuntimeError('STOP: combination diagnostic record_key order differs')
if not np.array_equal(
    residual_error_ledger['y_true'].to_numpy(dtype=np.int64), combination_truth,
):
    raise RuntimeError('STOP: combination diagnostic labels differ')
combination_duration = residual_error_ledger['duration_slice'].astype(str).to_numpy()
combination_overlap = residual_error_ledger['overlap'].to_numpy(dtype=bool)

def _combination_metric_row(representation_id, result, slice_type, slice_name, selected):
    selected = np.asarray(selected, dtype=bool)
    prediction = np.asarray(result.y_pred, dtype=np.int64)
    metrics = calculate_prediction_metrics(
        combination_truth[selected], prediction[selected],
    )
    return {
        'slice_type': slice_type, 'slice': slice_name,
        'representation_id': representation_id,
        'count': int(metrics['sample_count']),
        'accuracy': float(metrics['accuracy']),
        'confusion_matrix': metrics['confusion_matrix'],
        'recall_speaker_0': float(metrics['recall_speaker_0']),
        'recall_speaker_1': float(metrics['recall_speaker_1']),
        'correct_count': int(metrics['correct_count']),
    }

combination_slice_specs = [
    ('duration', duration_name, combination_duration == duration_name)
    for duration_name in RESIDUAL_DURATION_ORDER
]
combination_slice_specs.extend([
    ('overlap', 'overlap', combination_overlap),
    ('overlap', 'no_overlap', ~combination_overlap),
])
combination_focus_specs = (
    (
        'speaker0_0.5-1.0s_overlap',
        (combination_truth == 0)
        & (combination_duration == '0.5-1.0s')
        & combination_overlap,
    ),
    (
        'speaker0_1-2s_overlap',
        (combination_truth == 0)
        & (combination_duration == '1-2s')
        & combination_overlap,
    ),
)

combination_slice_rows = []
for slice_type, slice_name, selected in combination_slice_specs:
    reference_row = _combination_metric_row(
        'REF_52D', residual_combination_results['REF_52D'],
        slice_type, slice_name, selected,
    )
    for representation_id, result in residual_combination_results.items():
        row = _combination_metric_row(
            representation_id, result, slice_type, slice_name, selected,
        )
        row.update({
            'accuracy_delta_vs_52D': row['accuracy'] - reference_row['accuracy'],
            'recall_0_delta_vs_52D': (
                row['recall_speaker_0'] - reference_row['recall_speaker_0']
            ),
            'recall_1_delta_vs_52D': (
                row['recall_speaker_1'] - reference_row['recall_speaker_1']
            ),
            'correct_count_delta_vs_52D': (
                row['correct_count'] - reference_row['correct_count']
            ),
        })
        combination_slice_rows.append(row)
residual_combination_slice_summary = pd.DataFrame(combination_slice_rows)

combination_focus_rows = []
for focus_name, selected in combination_focus_specs:
    reference_row = _combination_metric_row(
        'REF_52D', residual_combination_results['REF_52D'],
        'focus', focus_name, selected,
    )
    for representation_id, result in residual_combination_results.items():
        row = _combination_metric_row(
            representation_id, result, 'focus', focus_name, selected,
        )
        row.update({
            'accuracy_delta_vs_52D': row['accuracy'] - reference_row['accuracy'],
            'recall_0_delta_vs_52D': (
                row['recall_speaker_0'] - reference_row['recall_speaker_0']
            ),
            'recall_1_delta_vs_52D': (
                row['recall_speaker_1'] - reference_row['recall_speaker_1']
            ),
            'correct_count_delta_vs_52D': (
                row['correct_count'] - reference_row['correct_count']
            ),
        })
        combination_focus_rows.append(row)
residual_combination_focus_summary = pd.DataFrame(combination_focus_rows)

combination_high_confidence_rows = []
for representation_id, result in residual_combination_results.items():
    prediction = np.asarray(result.y_pred, dtype=np.int64)
    confidence = np.max(np.asarray(result.probabilities, dtype=np.float64), axis=1)
    for error_type, selected in (
        ('FP', (confidence >= 0.90) & (combination_truth == 0) & (prediction == 1)),
        ('FN', (confidence >= 0.90) & (combination_truth == 1) & (prediction == 0)),
    ):
        combination_high_confidence_rows.append({
            'representation_id': representation_id,
            'confidence_threshold': 0.90,
            'error_type': error_type,
            'count': int(np.sum(selected)),
        })
residual_combination_high_confidence_errors = pd.DataFrame(
    combination_high_confidence_rows
)
combination_reference_high_confidence = (
    residual_combination_high_confidence_errors.loc[
        residual_combination_high_confidence_errors['representation_id'].eq('REF_52D'),
        ['error_type', 'count'],
    ].set_index('error_type')['count']
)
residual_combination_high_confidence_errors['count_delta_vs_52D'] = (
    residual_combination_high_confidence_errors.apply(
        lambda row: int(row['count'])
        - int(combination_reference_high_confidence.loc[row['error_type']]),
        axis=1,
    )
)

print('Duration')
display(residual_combination_slice_summary.loc[
    residual_combination_slice_summary['slice_type'].eq('duration')
])
print('Overlap')
display(residual_combination_slice_summary.loc[
    residual_combination_slice_summary['slice_type'].eq('overlap')
])
print('Required speaker-0 focus slices')
display(residual_combination_focus_summary)
print('High-confidence FP/FN')
display(residual_combination_high_confidence_errors)
print('STOP: automatic promotion is disabled; user interpretation is required.')


Duration


,slice_type,slice,representation_id,count,accuracy,confusion_matrix,recall_speaker_0,recall_speaker_1,correct_count,accuracy_delta_vs_52D,recall_0_delta_vs_52D,recall_1_delta_vs_52D,correct_count_delta_vs_52D
0,duration,<0.5s,REF_52D,14076,0.863527,"[[5963, 765], [1156, 6192]]",0.886296,0.842678,12155,0.000000,0.000000,0.000000,0
1,duration,<0.5s,AF10_AF11_COMBINATION,14076,0.869778,"[[5905, 823], [1010, 6338]]",0.877675,0.862548,12243,0.006252,-0.008621,0.019869,88
2,duration,<0.5s,AF10_AF12_COMBINATION,14076,0.869778,"[[5904, 824], [1009, 6339]]",0.877527,0.862684,12243,0.006252,-0.008769,0.020005,88
3,duration,<0.5s,AF11_AF12_COMBINATION,14076,0.867576,"[[5989, 739], [1125, 6223]]",0.890161,0.846897,12212,0.004049,0.003864,0.004219,57
4,duration,<0.5s,AF10_AF11_AF12_COMBINATION,14076,0.873046,"[[5898, 830], [957, 6391]]",0.876635,0.869760,12289,0.009520,-0.009661,0.027082,134
5,duration,0.5-1.0s,REF_52D,43199,0.825598,"[[14958, 4635], [2899, 20707]]",0.763436,0.877192,35665,0.000000,0.000000,0.000000,0
6,duration,0.5-1.0s,AF10_AF11_COMBINATION,43199,0.829626,"[[15140, 4453], [2907, 20699]]",0.772725,0.876853,35839,0.004028,0.009289,-0.000339,174
7,duration,0.5-1.0s,AF10_AF12_COMBINATION,43199,0.825158,"[[15254, 4339], [3214, 20392]]",0.778543,0.863848,35646,-0.000440,0.015107,-0.013344,-19
8,duration,0.5-1.0s,AF11_AF12_COMBINATION,43199,0.828931,"[[14709, 4884], [2506, 21100]]",0.750727,0.893841,35809,0.003333,-0.012709,0.016648,144
9,duration,0.5-1.0s,AF10_AF11_AF12_COMBINATION,43199,0.832380,"[[14827, 4766], [2475, 21131]]",0.756750,0.895154,35958,0.006783,-0.006686,0.017962,293


Overlap


,slice_type,slice,representation_id,count,accuracy,confusion_matrix,recall_speaker_0,recall_speaker_1,correct_count,accuracy_delta_vs_52D,recall_0_delta_vs_52D,recall_1_delta_vs_52D,correct_count_delta_vs_52D
25,overlap,overlap,REF_52D,78581,0.827846,"[[30704, 7721], [5807, 34349]]",0.799063,0.855389,65053,0.000000,0.000000,0.000000,0
26,overlap,overlap,AF10_AF11_COMBINATION,78581,0.834502,"[[30952, 7473], [5532, 34624]]",0.805517,0.862237,65576,0.006656,0.006454,0.006848,523
27,overlap,overlap,AF10_AF12_COMBINATION,78581,0.832886,"[[30885, 7540], [5592, 34564]]",0.803774,0.860743,65449,0.005039,0.004710,0.005354,396
28,overlap,overlap,AF11_AF12_COMBINATION,78581,0.831461,"[[30366, 8059], [5185, 34971]]",0.790267,0.870879,65337,0.003614,-0.008796,0.015490,284
29,overlap,overlap,AF10_AF11_AF12_COMBINATION,78581,0.837722,"[[30558, 7867], [4885, 35271]]",0.795264,0.878349,65829,0.009875,-0.003800,0.022960,776
30,overlap,no_overlap,REF_52D,99310,0.902517,"[[40362, 6564], [3117, 49267]]",0.860120,0.940497,89629,0.000000,0.000000,0.000000,0
31,overlap,no_overlap,AF10_AF11_COMBINATION,99310,0.906253,"[[40469, 6457], [2853, 49531]]",0.862400,0.945537,90000,0.003736,0.002280,0.005040,371
32,overlap,no_overlap,AF10_AF12_COMBINATION,99310,0.903977,"[[40452, 6474], [3062, 49322]]",0.862038,0.941547,89774,0.001460,0.001918,0.001050,145
33,overlap,no_overlap,AF11_AF12_COMBINATION,99310,0.904269,"[[40162, 6764], [2743, 49641]]",0.855858,0.947637,89803,0.001752,-0.004262,0.007140,174
34,overlap,no_overlap,AF10_AF11_AF12_COMBINATION,99310,0.906364,"[[40225, 6701], [2598, 49786]]",0.857201,0.950405,90011,0.003847,-0.002919,0.009908,382


Required speaker-0 focus slices


,slice_type,slice,representation_id,count,accuracy,confusion_matrix,recall_speaker_0,recall_speaker_1,correct_count,accuracy_delta_vs_52D,recall_0_delta_vs_52D,recall_1_delta_vs_52D,correct_count_delta_vs_52D
0,focus,speaker0_0.5-1.0s_overlap,REF_52D,7976,0.675652,"[[5389, 2587], [0, 0]]",0.675652,0.0,5389,0.000000,0.000000,0.0,0
1,focus,speaker0_0.5-1.0s_overlap,AF10_AF11_COMBINATION,7976,0.690948,"[[5511, 2465], [0, 0]]",0.690948,0.0,5511,0.015296,0.015296,0.0,122
2,focus,speaker0_0.5-1.0s_overlap,AF10_AF12_COMBINATION,7976,0.696841,"[[5558, 2418], [0, 0]]",0.696841,0.0,5558,0.021189,0.021189,0.0,169
3,focus,speaker0_0.5-1.0s_overlap,AF11_AF12_COMBINATION,7976,0.658475,"[[5252, 2724], [0, 0]]",0.658475,0.0,5252,-0.017177,-0.017177,0.0,-137
4,focus,speaker0_0.5-1.0s_overlap,AF10_AF11_AF12_COMBINATION,7976,0.669509,"[[5340, 2636], [0, 0]]",0.669509,0.0,5340,-0.006143,-0.006143,0.0,-49
5,focus,speaker0_1-2s_overlap,REF_52D,13517,0.767330,"[[10372, 3145], [0, 0]]",0.767330,0.0,10372,0.000000,0.000000,0.0,0
6,focus,speaker0_1-2s_overlap,AF10_AF11_COMBINATION,13517,0.774654,"[[10471, 3046], [0, 0]]",0.774654,0.0,10471,0.007324,0.007324,0.0,99
7,focus,speaker0_1-2s_overlap,AF10_AF12_COMBINATION,13517,0.771547,"[[10429, 3088], [0, 0]]",0.771547,0.0,10429,0.004217,0.004217,0.0,57
8,focus,speaker0_1-2s_overlap,AF11_AF12_COMBINATION,13517,0.759118,"[[10261, 3256], [0, 0]]",0.759118,0.0,10261,-0.008212,-0.008212,0.0,-111
9,focus,speaker0_1-2s_overlap,AF10_AF11_AF12_COMBINATION,13517,0.769254,"[[10398, 3119], [0, 0]]",0.769254,0.0,10398,0.001924,0.001924,0.0,26


High-confidence FP/FN


,representation_id,confidence_threshold,error_type,count,count_delta_vs_52D
0,REF_52D,0.9,FP,1223,0
1,REF_52D,0.9,FN,1367,0
2,AF10_AF11_COMBINATION,0.9,FP,1205,-18
3,AF10_AF11_COMBINATION,0.9,FN,1172,-195
4,AF10_AF12_COMBINATION,0.9,FP,1207,-16
5,AF10_AF12_COMBINATION,0.9,FN,1132,-235
6,AF11_AF12_COMBINATION,0.9,FP,1228,5
7,AF11_AF12_COMBINATION,0.9,FN,1118,-249
8,AF10_AF11_AF12_COMBINATION,0.9,FP,1327,104
9,AF10_AF11_AF12_COMBINATION,0.9,FN,1059,-308


STOP: automatic promotion is disabled; user interpretation is required.


=== AF10 + AF11 59D 결과 해석 ===

[무엇을 바꿨나?]
기존 52D feature에
- AF10: Speech Activity / Boundary Dynamics 4D
- AF11: Spectral Entropy Dynamics 3D
를 추가해 총 59D로 만들었다.

[결과]
Accuracy
- 52D : 86.95%
- 59D : 87.46%
- 변화 : +0.50%p
- 정답 수 : +894개

Speaker 0 Recall
- 83.26% -> 83.68% (+0.42%p)

Speaker 1 Recall
- 90.36% -> 90.94% (+0.58%p)

Overlap 구간 Accuracy
- 82.78% -> 83.45% (+0.67%p)

가장 어려웠던 Speaker0 + 0.5~1초 + Overlap
- 67.57% -> 69.09% (+1.53%p)

High-confidence error
- 2590개 -> 2377개 (-213개)

[해석]
59D는 단순히 전체 Accuracy만 오른 것이 아니다.
Speaker 0과 Speaker 1이 모두 개선됐고,
어려웠던 overlap / 짧은 발화 구간도 같이 좋아졌다.

따라서 AF10+AF11 59D를
현재 handcrafted acoustic feature의 새 reference로 선택한다.

주의:
이 결과는 고정된 internal validation 결과이다.
일부 세부 slice의 class recall은 감소할 수 있으므로,
최종 모델이라고 확정하지 않고 이후 비교 실험의 기준으로 사용한다.

## Stage 0 — AF10+AF11 59D internal reference freeze

This freeze records the existing saved execution outputs from `afe-residual-combination-prepare` (In[20]) and `afe-residual-combination-summary` (In[22]). No refit, feature extraction, or Official Validation was run for this audit.

| Audit item | Frozen value | Status |
|---|---|---|
| Representation / feature order | frozen 52D prefix + AF10 4D + AF11 3D = 59D | PASS |
| AF10 suffix | `active_frame_fraction`, `longest_active_run_fraction`, `activity_transition_rate`, `edge_activity_difference` | PASS |
| AF11 suffix | `spectral_entropy_mean`, `spectral_entropy_std`, `spectral_entropy_iqr` | PASS |
| Row / key / label alignment | exact canonical `record_key` order, matching `call_id`, labels indexed by the same fixed split | PASS |
| Fixed call_id GroupSplit | train 711,467 rows / 23,360 calls; internal val 177,891 rows / 5,840 calls; call overlap 0 | PASS |
| Scaler | `StandardScaler`; fit scope = internal train only; fit rows = 711,467 | PASS |
| Model | existing M04 `MLPClassifier`; hidden=(64,), ReLU, Adam, alpha=0.0001, batch=1024, lr=0.001 constant, max_iter=100, shuffle=True, early_stopping=False, random_state=42 | PASS |
| Historical `y_pred` artifact | **NOT PERSISTED** | DOCUMENTED LIMITATION |
| Exact historical prediction comparison | **NOT AVAILABLE** | NOT A FREEZE FAILURE |

| Stored internal reference metric | Frozen value |
|---|---:|
| Accuracy | `0.8745580158636468` |
| Confusion matrix | `[[71421, 13930], [8385, 84155]]` |
| Recall speaker 0 | `0.8367916017` |
| Recall speaker 1 | `0.9093905338` |
| Correct / total | `155576 / 177891` |

The historical prediction array was not persisted, so exact array comparison cannot be performed. This limitation does not invalidate the metric provenance retained in the executed Notebook outputs; it only limits prediction-level reproducibility auditing. Future reference runs must persist the prediction artifact or a deterministic prediction fingerprint.

**STAGE 0 FREEZE: PASS**

In [1]:
from pathlib import Path
import gc
import hashlib
import io
import json
import os
import sys
import zipfile

import numpy as np
import pandas as pd


STAGE1_REPRESENTATION_ID = "AF10_AF11_59D_STAGE1"
STAGE1_ARTIFACT_RELATIVE_PATH = Path(
    "data/processed/mission2_acoustic_feature_expansion/"
    "stage1_af10_af11_59d_m04_iter100_internal_predictions_v1.npz"
)
STAGE0_SPLIT_CONTRACT = {
    "total_rows": 889_358,
    "train_rows": 711_467,
    "validation_rows": 177_891,
    "train_calls": 23_360,
    "validation_calls": 5_840,
    "call_overlap": 0,
}
STAGE0_M04_CONFIG = {
    "hidden_layer_sizes": (64,),
    "activation": "relu",
    "solver": "adam",
    "alpha": 0.0001,
    "batch_size": 1024,
    "learning_rate": "constant",
    "learning_rate_init": 0.001,
    "max_iter": 100,
    "shuffle": True,
    "random_state": 42,
    "tol": 0.0001,
    "early_stopping": False,
    "n_iter_no_change": 10,
}
STAGE0_VALIDATION_ACCURACY = 0.8745580158636468
STAGE0_VALIDATION_CM = [[71_421, 13_930], [8_385, 84_155]]
STAGE0_VALIDATION_RECALL_0 = 71_421 / (71_421 + 13_930)
STAGE0_VALIDATION_RECALL_1 = 84_155 / (8_385 + 84_155)
STAGE0_VALIDATION_CORRECT = 155_576


def _stop(message):
    raise RuntimeError(f"STOP: {message}")


def _deterministic_npz_bytes(arrays):
    """Serialize arrays with fixed ZIP metadata so identical content has one SHA-256."""

    archive_buffer = io.BytesIO()
    with zipfile.ZipFile(
        archive_buffer,
        mode="w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as archive:
        for name in sorted(arrays):
            array_buffer = io.BytesIO()
            np.lib.format.write_array(
                array_buffer,
                np.ascontiguousarray(arrays[name]),
                allow_pickle=False,
            )
            member = zipfile.ZipInfo(
                filename=f"{name}.npy",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            member.compress_type = zipfile.ZIP_DEFLATED
            member.create_system = 3
            member.external_attr = 0o600 << 16
            archive.writestr(member, array_buffer.getvalue())
    return archive_buffer.getvalue()


def _load_and_verify_existing_artifact(path, expected_arrays):
    existing_bytes = path.read_bytes()
    existing_sha256 = hashlib.sha256(existing_bytes).hexdigest()
    prediction_identity_names = ("record_key", "y_true", "y_pred")
    metadata_json_audit = "NOT_PRESENT"
    with np.load(io.BytesIO(existing_bytes), allow_pickle=False) as existing:
        missing_identity_arrays = [
            name for name in prediction_identity_names if name not in existing.files
        ]
        if missing_identity_arrays:
            _stop(
                "existing Stage 1 artifact is missing immutable prediction arrays: "
                + ", ".join(missing_identity_arrays)
            )
        for name in prediction_identity_names:
            if not np.array_equal(existing[name], expected_arrays[name]):
                _stop(
                    f"existing Stage 1 artifact {name} differs; overwrite is forbidden"
                )
        if "metadata_json" in existing.files:
            try:
                json.loads(str(existing["metadata_json"].item()))
                metadata_json_audit = "READABLE"
            except (TypeError, ValueError):
                metadata_json_audit = "UNREADABLE_IGNORED"
    return existing_sha256, metadata_json_audit


REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from dcc_data.mission2_model_experiments import (  # noqa: E402
    E05_FEATURE_NAMES,
    EXPECTED_INTERNAL_SPLIT,
    EXPECTED_OFFICIAL_CALLS,
    EXPECTED_OFFICIAL_ROWS,
    calculate_prediction_metrics,
    create_fixed_internal_split,
    load_standalone_e05_cache,
    record_key_signature,
    standalone_group_split_indices,
    standalone_training_cache_paths,
)
from dcc_data.mission2_acoustic_feature_expansion import (  # noqa: E402
    BLOCK_BY_ID,
    CANONICAL_52D_FEATURE_NAMES,
    M04_ITER100_FEATURE_SCREENING_CONFIG,
    NEXT_CANDIDATE_BLOCK_IDS,
    RESIDUAL_TARGET_BLOCK_IDS,
    ROUND1_BLOCK_IDS,
    ROUND2_BLOCK_IDS,
    _legacy_acoustic_array_fingerprint,
    _prepare_representation,
    _run_prepared_representation,
    load_candidate_feature_cache,
)


if dict(EXPECTED_INTERNAL_SPLIT) != STAGE0_SPLIT_CONTRACT:
    _stop("EXPECTED_INTERNAL_SPLIT differs from the Stage 0 freeze")
if dict(M04_ITER100_FEATURE_SCREENING_CONFIG) != STAGE0_M04_CONFIG:
    _stop("M04-100 config differs from the Stage 0 freeze")

manifest_path = REPO_ROOT / "tmp/mission2_eda/utterance_manifest_v1.parquet"
candidate_cache_root = (
    REPO_ROOT / "data/processed/mission2_acoustic_feature_expansion"
)
artifact_path = REPO_ROOT / STAGE1_ARTIFACT_RELATIVE_PATH

stage1_e05 = load_standalone_e05_cache(
    standalone_training_cache_paths(REPO_ROOT),
    split="Training",
    expected_rows=EXPECTED_OFFICIAL_ROWS["Training"],
    expected_calls=EXPECTED_OFFICIAL_CALLS["Training"],
)
stage1_manifest = pd.read_parquet(manifest_path)
stage1_record_keys = np.asarray(stage1_e05.validated.record_keys).astype(str)
stage1_call_ids = stage1_e05.tracking["call_id"].astype(str).to_numpy()
stage1_labels = np.asarray(stage1_e05.validated.y, dtype=np.int64)

stage1_cache_af02 = load_candidate_feature_cache(
    manifest=stage1_manifest,
    cache_root=candidate_cache_root,
    split="Training",
    block_ids=ROUND1_BLOCK_IDS,
)
stage1_cache_af04 = load_candidate_feature_cache(
    manifest=stage1_manifest,
    cache_root=candidate_cache_root,
    split="Training",
    block_ids=ROUND2_BLOCK_IDS,
)
stage1_cache_af09 = load_candidate_feature_cache(
    manifest=stage1_manifest,
    cache_root=candidate_cache_root,
    split="Training",
    block_ids=NEXT_CANDIDATE_BLOCK_IDS,
)
stage1_cache_af10_af11 = load_candidate_feature_cache(
    manifest=stage1_manifest,
    cache_root=candidate_cache_root,
    split="Training",
    block_ids=RESIDUAL_TARGET_BLOCK_IDS,
)

if (
    stage1_record_keys.shape != (STAGE0_SPLIT_CONTRACT["total_rows"],)
    or stage1_call_ids.shape != stage1_record_keys.shape
    or stage1_labels.shape != stage1_record_keys.shape
    or len(np.unique(stage1_record_keys)) != len(stage1_record_keys)
):
    _stop("canonical record_key/call_id/label shape or uniqueness differs")
if set(np.unique(stage1_labels).tolist()) != {0, 1}:
    _stop("canonical labels are not exactly binary 0/1")

for cache_name, cache_frame in (
    ("AF02", stage1_cache_af02),
    ("AF04", stage1_cache_af04),
    ("AF09", stage1_cache_af09),
    ("AF10_AF11", stage1_cache_af10_af11),
):
    cache_keys = cache_frame["record_key"].astype(str).to_numpy()
    cache_calls = cache_frame["call_id"].astype(str).to_numpy()
    if not np.array_equal(cache_keys, stage1_record_keys):
        _stop(f"{cache_name} record_key alignment differs")
    if not np.array_equal(cache_calls, stage1_call_ids):
        _stop(f"{cache_name} call_id alignment differs")

stage1_af02_names = BLOCK_BY_ID["AF02_PITCH_VOICING"].feature_names
stage1_af04_names = BLOCK_BY_ID["AF04_CPP"].feature_names
stage1_af09_names = BLOCK_BY_ID[
    "AF09_MULTIBAND_ENVELOPE_COHERENCE"
].feature_names
stage1_af10_names = BLOCK_BY_ID[
    "AF10_SPEECH_ACTIVITY_BOUNDARY"
].feature_names
stage1_af11_names = BLOCK_BY_ID[
    "AF11_SPECTRAL_ENTROPY_DYNAMICS"
].feature_names
stage1_feature_names = (
    *E05_FEATURE_NAMES,
    *stage1_af02_names,
    *stage1_af04_names,
    *stage1_af09_names,
    *stage1_af10_names,
    *stage1_af11_names,
)
stage1_expected_feature_names = (
    *CANONICAL_52D_FEATURE_NAMES,
    *stage1_af10_names,
    *stage1_af11_names,
)
if stage1_feature_names != stage1_expected_feature_names:
    _stop("59D feature order differs from 52D prefix + AF10 4D + AF11 3D")
if len(stage1_feature_names) != 59:
    _stop(f"59D feature dimension differs: {len(stage1_feature_names)}")

stage1_feature_sources = (
    (stage1_e05.validated.X, E05_FEATURE_NAMES),
    (
        stage1_cache_af02.loc[:, stage1_af02_names].to_numpy(dtype=np.float32),
        stage1_af02_names,
    ),
    (
        stage1_cache_af04.loc[:, stage1_af04_names].to_numpy(dtype=np.float32),
        stage1_af04_names,
    ),
    (
        stage1_cache_af09.loc[:, stage1_af09_names].to_numpy(dtype=np.float32),
        stage1_af09_names,
    ),
    (
        stage1_cache_af10_af11.loc[:, stage1_af10_names].to_numpy(
            dtype=np.float32
        ),
        stage1_af10_names,
    ),
    (
        stage1_cache_af10_af11.loc[:, stage1_af11_names].to_numpy(
            dtype=np.float32
        ),
        stage1_af11_names,
    ),
)
stage1_matrix = np.concatenate(
    [matrix for matrix, _ in stage1_feature_sources], axis=1
).astype(np.float32, copy=False)
if stage1_matrix.shape != (STAGE0_SPLIT_CONTRACT["total_rows"], 59):
    _stop(f"59D matrix shape differs: {stage1_matrix.shape}")
if not np.isfinite(stage1_matrix).all():
    _stop("59D matrix contains NaN or inf")
if not np.array_equal(stage1_matrix[:, :42], stage1_e05.validated.X):
    _stop("59D assembly changed the immutable E05 prefix")

stage1_train_idx, stage1_validation_idx, stage1_split_audit = (
    create_fixed_internal_split(
        stage1_call_ids,
        split_function=standalone_group_split_indices,
        expected_contract=STAGE0_SPLIT_CONTRACT,
    )
)
if stage1_split_audit != {
    "seed": 42,
    "test_size": 0.2,
    **STAGE0_SPLIT_CONTRACT,
}:
    _stop(f"fixed GroupSplit audit differs: {stage1_split_audit}")
if set(stage1_call_ids[stage1_train_idx]) & set(
    stage1_call_ids[stage1_validation_idx]
):
    _stop("fixed GroupSplit has call_id overlap")

stage1_prepared = _prepare_representation(
    representation_id=STAGE1_REPRESENTATION_ID,
    X=stage1_matrix,
    y=stage1_labels,
    record_keys=stage1_record_keys,
    feature_names=stage1_feature_names,
    train_idx=stage1_train_idx,
    validation_idx=stage1_validation_idx,
    split_audit=stage1_split_audit,
)
if tuple(stage1_prepared.feature_names) != stage1_expected_feature_names:
    _stop("prepared 59D feature order differs")
if not np.array_equal(
    stage1_prepared.training_record_keys,
    stage1_record_keys[stage1_train_idx],
):
    _stop("prepared training record_key alignment differs")
if not np.array_equal(
    stage1_prepared.evaluation_record_keys,
    stage1_record_keys[stage1_validation_idx],
):
    _stop("prepared validation record_key alignment differs")
if not np.array_equal(stage1_prepared.y_train, stage1_labels[stage1_train_idx]):
    _stop("prepared training labels differ")
if not np.array_equal(
    stage1_prepared.y_evaluation,
    stage1_labels[stage1_validation_idx],
):
    _stop("prepared validation labels differ")
if int(stage1_prepared.scaler.n_samples_seen_) != STAGE0_SPLIT_CONTRACT[
    "train_rows"
]:
    _stop("StandardScaler fit rows differ from internal train rows")
if stage1_prepared.X_train_scaled.shape != (711_467, 59):
    _stop(f"scaled train shape differs: {stage1_prepared.X_train_scaled.shape}")
if stage1_prepared.X_evaluation_scaled.shape != (177_891, 59):
    _stop(
        f"scaled internal validation shape differs: "
        f"{stage1_prepared.X_evaluation_scaled.shape}"
    )

stage1_prefit_audit = pd.DataFrame(
    [
        {
            "representation": STAGE1_REPRESENTATION_ID,
            "feature_dimension": len(stage1_prepared.feature_names),
            "feature_order": "52D prefix + AF10 4D + AF11 3D",
            "record_key_call_id_label_alignment": "PASS",
            "train_rows": stage1_split_audit["train_rows"],
            "validation_rows": stage1_split_audit["validation_rows"],
            "train_calls": stage1_split_audit["train_calls"],
            "validation_calls": stage1_split_audit["validation_calls"],
            "call_overlap": stage1_split_audit["call_overlap"],
            "scaler_fit_scope": "internal_train_only",
            "scaler_fit_rows": int(stage1_prepared.scaler.n_samples_seen_),
            "model": "MLPClassifier",
            "max_iter": STAGE0_M04_CONFIG["max_iter"],
            "random_state": STAGE0_M04_CONFIG["random_state"],
        }
    ]
)

del (
    stage1_matrix,
    stage1_feature_sources,
    stage1_cache_af02,
    stage1_cache_af04,
    stage1_cache_af09,
    stage1_cache_af10_af11,
    stage1_manifest,
    stage1_e05,
)
gc.collect()

# The only model fit in Stage 1.
stage1_result = _run_prepared_representation(
    stage1_prepared,
    experiment_id="AF10_AF11_59D_M04_ITER100_STAGE1",
    allow_fit=True,
)
if dict(stage1_result.model_config) != STAGE0_M04_CONFIG:
    _stop("fitted model config differs from Stage 0")
if stage1_result.fit_audit["seed"] != 42:
    _stop("fitted model seed differs from Stage 0")
if stage1_result.fit_audit["scaler_fit_scope"] != "internal_train_only":
    _stop("fitted result scaler scope differs from Stage 0")

stage1_train_prediction = stage1_result.model.predict(
    stage1_prepared.X_train_scaled
).astype(np.int64)
stage1_validation_prediction = np.asarray(stage1_result.y_pred, dtype=np.int64)
stage1_train_metrics = calculate_prediction_metrics(
    stage1_prepared.y_train,
    stage1_train_prediction,
)
stage1_validation_metrics = calculate_prediction_metrics(
    stage1_prepared.y_evaluation,
    stage1_validation_prediction,
)
stage1_train_accuracy = float(stage1_train_metrics["accuracy"])
stage1_validation_accuracy = float(stage1_validation_metrics["accuracy"])
stage1_generalization_gap = stage1_train_accuracy - stage1_validation_accuracy

stage1_postfit_mismatches = []
if stage1_validation_accuracy != STAGE0_VALIDATION_ACCURACY:
    stage1_postfit_mismatches.append("validation_accuracy")
if stage1_validation_metrics["confusion_matrix"] != STAGE0_VALIDATION_CM:
    stage1_postfit_mismatches.append("validation_confusion_matrix")
if (
    float(stage1_validation_metrics["recall_speaker_0"])
    != STAGE0_VALIDATION_RECALL_0
):
    stage1_postfit_mismatches.append("validation_recall_speaker_0")
if (
    float(stage1_validation_metrics["recall_speaker_1"])
    != STAGE0_VALIDATION_RECALL_1
):
    stage1_postfit_mismatches.append("validation_recall_speaker_1")
if int(stage1_validation_metrics["correct_count"]) != STAGE0_VALIDATION_CORRECT:
    stage1_postfit_mismatches.append("validation_correct_count")
if stage1_postfit_mismatches:
    _stop(
        "Stage 1 fit did not reproduce the frozen Stage 0 validation reference: "
        + ", ".join(stage1_postfit_mismatches)
    )

stage1_validation_record_keys = (
    stage1_prepared.evaluation_record_keys.astype(str).copy()
)
stage1_validation_y_true = stage1_prepared.y_evaluation.astype(np.int64).copy()
stage1_validation_y_pred = stage1_validation_prediction.astype(np.int64).copy()
stage1_row_key_fingerprint = record_key_signature(
    stage1_validation_record_keys
)
stage1_y_true_fingerprint = _legacy_acoustic_array_fingerprint(stage1_validation_y_true)
stage1_prediction_fingerprint = _legacy_acoustic_array_fingerprint(stage1_validation_y_pred)
stage1_n_iter = int(stage1_result.model.n_iter_)
stage1_max_iter_reached = stage1_n_iter >= STAGE0_M04_CONFIG["max_iter"]
stage1_convergence_warning = bool(
    stage1_result.fit_audit["convergence_warning_raised"]
)
stage1_convergence_status = (
    "MAX_ITER_REACHED_NOT_CONVERGED"
    if stage1_max_iter_reached and stage1_convergence_warning
    else "MAX_ITER_REACHED"
    if stage1_max_iter_reached
    else "CONVERGED_BEFORE_MAX_ITER"
)
stage1_final_training_loss = float(stage1_result.model.loss_)

stage1_artifact_metadata = {
    "artifact_schema": "mission2_stage1_internal_predictions_v1",
    "representation_id": STAGE1_REPRESENTATION_ID,
    "feature_names": list(stage1_prepared.feature_names),
    "feature_dimension": len(stage1_prepared.feature_names),
    "model_name": stage1_result.model_name,
    "model_config": dict(stage1_result.model_config),
    "split_audit": dict(stage1_split_audit),
    "scaler_fit_scope": "internal_train_only",
    "scaler_fit_rows": int(stage1_prepared.scaler.n_samples_seen_),
    "validation_row_key_fingerprint": stage1_row_key_fingerprint,
    "validation_y_true_fingerprint": stage1_y_true_fingerprint,
    "validation_prediction_fingerprint": stage1_prediction_fingerprint,
    "training_accuracy": stage1_train_accuracy,
    "internal_validation_accuracy": stage1_validation_accuracy,
    "generalization_gap": stage1_generalization_gap,
    "validation_confusion_matrix": stage1_validation_metrics[
        "confusion_matrix"
    ],
    "validation_recall_speaker_0": float(
        stage1_validation_metrics["recall_speaker_0"]
    ),
    "validation_recall_speaker_1": float(
        stage1_validation_metrics["recall_speaker_1"]
    ),
    "n_iter": stage1_n_iter,
    "convergence_status": stage1_convergence_status,
    "final_training_loss": stage1_final_training_loss,
}
stage1_artifact_arrays = {
    "metadata_json": np.asarray(
        json.dumps(
            stage1_artifact_metadata,
            sort_keys=True,
            separators=(",", ":"),
        )
    ),
    "record_key": stage1_validation_record_keys,
    "y_pred": stage1_validation_y_pred,
    "y_true": stage1_validation_y_true,
}
stage1_artifact_bytes = _deterministic_npz_bytes(stage1_artifact_arrays)
stage1_expected_artifact_sha256 = hashlib.sha256(
    stage1_artifact_bytes
).hexdigest()
artifact_path.parent.mkdir(parents=True, exist_ok=True)
if artifact_path.exists():
    (
        stage1_artifact_sha256,
        stage1_artifact_metadata_json_audit,
    ) = _load_and_verify_existing_artifact(
        artifact_path,
        stage1_artifact_arrays,
    )
    stage1_artifact_action = "REUSED_EXACT_IMMUTABLE_PREDICTIONS"
else:
    with artifact_path.open("xb") as artifact_file:
        artifact_file.write(stage1_artifact_bytes)
        artifact_file.flush()
        os.fsync(artifact_file.fileno())
    stage1_artifact_sha256 = hashlib.sha256(
        artifact_path.read_bytes()
    ).hexdigest()
    if stage1_artifact_sha256 != stage1_expected_artifact_sha256:
        _stop("new Stage 1 artifact SHA-256 verification failed")
    stage1_artifact_action = "CREATED_IMMUTABLE_ARTIFACT"
    stage1_artifact_metadata_json_audit = "READABLE"

stage1_result_table = pd.DataFrame(
    [
        {
            "training_accuracy": stage1_train_accuracy,
            "internal_validation_accuracy": stage1_validation_accuracy,
            "generalization_gap": stage1_generalization_gap,
            "validation_confusion_matrix": stage1_validation_metrics[
                "confusion_matrix"
            ],
            "validation_recall_speaker_0": float(
                stage1_validation_metrics["recall_speaker_0"]
            ),
            "validation_recall_speaker_1": float(
                stage1_validation_metrics["recall_speaker_1"]
            ),
            "model_n_iter": stage1_n_iter,
            "max_iter_reached": stage1_max_iter_reached,
            "convergence_warning_raised": stage1_convergence_warning,
            "convergence_status": stage1_convergence_status,
            "final_training_loss": stage1_final_training_loss,
        }
    ]
)
stage1_artifact_audit = pd.DataFrame(
    [
        {
            "artifact_action": stage1_artifact_action,
            "artifact_path": str(STAGE1_ARTIFACT_RELATIVE_PATH).replace("\\", "/"),
            "artifact_sha256": stage1_artifact_sha256,
            "metadata_json_audit": stage1_artifact_metadata_json_audit,
            "validation_row_key_fingerprint": stage1_row_key_fingerprint,
            "validation_y_true_fingerprint": stage1_y_true_fingerprint,
            "validation_prediction_fingerprint": stage1_prediction_fingerprint,
            "artifact_record_key_rows": len(stage1_validation_record_keys),
            "artifact_y_true_rows": len(stage1_validation_y_true),
            "artifact_y_pred_rows": len(stage1_validation_y_pred),
        }
    ]
)

print("Stage 1 pre-fit contract audit")
print(stage1_prefit_audit.to_string(index=False))
print("\nStage 1 result")
print(stage1_result_table.to_string(index=False))
print("\nStage 1 prediction artifact audit")
print(stage1_artifact_audit.to_string(index=False))
print("\nSTAGE 1: PASS")


Stage 1 pre-fit contract audit
      representation  feature_dimension                  feature_order record_key_call_id_label_alignment  train_rows  validation_rows  train_calls  validation_calls  call_overlap    scaler_fit_scope  scaler_fit_rows         model  max_iter  random_state
AF10_AF11_59D_STAGE1                 59 52D prefix + AF10 4D + AF11 3D                               PASS      711467           177891        23360              5840             0 internal_train_only           711467 MLPClassifier       100            42

Stage 1 result
 training_accuracy  internal_validation_accuracy  generalization_gap     validation_confusion_matrix  validation_recall_speaker_0  validation_recall_speaker_1  model_n_iter  max_iter_reached  convergence_warning_raised             convergence_status  final_training_loss
          0.876611                      0.874558            0.002053 [[71421, 13930], [8385, 84155]]                     0.836792                     0.909391           100

### Stage 2 — AF05 Delta-MFCC 단독 추가 실험

**목적:**  
현재 59D feature에 `AF05 Delta-MFCC 2D`만 추가했을 때 성능이 실제로 좋아지는지 확인한다.

**비교:**  
- 기존 기준: 59D
- 새 후보: 59D + AF05 2D = 61D
- 모델, split, seed, scaler 등 다른 조건은 모두 동일하게 유지

**확인하려는 것:**  
AF05가 현재 59D에 없는 추가적인 음성 정보를 제공하는지 확인한다.

**주의:**  
이번 실험에서는 AF05 외 다른 feature나 모델은 바꾸지 않는다.  
따라서 성능 차이는 가능한 한 AF05 추가 효과로 해석한다.

In [1]:
from pathlib import Path
import gc
import json
import sys

import numpy as np
import pandas as pd


STAGE2_REPRESENTATION_ID = "AF10_AF11_AF05_61D_STAGE2"
STAGE1_ARTIFACT_RELATIVE_PATH = Path(
    "data/processed/mission2_acoustic_feature_expansion/"
    "stage1_af10_af11_59d_m04_iter100_internal_predictions_v1.npz"
)
STAGE2_SPLIT_CONTRACT = {
    "total_rows": 889_358,
    "train_rows": 711_467,
    "validation_rows": 177_891,
    "train_calls": 23_360,
    "validation_calls": 5_840,
    "call_overlap": 0,
}
STAGE2_M04_CONFIG = {
    "hidden_layer_sizes": (64,),
    "activation": "relu",
    "solver": "adam",
    "alpha": 0.0001,
    "batch_size": 1024,
    "learning_rate": "constant",
    "learning_rate_init": 0.001,
    "max_iter": 100,
    "shuffle": True,
    "random_state": 42,
    "tol": 0.0001,
    "early_stopping": False,
    "n_iter_no_change": 10,
}
STAGE1_REFERENCE_ACCURACY = 0.8745580158636468
STAGE1_REFERENCE_CM = [[71_421, 13_930], [8_385, 84_155]]
STAGE1_REFERENCE_RECALL_0 = 71_421 / (71_421 + 13_930)
STAGE1_REFERENCE_RECALL_1 = 84_155 / (8_385 + 84_155)
STAGE1_REFERENCE_CORRECT = 155_576
STAGE1_HISTORICAL_HCE_THRESHOLD = 0.90
STAGE1_HISTORICAL_HCE_COUNTS = {"FP": 1_205, "FN": 1_172}
STAGE2_DURATION_ORDER = (
    "<0.5s",
    "0.5-1.0s",
    "1-2s",
    "2-5s",
    ">=5s",
)
STAGE2_DURATION_BINS = [-np.inf, 0.5, 1.0, 2.0, 5.0, np.inf]


def _stop(message):
    raise RuntimeError(f"STOP: {message}")


def _load_stage1_prediction_identity(path):
    """Read the immutable Stage 1 prediction identity without modifying it."""

    if not path.is_file():
        _stop(f"Stage 1 prediction artifact is missing: {path}")
    with np.load(path, allow_pickle=False) as artifact:
        required = ("record_key", "y_true", "y_pred")
        missing = [name for name in required if name not in artifact.files]
        if missing:
            _stop(
                "Stage 1 artifact is missing immutable prediction arrays: "
                + ", ".join(missing)
            )
        record_key = artifact["record_key"].astype(str).copy()
        y_true = artifact["y_true"].astype(np.int64).copy()
        y_pred = artifact["y_pred"].astype(np.int64).copy()
        metadata_audit = "NOT_PRESENT"
        if "metadata_json" in artifact.files:
            try:
                json.loads(str(artifact["metadata_json"].item()))
                metadata_audit = "READABLE"
            except (TypeError, ValueError):
                metadata_audit = "UNREADABLE_IGNORED"
    return record_key, y_true, y_pred, metadata_audit


def _slice_accuracy_row(
    slice_type,
    slice_name,
    selected,
    truth,
    reference_prediction,
    candidate_prediction,
):
    selected = np.asarray(selected, dtype=bool)
    if selected.shape != truth.shape or not np.any(selected):
        _stop(f"invalid or empty Stage 2 slice: {slice_type}/{slice_name}")
    reference_metrics = calculate_prediction_metrics(
        truth[selected],
        reference_prediction[selected],
    )
    candidate_metrics = calculate_prediction_metrics(
        truth[selected],
        candidate_prediction[selected],
    )
    return {
        "slice_type": slice_type,
        "slice": slice_name,
        "count": int(np.sum(selected)),
        "reference_59d_accuracy": float(reference_metrics["accuracy"]),
        "candidate_61d_accuracy": float(candidate_metrics["accuracy"]),
        "accuracy_delta_61d_minus_59d": float(
            candidate_metrics["accuracy"] - reference_metrics["accuracy"]
        ),
        "reference_59d_correct": int(reference_metrics["correct_count"]),
        "candidate_61d_correct": int(candidate_metrics["correct_count"]),
        "correct_delta_61d_minus_59d": int(
            candidate_metrics["correct_count"]
            - reference_metrics["correct_count"]
        ),
    }


REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from dcc_data.mission2_model_experiments import (  # noqa: E402
    E05_FEATURE_NAMES,
    EXPECTED_INTERNAL_SPLIT,
    _overlap_mask,
    calculate_prediction_metrics,
    create_fixed_internal_split,
    load_standalone_e05_cache,
    standalone_group_split_indices,
    standalone_training_cache_paths,
)
from dcc_data.mission2_acoustic_feature_expansion import (  # noqa: E402
    BLOCK_BY_ID,
    CANONICAL_52D_FEATURE_NAMES,
    M04_ITER100_FEATURE_SCREENING_CONFIG,
    NEXT_CANDIDATE_BLOCK_IDS,
    RESIDUAL_TARGET_BLOCK_IDS,
    ROUND1_BLOCK_IDS,
    ROUND2_BLOCK_IDS,
    _prepare_representation,
    _run_prepared_representation,
    load_candidate_feature_cache,
)


# Fail fast before fitting: frozen config, cache identity, exact feature order,
# row alignment, fixed GroupSplit, Stage 1 artifact alignment, and scaler scope.
if dict(EXPECTED_INTERNAL_SPLIT) != STAGE2_SPLIT_CONTRACT:
    _stop("fixed internal split differs from the Stage 0/1 contract")
if dict(M04_ITER100_FEATURE_SCREENING_CONFIG) != STAGE2_M04_CONFIG:
    _stop("M04-100 config differs from the Stage 1 contract")

stage2_af05_block = BLOCK_BY_ID.get("AF05_DELTA_MFCC_DYNAMICS")
stage2_expected_af05_names = (
    "delta_mfcc_norm_mean",
    "delta_mfcc_norm_std",
)
if stage2_af05_block is None:
    _stop("AF05_DELTA_MFCC_DYNAMICS is not registered")
if tuple(stage2_af05_block.feature_names) != stage2_expected_af05_names:
    _stop(
        "AF05 feature names/order differ from Delta-MFCC mean/std: "
        f"{stage2_af05_block.feature_names}"
    )
if int(stage2_af05_block.output_dimension) != 2:
    _stop(f"AF05 output dimension differs: {stage2_af05_block.output_dimension}")

stage2_manifest_path = REPO_ROOT / "tmp/mission2_eda/utterance_manifest_v1.parquet"
stage2_cache_root = REPO_ROOT / "data/processed/mission2_acoustic_feature_expansion"
stage2_stage1_artifact_path = REPO_ROOT / STAGE1_ARTIFACT_RELATIVE_PATH

stage2_e05 = load_standalone_e05_cache(
    standalone_training_cache_paths(REPO_ROOT),
    split="Training",
    expected_rows=STAGE2_SPLIT_CONTRACT["total_rows"],
    expected_calls=(
        STAGE2_SPLIT_CONTRACT["train_calls"]
        + STAGE2_SPLIT_CONTRACT["validation_calls"]
    ),
)
if not stage2_manifest_path.is_file():
    _stop(f"Training manifest is missing: {stage2_manifest_path}")
stage2_manifest = pd.read_parquet(stage2_manifest_path)
stage2_record_keys = np.asarray(stage2_e05.validated.record_keys).astype(str)
stage2_call_ids = stage2_e05.tracking["call_id"].astype(str).to_numpy()
stage2_labels = np.asarray(stage2_e05.validated.y, dtype=np.int64)

stage2_cache_af02 = load_candidate_feature_cache(
    manifest=stage2_manifest,
    cache_root=stage2_cache_root,
    split="Training",
    block_ids=ROUND1_BLOCK_IDS,
)
stage2_cache_af04_af05 = load_candidate_feature_cache(
    manifest=stage2_manifest,
    cache_root=stage2_cache_root,
    split="Training",
    block_ids=ROUND2_BLOCK_IDS,
)
stage2_cache_af09 = load_candidate_feature_cache(
    manifest=stage2_manifest,
    cache_root=stage2_cache_root,
    split="Training",
    block_ids=NEXT_CANDIDATE_BLOCK_IDS,
)
stage2_cache_af10_af11 = load_candidate_feature_cache(
    manifest=stage2_manifest,
    cache_root=stage2_cache_root,
    split="Training",
    block_ids=RESIDUAL_TARGET_BLOCK_IDS,
)

if (
    stage2_record_keys.shape != (STAGE2_SPLIT_CONTRACT["total_rows"],)
    or stage2_call_ids.shape != stage2_record_keys.shape
    or stage2_labels.shape != stage2_record_keys.shape
    or len(np.unique(stage2_record_keys)) != len(stage2_record_keys)
):
    _stop("canonical record_key/call_id/label shape or uniqueness differs")
if set(np.unique(stage2_labels).tolist()) != {0, 1}:
    _stop("canonical labels are not exactly binary 0/1")

for stage2_cache_name, stage2_cache_frame in (
    ("AF02", stage2_cache_af02),
    ("AF04_AF05", stage2_cache_af04_af05),
    ("AF09", stage2_cache_af09),
    ("AF10_AF11", stage2_cache_af10_af11),
):
    stage2_cache_keys = stage2_cache_frame["record_key"].astype(str).to_numpy()
    stage2_cache_calls = stage2_cache_frame["call_id"].astype(str).to_numpy()
    if not np.array_equal(stage2_cache_keys, stage2_record_keys):
        _stop(f"{stage2_cache_name} record_key alignment differs")
    if not np.array_equal(stage2_cache_calls, stage2_call_ids):
        _stop(f"{stage2_cache_name} call_id alignment differs")

stage2_af02_names = BLOCK_BY_ID["AF02_PITCH_VOICING"].feature_names
stage2_af04_names = BLOCK_BY_ID["AF04_CPP"].feature_names
stage2_af09_names = BLOCK_BY_ID[
    "AF09_MULTIBAND_ENVELOPE_COHERENCE"
].feature_names
stage2_af10_names = BLOCK_BY_ID[
    "AF10_SPEECH_ACTIVITY_BOUNDARY"
].feature_names
stage2_af11_names = BLOCK_BY_ID[
    "AF11_SPECTRAL_ENTROPY_DYNAMICS"
].feature_names
stage2_af05_names = stage2_af05_block.feature_names

stage2_59d_feature_names = (
    *E05_FEATURE_NAMES,
    *stage2_af02_names,
    *stage2_af04_names,
    *stage2_af09_names,
    *stage2_af10_names,
    *stage2_af11_names,
)
stage2_expected_59d_feature_names = (
    *CANONICAL_52D_FEATURE_NAMES,
    *stage2_af10_names,
    *stage2_af11_names,
)
if stage2_59d_feature_names != stage2_expected_59d_feature_names:
    _stop("frozen 59D order differs from 52D prefix + AF10 4D + AF11 3D")
if len(stage2_59d_feature_names) != 59:
    _stop(f"frozen reference dimension differs: {len(stage2_59d_feature_names)}")

stage2_61d_feature_names = (
    *stage2_59d_feature_names,
    *stage2_af05_names,
)
if stage2_61d_feature_names[:59] != stage2_59d_feature_names:
    _stop("61D representation changed the frozen 59D prefix")
if stage2_61d_feature_names[-2:] != stage2_expected_af05_names:
    _stop("61D representation does not end with the exact AF05 2D order")
if len(stage2_61d_feature_names) != 61:
    _stop(f"candidate feature dimension differs: {len(stage2_61d_feature_names)}")

stage2_matrix_59d = np.concatenate(
    [
        stage2_e05.validated.X,
        stage2_cache_af02.loc[:, stage2_af02_names].to_numpy(dtype=np.float32),
        stage2_cache_af04_af05.loc[:, stage2_af04_names].to_numpy(
            dtype=np.float32
        ),
        stage2_cache_af09.loc[:, stage2_af09_names].to_numpy(dtype=np.float32),
        stage2_cache_af10_af11.loc[:, stage2_af10_names].to_numpy(
            dtype=np.float32
        ),
        stage2_cache_af10_af11.loc[:, stage2_af11_names].to_numpy(
            dtype=np.float32
        ),
    ],
    axis=1,
).astype(np.float32, copy=False)
stage2_af05_matrix = stage2_cache_af04_af05.loc[
    :, stage2_af05_names
].to_numpy(dtype=np.float32)
stage2_matrix_61d = np.concatenate(
    [stage2_matrix_59d, stage2_af05_matrix],
    axis=1,
).astype(np.float32, copy=False)

if stage2_matrix_59d.shape != (STAGE2_SPLIT_CONTRACT["total_rows"], 59):
    _stop(f"frozen 59D matrix shape differs: {stage2_matrix_59d.shape}")
if stage2_af05_matrix.shape != (STAGE2_SPLIT_CONTRACT["total_rows"], 2):
    _stop(f"AF05 matrix shape differs: {stage2_af05_matrix.shape}")
if stage2_matrix_61d.shape != (STAGE2_SPLIT_CONTRACT["total_rows"], 61):
    _stop(f"candidate 61D matrix shape differs: {stage2_matrix_61d.shape}")
if not np.isfinite(stage2_matrix_61d).all():
    _stop("candidate 61D matrix contains NaN or inf")
if not np.array_equal(stage2_matrix_61d[:, :59], stage2_matrix_59d):
    _stop("candidate 61D values changed the frozen 59D prefix")

stage2_train_idx, stage2_validation_idx, stage2_split_audit = (
    create_fixed_internal_split(
        stage2_call_ids,
        split_function=standalone_group_split_indices,
        expected_contract=STAGE2_SPLIT_CONTRACT,
    )
)
if stage2_split_audit != {
    "seed": 42,
    "test_size": 0.2,
    **STAGE2_SPLIT_CONTRACT,
}:
    _stop(f"fixed GroupSplit audit differs: {stage2_split_audit}")
if set(stage2_call_ids[stage2_train_idx]) & set(
    stage2_call_ids[stage2_validation_idx]
):
    _stop("fixed GroupSplit has call_id overlap")

(
    stage2_reference_record_keys,
    stage2_reference_y_true,
    stage2_reference_y_pred,
    stage2_reference_metadata_audit,
) = _load_stage1_prediction_identity(stage2_stage1_artifact_path)
stage2_validation_record_keys = stage2_record_keys[stage2_validation_idx]
stage2_validation_labels = stage2_labels[stage2_validation_idx]
if not np.array_equal(stage2_reference_record_keys, stage2_validation_record_keys):
    _stop("Stage 1 artifact record_key order differs from the fixed validation split")
if not np.array_equal(stage2_reference_y_true, stage2_validation_labels):
    _stop("Stage 1 artifact y_true differs from the fixed validation labels")
if set(np.unique(stage2_reference_y_pred).tolist()) - {0, 1}:
    _stop("Stage 1 artifact y_pred contains a non-binary class")

stage2_reference_metrics = calculate_prediction_metrics(
    stage2_reference_y_true,
    stage2_reference_y_pred,
)
stage2_reference_mismatches = []
if float(stage2_reference_metrics["accuracy"]) != STAGE1_REFERENCE_ACCURACY:
    stage2_reference_mismatches.append("accuracy")
if stage2_reference_metrics["confusion_matrix"] != STAGE1_REFERENCE_CM:
    stage2_reference_mismatches.append("confusion_matrix")
if (
    float(stage2_reference_metrics["recall_speaker_0"])
    != STAGE1_REFERENCE_RECALL_0
):
    stage2_reference_mismatches.append("recall_speaker_0")
if (
    float(stage2_reference_metrics["recall_speaker_1"])
    != STAGE1_REFERENCE_RECALL_1
):
    stage2_reference_mismatches.append("recall_speaker_1")
if int(stage2_reference_metrics["correct_count"]) != STAGE1_REFERENCE_CORRECT:
    stage2_reference_mismatches.append("correct_count")
if stage2_reference_mismatches:
    _stop(
        "Stage 1 prediction artifact differs from the frozen 59D reference: "
        + ", ".join(stage2_reference_mismatches)
    )

stage2_prepared = _prepare_representation(
    representation_id=STAGE2_REPRESENTATION_ID,
    X=stage2_matrix_61d,
    y=stage2_labels,
    record_keys=stage2_record_keys,
    feature_names=stage2_61d_feature_names,
    train_idx=stage2_train_idx,
    validation_idx=stage2_validation_idx,
    split_audit=stage2_split_audit,
)
if tuple(stage2_prepared.feature_names) != stage2_61d_feature_names:
    _stop("prepared candidate feature order differs from the exact 61D contract")
if not np.array_equal(
    stage2_prepared.training_record_keys,
    stage2_record_keys[stage2_train_idx],
):
    _stop("prepared training record_key alignment differs")
if not np.array_equal(
    stage2_prepared.evaluation_record_keys,
    stage2_reference_record_keys,
):
    _stop("prepared validation record_key alignment differs from Stage 1")
if not np.array_equal(stage2_prepared.y_train, stage2_labels[stage2_train_idx]):
    _stop("prepared training labels differ")
if not np.array_equal(stage2_prepared.y_evaluation, stage2_reference_y_true):
    _stop("prepared validation labels differ from Stage 1")
if int(stage2_prepared.scaler.n_samples_seen_) != STAGE2_SPLIT_CONTRACT[
    "train_rows"
]:
    _stop("StandardScaler was not fit on internal-train rows only")
if stage2_prepared.X_train_scaled.shape != (711_467, 61):
    _stop(f"scaled training shape differs: {stage2_prepared.X_train_scaled.shape}")
if stage2_prepared.X_evaluation_scaled.shape != (177_891, 61):
    _stop(
        "scaled internal-validation shape differs: "
        f"{stage2_prepared.X_evaluation_scaled.shape}"
    )

stage2_prefit_audit = pd.DataFrame(
    [
        {
            "baseline": "frozen AF10+AF11 59D",
            "candidate": "frozen 59D prefix + AF05 Delta-MFCC 2D",
            "candidate_dimension": len(stage2_prepared.feature_names),
            "af05_feature_order": stage2_af05_names,
            "row_key_call_label_alignment": "PASS",
            "train_rows": stage2_split_audit["train_rows"],
            "validation_rows": stage2_split_audit["validation_rows"],
            "train_calls": stage2_split_audit["train_calls"],
            "validation_calls": stage2_split_audit["validation_calls"],
            "call_overlap": stage2_split_audit["call_overlap"],
            "scaler_fit_scope": "internal_train_only",
            "scaler_fit_rows": int(stage2_prepared.scaler.n_samples_seen_),
            "model": "MLPClassifier",
            "max_iter": STAGE2_M04_CONFIG["max_iter"],
            "random_state": STAGE2_M04_CONFIG["random_state"],
            "stage1_artifact_metadata": stage2_reference_metadata_audit,
        }
    ]
)

stage2_validation_tracking = stage2_e05.tracking.iloc[
    stage2_validation_idx
].reset_index(drop=True).copy()
if not np.array_equal(
    stage2_validation_tracking["record_key"].astype(str).to_numpy(),
    stage2_reference_record_keys,
):
    _stop("slice tracking record_key order differs from Stage 1 artifact")

del (
    stage2_matrix_59d,
    stage2_af05_matrix,
    stage2_matrix_61d,
    stage2_cache_af02,
    stage2_cache_af04_af05,
    stage2_cache_af09,
    stage2_cache_af10_af11,
    stage2_manifest,
    stage2_e05,
)
gc.collect()

# This is the only fit in Stage 2. Running this cell fits exact M04-100 once.
stage2_result = _run_prepared_representation(
    stage2_prepared,
    experiment_id="AF10_AF11_AF05_61D_M04_ITER100_STAGE2",
    allow_fit=True,
)
if dict(stage2_result.model_config) != STAGE2_M04_CONFIG:
    _stop("fitted model config differs from Stage 1")
if int(stage2_result.fit_audit["seed"]) != 42:
    _stop("fitted model seed differs from Stage 1")
if stage2_result.fit_audit["scaler_fit_scope"] != "internal_train_only":
    _stop("fitted result scaler scope differs from Stage 1")

stage2_train_prediction = stage2_result.model.predict(
    stage2_prepared.X_train_scaled
).astype(np.int64)
stage2_validation_prediction = np.asarray(stage2_result.y_pred, dtype=np.int64)
stage2_validation_probabilities = np.asarray(
    stage2_result.probabilities,
    dtype=np.float64,
)
if stage2_validation_prediction.shape != stage2_reference_y_pred.shape:
    _stop("61D prediction shape differs from Stage 1")
if stage2_validation_probabilities.shape != (177_891, 2):
    _stop(
        "61D validation probability shape differs: "
        f"{stage2_validation_probabilities.shape}"
    )
if not np.isfinite(stage2_validation_probabilities).all():
    _stop("61D validation probabilities contain NaN or inf")

stage2_train_metrics = calculate_prediction_metrics(
    stage2_prepared.y_train,
    stage2_train_prediction,
)
stage2_validation_metrics = calculate_prediction_metrics(
    stage2_reference_y_true,
    stage2_validation_prediction,
)
stage2_train_accuracy = float(stage2_train_metrics["accuracy"])
stage2_validation_accuracy = float(stage2_validation_metrics["accuracy"])
stage2_generalization_gap = stage2_train_accuracy - stage2_validation_accuracy
stage2_n_iter = int(stage2_result.model.n_iter_)
stage2_max_iter_reached = stage2_n_iter >= STAGE2_M04_CONFIG["max_iter"]
stage2_convergence_warning = bool(
    stage2_result.fit_audit["convergence_warning_raised"]
)
stage2_convergence_status = (
    "MAX_ITER_REACHED_NOT_CONVERGED"
    if stage2_max_iter_reached and stage2_convergence_warning
    else "MAX_ITER_REACHED"
    if stage2_max_iter_reached
    else "CONVERGED_BEFORE_MAX_ITER"
)
stage2_final_training_loss = float(stage2_result.model.loss_)

stage2_result_table = pd.DataFrame(
    [
        {
            "training_accuracy": stage2_train_accuracy,
            "internal_validation_accuracy": stage2_validation_accuracy,
            "generalization_gap": stage2_generalization_gap,
            "validation_confusion_matrix": stage2_validation_metrics[
                "confusion_matrix"
            ],
            "validation_recall_speaker_0": float(
                stage2_validation_metrics["recall_speaker_0"]
            ),
            "validation_recall_speaker_1": float(
                stage2_validation_metrics["recall_speaker_1"]
            ),
            "model_n_iter": stage2_n_iter,
            "max_iter_reached": stage2_max_iter_reached,
            "convergence_warning_raised": stage2_convergence_warning,
            "convergence_status": stage2_convergence_status,
            "final_training_loss": stage2_final_training_loss,
        }
    ]
)

stage2_reference_correct = stage2_reference_y_pred == stage2_reference_y_true
stage2_candidate_correct = stage2_validation_prediction == stage2_reference_y_true
stage2_wrong_to_correct = int(
    np.sum((~stage2_reference_correct) & stage2_candidate_correct)
)
stage2_correct_to_wrong = int(
    np.sum(stage2_reference_correct & (~stage2_candidate_correct))
)
stage2_net_correction = stage2_wrong_to_correct - stage2_correct_to_wrong
if stage2_net_correction != (
    int(stage2_validation_metrics["correct_count"])
    - int(stage2_reference_metrics["correct_count"])
):
    _stop("paired net correction differs from the correct-count delta")

stage2_overall_comparison = pd.DataFrame(
    [
        {
            "reference": "frozen AF10+AF11 59D",
            "candidate": "59D + AF05 Delta-MFCC 2D = 61D",
            "reference_accuracy": float(stage2_reference_metrics["accuracy"]),
            "candidate_accuracy": stage2_validation_accuracy,
            "accuracy_delta_61d_minus_59d": float(
                stage2_validation_accuracy
                - float(stage2_reference_metrics["accuracy"])
            ),
            "reference_recall_speaker_0": float(
                stage2_reference_metrics["recall_speaker_0"]
            ),
            "candidate_recall_speaker_0": float(
                stage2_validation_metrics["recall_speaker_0"]
            ),
            "recall_0_delta_61d_minus_59d": float(
                stage2_validation_metrics["recall_speaker_0"]
                - stage2_reference_metrics["recall_speaker_0"]
            ),
            "reference_recall_speaker_1": float(
                stage2_reference_metrics["recall_speaker_1"]
            ),
            "candidate_recall_speaker_1": float(
                stage2_validation_metrics["recall_speaker_1"]
            ),
            "recall_1_delta_61d_minus_59d": float(
                stage2_validation_metrics["recall_speaker_1"]
                - stage2_reference_metrics["recall_speaker_1"]
            ),
            "59d_wrong_to_61d_correct": stage2_wrong_to_correct,
            "59d_correct_to_61d_wrong": stage2_correct_to_wrong,
            "net_correction": stage2_net_correction,
        }
    ]
)
stage2_cm_change = pd.DataFrame(
    np.asarray(stage2_validation_metrics["confusion_matrix"], dtype=np.int64)
    - np.asarray(stage2_reference_metrics["confusion_matrix"], dtype=np.int64),
    index=("true_0", "true_1"),
    columns=("pred_0", "pred_1"),
)

# Historical 59D probabilities were not persisted. HCE comparison is aggregate
# only; no sample-level paired HCE claim is made.
stage2_candidate_confidence = np.max(stage2_validation_probabilities, axis=1)
stage2_candidate_hce_counts = {
    "FP": int(
        np.sum(
            (stage2_candidate_confidence >= STAGE1_HISTORICAL_HCE_THRESHOLD)
            & (stage2_reference_y_true == 0)
            & (stage2_validation_prediction == 1)
        )
    ),
    "FN": int(
        np.sum(
            (stage2_candidate_confidence >= STAGE1_HISTORICAL_HCE_THRESHOLD)
            & (stage2_reference_y_true == 1)
            & (stage2_validation_prediction == 0)
        )
    ),
}
stage2_hce_rows = []
for stage2_error_type in ("FP", "FN"):
    stage2_historical_count = STAGE1_HISTORICAL_HCE_COUNTS[stage2_error_type]
    stage2_candidate_count = stage2_candidate_hce_counts[stage2_error_type]
    stage2_hce_rows.append(
        {
            "scope": "historical_aggregate_reference_only",
            "confidence_threshold": STAGE1_HISTORICAL_HCE_THRESHOLD,
            "error_type": stage2_error_type,
            "reference_59d_count": stage2_historical_count,
            "candidate_61d_count": stage2_candidate_count,
            "count_delta_61d_minus_59d": (
                stage2_candidate_count - stage2_historical_count
            ),
            "sample_level_hce_pairing": "NOT AVAILABLE / NOT PERFORMED",
        }
    )
stage2_hce_rows.append(
    {
        "scope": "historical_aggregate_reference_only",
        "confidence_threshold": STAGE1_HISTORICAL_HCE_THRESHOLD,
        "error_type": "TOTAL",
        "reference_59d_count": int(sum(STAGE1_HISTORICAL_HCE_COUNTS.values())),
        "candidate_61d_count": int(sum(stage2_candidate_hce_counts.values())),
        "count_delta_61d_minus_59d": int(
            sum(stage2_candidate_hce_counts.values())
            - sum(STAGE1_HISTORICAL_HCE_COUNTS.values())
        ),
        "sample_level_hce_pairing": "NOT AVAILABLE / NOT PERFORMED",
    }
)
stage2_hce_comparison = pd.DataFrame(stage2_hce_rows)

stage2_duration_sec = (
    pd.to_numeric(
        stage2_validation_tracking["endAt_ms"],
        errors="raise",
    ).to_numpy(dtype=np.float64)
    - pd.to_numeric(
        stage2_validation_tracking["startAt_ms"],
        errors="raise",
    ).to_numpy(dtype=np.float64)
) / 1_000.0
stage2_duration_slice = pd.cut(
    stage2_duration_sec,
    bins=STAGE2_DURATION_BINS,
    labels=STAGE2_DURATION_ORDER,
    right=False,
    ordered=True,
).astype(str)
stage2_overlap = _overlap_mask(stage2_validation_tracking)

stage2_slice_specs = [
    ("duration", name, stage2_duration_slice == name)
    for name in STAGE2_DURATION_ORDER
]
stage2_slice_specs.extend(
    [
        ("overlap", "overlap", stage2_overlap),
        ("overlap", "no_overlap", ~stage2_overlap),
        (
            "focus",
            "speaker0_0.5-1.0s_overlap",
            (stage2_reference_y_true == 0)
            & (stage2_duration_slice == "0.5-1.0s")
            & stage2_overlap,
        ),
        (
            "focus",
            "speaker0_1-2s_overlap",
            (stage2_reference_y_true == 0)
            & (stage2_duration_slice == "1-2s")
            & stage2_overlap,
        ),
    ]
)
stage2_slice_comparison = pd.DataFrame(
    [
        _slice_accuracy_row(
            slice_type,
            slice_name,
            selected,
            stage2_reference_y_true,
            stage2_reference_y_pred,
            stage2_validation_prediction,
        )
        for slice_type, slice_name, selected in stage2_slice_specs
    ]
)

stage2_paired_prediction_change = pd.DataFrame(
    [
        {
            "validation_rows": len(stage2_reference_y_true),
            "59d_wrong_to_61d_correct": stage2_wrong_to_correct,
            "59d_correct_to_61d_wrong": stage2_correct_to_wrong,
            "net_correction": stage2_net_correction,
            "record_key_alignment": "EXACT",
            "y_true_alignment": "EXACT",
        }
    ]
)

print("Stage 2 pre-fit contract audit")
print(stage2_prefit_audit.to_string(index=False))
print("\nStage 2 61D result")
print(stage2_result_table.to_string(index=False))
print("\n59D reference vs 61D candidate")
print(stage2_overall_comparison.to_string(index=False))
print("\nConfusion-matrix change (61D - 59D)")
print(stage2_cm_change.to_string())
print("\nHCE aggregate comparison")
print(
    "59D HCE is a historical aggregate reference (FP 1205 / FN 1172). "
    "Only aggregate count changes are compared; sample-level HCE pairing "
    "is not performed."
)
print(stage2_hce_comparison.to_string(index=False))
print("\nMajor slice Accuracy comparison")
print(stage2_slice_comparison.to_string(index=False))
print("\nPaired prediction change aligned by record_key/y_true")
print(stage2_paired_prediction_change.to_string(index=False))
print("\nSTAGE 2: PASS")
print("STOP: result interpretation and the next experiment decision belong to the user.")


Stage 2 pre-fit contract audit
            baseline                              candidate  candidate_dimension                          af05_feature_order row_key_call_label_alignment  train_rows  validation_rows  train_calls  validation_calls  call_overlap    scaler_fit_scope  scaler_fit_rows         model  max_iter  random_state stage1_artifact_metadata
frozen AF10+AF11 59D frozen 59D prefix + AF05 Delta-MFCC 2D                   61 (delta_mfcc_norm_mean, delta_mfcc_norm_std)                         PASS      711467           177891        23360              5840             0 internal_train_only           711467 MLPClassifier       100            42                 READABLE

Stage 2 61D result
 training_accuracy  internal_validation_accuracy  generalization_gap     validation_confusion_matrix  validation_recall_speaker_0  validation_recall_speaker_1  model_n_iter  max_iter_reached  convergence_warning_raised             convergence_status  final_training_loss
          0.876798    

### Stage 2 결과 해석 — AF05 Delta-MFCC 2D 추가

이번 실험에서는 기존 **59D feature에 AF05 Delta-MFCC 2D만 추가하여 61D**로 만들고, 모델과 split 등 다른 조건은 그대로 유지했다.

#### 1. 전체 Accuracy 변화

- 59D Validation Accuracy: **0.874558**
- 61D Validation Accuracy: **0.874738**
- 변화: **+0.000180 ≈ +0.018%p**

즉 AF05를 추가했을 때 전체 정확도는 아주 조금 올라갔다.

177,891개의 Validation sample 기준으로 보면,

- 59D에서 틀렸지만 61D에서 맞은 샘플: **4,419개**
- 59D에서 맞았지만 61D에서 틀린 샘플: **4,387개**
- 순수하게 더 맞춘 샘플: **32개**

따라서 AF05 때문에 예측이 많이 바뀌기는 했지만, 좋아진 경우와 나빠진 경우가 거의 비슷해서 **전체적인 성능 향상은 매우 작았다.**

---

#### 2. speaker별 성능은 서로 반대 방향으로 변했다

**speaker 0 Recall**

- 59D: **0.836792**
- 61D: **0.851472**
- 약 **+1.47%p 개선**

즉 119대원(speaker 0)을 놓치는 경우는 꽤 줄었다.

반면 **speaker 1 Recall**

- 59D: **0.909391**
- 61D: **0.896196**
- 약 **-1.32%p 감소**

즉 신고자(speaker 1)는 오히려 조금 더 많이 놓치게 되었다.

따라서 AF05는 두 클래스를 모두 좋아지게 만든 feature라기보다는,
**speaker 0 쪽으로 분류 경계를 이동시키는 효과**가 나타난 것으로 보인다.

---

#### 3. 특정 취약 구간에서는 개선이 있었다

특히 기존에 어려웠던

**speaker 0 + 0.5~1초 + overlap**

구간에서는 Accuracy가

- 59D: **0.690948**
- 61D: **0.758149**

로 약 **+6.72%p** 크게 개선되었다.

또한 `<0.5초`, `1~2초`, `2~5초`, overlap 전체에서도 소폭 개선이 있었다.

하지만 `0.5~1초 전체` 구간은 오히려 약 **-0.43%p** 떨어졌고,
no-overlap 구간도 거의 변화가 없거나 아주 조금 감소했다.

즉 AF05가 **모든 발화를 전반적으로 개선한 것은 아니고 특정 어려운 구간에서 도움이 된 것**으로 볼 수 있다.

---

#### 4. High Confidence Error는 감소했다

confidence ≥ 0.90인 높은 확신의 오답(HCE)은

- 59D: **2,377개**
- 61D: **2,305개**
- 변화: **-72개**

로 줄었다.

따라서 전체 Accuracy 향상은 매우 작지만,
**높은 확신으로 틀리는 오류는 조금 줄어든 점**은 긍정적이다.

다만 59D의 HCE는 과거 집계값만 남아 있으므로,
개별 sample 단위로 어떤 HCE가 고쳐졌는지는 비교할 수 없다.

---

### 최종 해석

AF05 Delta-MFCC 2D를 추가했을 때 **전체 Accuracy 개선은 약 +0.018%p로 매우 작았다.**

따라서 AF05가 59D에 완전히 새로운 강한 정보를 추가했다고 보기는 어렵다.

다만 speaker 0 Recall과 일부 어려운 overlap/짧은 발화 구간에서는 개선이 확인되었고 HCE도 감소했다. 반대로 speaker 1 Recall은 감소했다.

즉 이번 결과는

> **AF05가 특정 오류에는 도움이 되지만, 전체 성능을 안정적으로 크게 높여주는 feature는 아니다.**

라고 해석하는 것이 적절하다.

AF05는 전체 Accuracy 개선은 미미했지만 speaker 0 및 짧은 overlap 구간에서 뚜렷한 개선을 보였다. 다만 speaker 1 Recall 하락과 전체 순증가 32개로 class trade-off가 커, 61D로 reference를 변경하지 않고 보조 후보로 보류한다.

### Stage 3 — Handcrafted Feature 탐색 종료

AF05 추가는 특정 취약 구간에서는 개선이 있었지만 전체 Accuracy 개선은 매우 작았고 class trade-off도 발생했다.

따라서 AF05는 보조 후보로 보류하고, 기존 AF10+AF11 59D를 handcrafted reference로 유지한다.

이 시점에서 추가 handcrafted feature 조합 탐색은 종료하고,
다음 단계에서는 표준 acoustic representation과 공정 비교한다.